In [ ]:
# -*- coding: utf-8 -*-
"""Insect Detection and Counting on Sticky Traps - YOLOv8

Automatically generated by Colaboratory.

This notebook trains a YOLOv8 model to detect and count insects on yellow sticky traps.
"""

# 1. Install Required Libraries

In [ ]:
!pip install ultralytics roboflow supervision
!pip install torch torchvision torchaudio
!pip install opencv-python matplotlib pandas numpy
!pip install supervision
!pip install -q ultralytics supervision transformers pillow scikit-learn

# 2. Import Libraries

In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
import shutil

from ultralytics import YOLO
import supervision as sv
from IPython.display import display, Image as IPImage

In [ ]:

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

# 2.5. RESTORE

In [ ]:
# ==============================================================================
#  RESTORE MODEL + RESULTS — DIRECT COPY (matches your unzipped Kaggle input)
#  ─────────────────────────────────────────────────────────────────────────────
#  Your "thesis_model_backup" input dataset is already unzipped on Kaggle
#  (runs/, thesis_results/, BACKUP_METADATA.json, *.yaml all sitting directly
#  in the folder) -- so this copies straight from it instead of extracting
#  a zip. This REPLACES the previous zip-based restore cell (Cell 7).
# ==============================================================================

import shutil
from pathlib import Path

BASE = Path("/kaggle/working")
SRC = Path("/kaggle/input/datasets/your-kaggle-username/thesis-model-backup")

if not SRC.exists():
    raise FileNotFoundError(
        f"Source not found: {SRC}\n"
        "Check the exact Kaggle input path from the left dataset panel."
    )

ITEMS_TO_RESTORE = ["runs", "thesis_results", "dataset.yaml",
                     "exp1.yaml", "exp2_eval.yaml", "exp3.yaml",
                     "target_domain_val.yaml"]

restored, skipped = [], []
for name in ITEMS_TO_RESTORE:
    src = SRC / name
    dst = BASE / name
    if not src.exists():
        skipped.append(name)
        continue
    if dst.exists():
        shutil.rmtree(dst) if dst.is_dir() else dst.unlink()
    if src.is_dir():
        shutil.copytree(src, dst)
        n = sum(1 for _ in dst.rglob("*") if _.is_file())
        print(f"  ✓ restored folder  {name:<20} ({n} files)")
    else:
        shutil.copy2(src, dst)
        print(f"  ✓ restored file    {name}")
    restored.append(name)

print(f"\nRestored {len(restored)} items, skipped {len(skipped)} {skipped if skipped else ''}")

# ── Verify critical weights ──────────────────────────────────────────────────
critical = [
    BASE / "runs/detect/exp1_base_3class/weights/best.pt",
    BASE / "runs/detect/exp3_4class_synthetic_finetuned/weights/best.pt",
]
all_ok = True
print("\nVerifying trained weights:")
for p in critical:
    exists = p.exists()
    print(f"  {'✓' if exists else '✗ MISSING'}  {p.relative_to(BASE)}")
    all_ok = all_ok and exists

print("\n✓ Both models restored -- Exp 1 and Exp 3 training can be SKIPPED."
      if all_ok else
      "\n⚠ Some weights missing -- check the runs/ folder contents inside thesis_model_backup.")


# **SECTION 1:**

# 3. Dataset Setup

## Option A: Download from GitHub (Yellow Sticky Traps Dataset)
Run this cell to download the recommended dataset directly

In [ ]:
# Download the Yellow Sticky Traps Dataset
!git clone https://github.com/md-121/yellow-sticky-traps-dataset.git

# Check the dataset structure
!ls -la yellow-sticky-traps-dataset/

In [ ]:



#step-1

from pathlib import Path

# Path to the downloaded dataset
dataset_path = Path('/kaggle/working/yellow-sticky-traps-dataset')

# List the contents of the main directory
print("Contents of dataset root:")
for item in dataset_path.iterdir():
    print(f"  - {item.name}")

# Check the 'annotations' and 'images' subdirectories
annotations_path = dataset_path / 'annotations'
images_path = dataset_path / 'images'

print(f"\nAnnotations folder exists: {annotations_path.exists()}")
print(f"Images folder exists: {images_path.exists()}")

if annotations_path.exists():
    xml_files = list(annotations_path.glob('*.xml'))
    print(f"Number of XML annotation files: {len(xml_files)}")

if images_path.exists():
    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.JPG'))
    print(f"Number of JPG image files: {len(image_files)}")




#step-2

import xml.etree.ElementTree as ET

def convert_voc_to_yolo(xml_file, classes, output_file):
    """
    Convert PASCAL VOC XML annotation to YOLO format.
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Get image dimensions
    size = root.find('size')
    img_width = int(size.find('width').text)
    img_height = int(size.find('height').text)

    yolo_annotations = []

    for obj in root.findall('object'):
        # Get the class name as it appears in the XML
        class_name = obj.find('name').text

        # Map the class name to its index in your CLASSES list
        if class_name in classes:
            class_id = classes.index(class_name)
        else:
            # Skip if class name is not in your defined list (e.g., 'NC')
            print(f"Warning: Unknown class '{class_name}' found in {xml_file}. Skipping.")
            continue

        # Get bounding box coordinates
        bbox = obj.find('bndbox')
        xmin = float(bbox.find('xmin').text)
        ymin = float(bbox.find('ymin').text)
        xmax = float(bbox.find('xmax').text)
        ymax = float(bbox.find('ymax').text)

        # Convert to YOLO format (center x, center y, width, height) - normalized
        x_center = (xmin + xmax) / 2 / img_width
        y_center = (ymin + ymax) / 2 / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height

        yolo_annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # Write to file
    with open(output_file, 'w') as f:
        f.write('\n'.join(yolo_annotations))

    return len(yolo_annotations)


# Example based on likely output
CLASSES = ['MR', 'NC', 'WF']   # Order doesn't matter, but be consistent


#step-3



import shutil
from sklearn.model_selection import train_test_split

# Define paths
DATASET_PATH = Path('/kaggle/working/yellow-sticky-traps-dataset')
ANNOTATIONS_PATH = DATASET_PATH / 'annotations'
IMAGES_PATH = DATASET_PATH / 'images'
OUTPUT_PATH = Path('/kaggle/working/yolo_dataset') # Output directory for YOLO-formatted data

# Create output directories
for split in ['train', 'val', 'test']:
    (OUTPUT_PATH / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_PATH / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Get all image files
image_files = list(IMAGES_PATH.glob('*.jpg')) + list(IMAGES_PATH.glob('*.JPG')) + list(IMAGES_PATH.glob('*.jpeg'))
print(f"Found {len(image_files)} images")

# Split into train (70%), val (15%), test (15%)
train_files, temp_files = train_test_split(image_files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)
print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

# Convert and copy files
total_annotations = 0

for split, files in [('train', train_files), ('val', val_files), ('test', test_files)]:
    for img_path in files:
        # Copy image
        shutil.copy(img_path, OUTPUT_PATH / 'images' / split / img_path.name)

        # Find corresponding XML annotation
        xml_path = ANNOTATIONS_PATH / f"{img_path.stem}.xml"
        if xml_path.exists():
            yolo_path = OUTPUT_PATH / 'labels' / split / f"{img_path.stem}.txt"
            num_annots = convert_voc_to_yolo(xml_path, CLASSES, yolo_path)
            total_annotations += num_annots
            if num_annots == 0:
                print(f"Warning: No valid annotations found for {img_path.name}")
        else:
            print(f"Warning: No XML file found for {img_path.name}")

print(f"Dataset preparation complete. Total annotations processed: {total_annotations}")



#step-4


# Create dataset.yaml file
yaml_content = f"""
# YOLOv8 dataset configuration for Yellow Sticky Traps
path: {OUTPUT_PATH.absolute()}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val      # val images (relative to 'path')
test: images/test    # test images (relative to 'path')

# Classes
nc: {len(CLASSES)}   # number of classes
names: {CLASSES}     # class names
"""

yaml_path = '/kaggle/working/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"dataset.yaml created at {yaml_path}")
print(yaml_content)



In [ ]:
#step-5

import supervision as sv
import cv2
import numpy as np
from pathlib import Path

# Paths to your dataset
image_dir = Path('/kaggle/working/yolo_dataset/images/train')
label_dir = Path('/kaggle/working/yolo_dataset/labels/train')
class_names = ['MR', 'NT', 'WF']  # Your class names

# Get a random image and its label file
image_path = list(image_dir.glob('*.jpg'))[0]
label_path = label_dir / f"{image_path.stem}.txt"

# Load the image
image = cv2.imread(str(image_path))
height, width, _ = image.shape

# Parse YOLO annotations
xyxy_list = []
class_id_list = []

if label_path.exists():
    with open(label_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        parts = list(map(float, line.strip().split()))
        if len(parts) == 5:
            class_id, x_center, y_center, box_width, box_height = parts

            # Convert from YOLO format (normalized center) to xyxy (absolute)
            x1 = int((x_center - box_width / 2) * width)
            y1 = int((y_center - box_height / 2) * height)
            x2 = int((x_center + box_width / 2) * width)
            y2 = int((y_center + box_height / 2) * height)

            xyxy_list.append([x1, y1, x2, y2])
            class_id_list.append(int(class_id))

# Create a supervision Detections object
if xyxy_list:
    detections = sv.Detections(
        xyxy=np.array(xyxy_list),
        class_id=np.array(class_id_list),
        confidence=np.array([1.0] * len(xyxy_list))  # Dummy confidence for visualization
    )

    # Annotate and display the image
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()

    annotated_image = box_annotator.annotate(scene=image, detections=detections)
    annotated_image = label_annotator.annotate(
        scene=annotated_image, 
        detections=detections, 
        labels=[class_names[class_id] for class_id in class_id_list]
    )

    # Show the result
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 12))
    plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
else:
    print(f"No labels found in {label_path}")

# 4. Convert PASCAL VOC XML to YOLO Format

YOLOv8 expects annotations in a specific format. This function converts the XML files.

In [ ]:
def convert_voc_to_yolo(xml_file, classes, output_file):
    """
    Convert PASCAL VOC XML annotation to YOLO format.
    
    Args:
        xml_file: Path to XML annotation file
        classes: List of class names
        output_file: Path to output YOLO txt file
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Get image dimensions
    size = root.find('size')
    img_width = int(size.find('width').text)
    img_height = int(size.find('height').text)
    
    yolo_annotations = []
    
    for obj in root.findall('object'):
        # Get class name and convert to class ID
        class_name = obj.find('name').text
        if class_name in classes:
            class_id = classes.index(class_name)
        else:
            continue
        
        # Get bounding box coordinates
        bbox = obj.find('bndbox')
        xmin = float(bbox.find('xmin').text)
        ymin = float(bbox.find('ymin').text)
        xmax = float(bbox.find('xmax').text)
        ymax = float(bbox.find('ymax').text)
        
        # Convert to YOLO format (center x, center y, width, height) - normalized
        x_center = (xmin + xmax) / 2 / img_width
        y_center = (ymin + ymax) / 2 / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height
        
        yolo_annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    # Write to file
    with open(output_file, 'w') as f:
        f.write('\n'.join(yolo_annotations))
    
    return len(yolo_annotations)

# Define class names for the Yellow Sticky Traps Dataset
CLASSES = ['MR', 'NC', 'WF']   # use exactly what you found
print(f"Classes: {CLASSES}")

In [ ]:
from pathlib import Path

dataset_path = Path('/kaggle/working/yellow-sticky-traps-dataset')
image_files = list(dataset_path.glob('images/*.jpg')) + list(dataset_path.glob('images/*.JPG'))

print(f"Total images in dataset: {len(image_files)}")

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

# Define paths
DATASET_PATH = Path('yellow-sticky-traps-dataset')
ANNOTATIONS_PATH = DATASET_PATH / 'annotations'
IMAGES_PATH = DATASET_PATH / 'images'

# Check if the folders exist
print(f"Annotations folder exists: {ANNOTATIONS_PATH.exists()}")
print(f"Images folder exists: {IMAGES_PATH.exists()}")

# If they exist, inspect a sample XML file
if ANNOTATIONS_PATH.exists():
    xml_files = list(ANNOTATIONS_PATH.glob('*.xml'))
    if xml_files:
        sample_xml = xml_files[0]
        print(f"\nInspecting sample XML file: {sample_xml.name}")

        tree = ET.parse(sample_xml)
        root = tree.getroot()

        # Print class names found
        classes_found = []
        for obj in root.findall('object'):
            class_name = obj.find('name').text
            classes_found.append(class_name)
            bbox = obj.find('bndbox')
            print(f"Class: {class_name}")
            print(f"  Bounding Box: {bbox.find('xmin').text}, {bbox.find('ymin').text}, {bbox.find('xmax').text}, {bbox.find('ymax').text}")

        # Get unique class names
        unique_classes = list(set(classes_found))
        print(f"\nUnique class names in dataset: {unique_classes}")
    else:
        print("\nNo XML files found in the annotations folder.")
else:
    print("\nAnnotations folder not found. Check the dataset structure.")

# 5. Prepare Dataset for YOLO Training

Organize images and labels into the YOLO directory structure:
```
dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

In [ ]:
# Paths
DATASET_PATH = Path('yellow-sticky-traps-dataset')
ANNOTATIONS_PATH = DATASET_PATH / 'annotations'
IMAGES_PATH = DATASET_PATH / 'images'
OUTPUT_PATH = Path('yolo_dataset')

# Create output directories
for split in ['train', 'val', 'test']:
    (OUTPUT_PATH / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_PATH / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Get all image files (JPG and JPEG)
image_files = list(IMAGES_PATH.glob('*.jpg')) + list(IMAGES_PATH.glob('*.JPG')) + list(IMAGES_PATH.glob('*.jpeg'))

print(f"Found {len(image_files)} images")

# Split into train (70%), val (15%), test (15%)
train_files, temp_files = train_test_split(image_files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

# Convert and copy files
total_annotations = 0

# Process training set
for img_path in train_files:
    # Copy image
    shutil.copy(img_path, OUTPUT_PATH / 'images' / 'train' / img_path.name)
    
    # Find corresponding XML annotation
    xml_path = ANNOTATIONS_PATH / f"{img_path.stem}.xml"
    if xml_path.exists():
        yolo_path = OUTPUT_PATH / 'labels' / 'train' / f"{img_path.stem}.txt"
        num_annots = convert_voc_to_yolo(xml_path, CLASSES, yolo_path)
        total_annotations += num_annots

# Process validation set
for img_path in val_files:
    shutil.copy(img_path, OUTPUT_PATH / 'images' / 'val' / img_path.name)
    xml_path = ANNOTATIONS_PATH / f"{img_path.stem}.xml"
    if xml_path.exists():
        yolo_path = OUTPUT_PATH / 'labels' / 'val' / f"{img_path.stem}.txt"
        num_annots = convert_voc_to_yolo(xml_path, CLASSES, yolo_path)
        total_annotations += num_annots

# Process test set
for img_path in test_files:
    shutil.copy(img_path, OUTPUT_PATH / 'images' / 'test' / img_path.name)
    xml_path = ANNOTATIONS_PATH / f"{img_path.stem}.xml"
    if xml_path.exists():
        yolo_path = OUTPUT_PATH / 'labels' / 'test' / f"{img_path.stem}.txt"
        num_annots = convert_voc_to_yolo(xml_path, CLASSES, yolo_path)
        total_annotations += num_annots

print(f"Dataset preparation complete. Total annotations: {total_annotations}")

# 6. Create YAML Configuration File

In [ ]:
# Create dataset.yaml file for YOLOv8
yaml_content = f"""
# YOLOv8 dataset configuration for Yellow Sticky Traps
path: {OUTPUT_PATH.absolute()}  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images
test: images/test  # test images

# Classes
nc: {len(CLASSES)}  # number of classes
names: {CLASSES}  # class names
"""

yaml_path = 'dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("dataset.yaml created:")
print(yaml_content)

# 7. Train YOLOv8 Model

This will train on the prepared dataset. You can adjust epochs and image size.

In [ ]:
# Load a pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')  # 'n' for nano (fastest), use 's', 'm', 'l', 'x' for better accuracy

# Train the model
results = model.train(
    data='dataset.yaml',
    epochs=50,                # Number of training epochs (adjust based on convergence)
    imgsz=640,                # Image size
    batch=8,                  # Batch size (reduce if GPU memory is low)
    device=0,                 # Use GPU (0) or CPU ('cpu')
    workers=4,                # Number of data loading workers
    patience=10,              # Early stopping patience
    save=True,                # Save checkpoints
    project='insect_detector',  # Project name
    name='yolov8n_insects',      # Experiment name
    exist_ok=True
)

print("Training complete!")

# 8. Evaluate Model on Test Set

In [ ]:
# Correct path: include runs/detect/
best_model = YOLO('runs/detect/insect_detector/yolov8n_insects/weights/best.pt')

# Run evaluation
metrics = best_model.val(data='dataset.yaml')

print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

# 9. Detect and Count Insects on New Images

This function takes an image path and returns:
- Number of detected insects
- Visualization with bounding boxes

In [ ]:
def detect_and_count(image_path, model, confidence_threshold=0.25):
    """
    Detect insects in an image, draw bounding boxes, and count them.
    
    Args:
        image_path: Path to the image
        model: Trained YOLOv8 model
        confidence_threshold: Minimum confidence for detection
    
    Returns:
        count: Number of insects detected
        annotated_image: Image with bounding boxes drawn
        detections: DataFrame with detection details
    """
    # Run inference
    results = model(image_path, conf=confidence_threshold)
    
    # Extract results
    result = results[0]
    boxes = result.boxes
    
    if boxes is None or len(boxes) == 0:
        print("No insects detected.")
        count = 0
        annotated_image = cv2.imread(str(image_path))
        detections = pd.DataFrame()
    else:
        # Get detection details
        class_ids = boxes.cls.cpu().numpy().astype(int)
        confidences = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()
        
        # Create DataFrame
        detections = pd.DataFrame({
            'class': [CLASSES[i] for i in class_ids],
            'confidence': confidences,
            'xmin': xyxy[:, 0],
            'ymin': xyxy[:, 1],
            'xmax': xyxy[:, 2],
            'ymax': xyxy[:, 3]
        })
        
        count = len(detections)
        
        # Get annotated image
        annotated_image = result.plot()
    
    # Print summary
    print(f"\n{'='*50}")
    print(f"Image: {Path(image_path).name}")
    print(f"Total insects detected: {count}")
    if count > 0:
        print(f"\nDetection details:")
        class_counts = detections['class'].value_counts()
        for cls, cnt in class_counts.items():
            print(f"  - {cls}: {cnt}")
    print(f"{'='*50}")
    
    return count, annotated_image, detections

# Test on a sample image from test set
test_images = list((OUTPUT_PATH / 'images' / 'test').glob('*'))
if test_images:
    sample_image = test_images[0]
    count, annotated, detections = detect_and_count(sample_image, best_model)
    
    # Display result
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'Detected Insects: {count}', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Display detection table
    if not detections.empty:
        print("\nDetection Details:")
        display(detections)

# 10. Batch Processing Function

Process multiple images and save results with bounding boxes.

In [ ]:
def batch_detect_and_count(image_folder, model, output_folder, confidence_threshold=0.25):
    """
    Process all images in a folder, detect and count insects.
    
    Args:
        image_folder: Path to folder containing images
        model: Trained YOLOv8 model
        output_folder: Path to save annotated images
        confidence_threshold: Minimum confidence for detection
    
    Returns:
        results_df: DataFrame with per-image results
    """
    os.makedirs(output_folder, exist_ok=True)
    
    image_paths = list(Path(image_folder).glob('*.jpg')) + list(Path(image_folder).glob('*.jpeg')) + list(Path(image_folder).glob('*.png'))
    
    results_list = []
    
    for img_path in image_paths:
        count, annotated, detections = detect_and_count(img_path, model, confidence_threshold)
        
        # Save annotated image
        output_path = Path(output_folder) / f"detected_{img_path.name}"
        cv2.imwrite(str(output_path), annotated)
        
        # Store results
        results_list.append({
            'image': img_path.name,
            'total_count': count,
            'detections': detections
        })
    
    print(f"\nProcessed {len(image_paths)} images. Results saved to {output_folder}")
    
    return results_list

# Test batch processing on a small set
test_folder = OUTPUT_PATH / 'images' / 'test'
output_folder = 'detected_results'

if test_images:
    results = batch_detect_and_count(test_folder, best_model, output_folder, confidence_threshold=0.25)
    
    # Show summary
    print("\nSummary:")
    for r in results:
        print(f"  {r['image']}: {r['total_count']} insects")

# 11. Visualize Training Results

In [ ]:
# ============================================================
# Plot YOLO training results from current project
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import os

BASE = Path("/kaggle/working")

def find_yolo_results_csv():
    candidates = list((BASE / "runs" / "detect").rglob("results.csv"))
    
    if len(candidates) == 0:
        print("No YOLO results.csv files found under:")
        print(BASE / "runs" / "detect")
        return []

    print("Found results.csv files:")
    for i, p in enumerate(candidates):
        print(f"{i}: {p}")

    return candidates


def plot_training_results(results_path):
    results_path = Path(results_path)

    if not results_path.exists():
        print(f"Results file not found: {results_path}")
        return

    df = pd.read_csv(results_path)

    # YOLO sometimes saves column names with spaces
    df.columns = [c.strip() for c in df.columns]

    print("\nPlotting:")
    print(results_path)
    print("Columns:")
    print(df.columns.tolist())

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Training losses
    for col, label in [
        ("train/box_loss", "Box Loss"),
        ("train/cls_loss", "Class Loss"),
        ("train/dfl_loss", "DFL Loss"),
    ]:
        if col in df.columns:
            axes[0, 0].plot(df["epoch"], df[col], label=label)

    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].set_title("Training Losses")
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # mAP
    for col, label in [
        ("metrics/mAP50(B)", "mAP50"),
        ("metrics/mAP50-95(B)", "mAP50-95"),
    ]:
        if col in df.columns:
            axes[0, 1].plot(df["epoch"], df[col], label=label)

    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("mAP")
    axes[0, 1].set_title("Mean Average Precision")
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Precision and recall
    for col, label in [
        ("metrics/precision(B)", "Precision"),
        ("metrics/recall(B)", "Recall"),
    ]:
        if col in df.columns:
            axes[1, 0].plot(df["epoch"], df[col], label=label)

    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Score")
    axes[1, 0].set_title("Precision and Recall")
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    # Validation losses if available
    plotted_val = False
    for col, label in [
        ("val/box_loss", "Val Box Loss"),
        ("val/cls_loss", "Val Class Loss"),
        ("val/dfl_loss", "Val DFL Loss"),
    ]:
        if col in df.columns:
            axes[1, 1].plot(df["epoch"], df[col], label=label)
            plotted_val = True

    if plotted_val:
        axes[1, 1].set_xlabel("Epoch")
        axes[1, 1].set_ylabel("Loss")
        axes[1, 1].set_title("Validation Losses")
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    else:
        axes[1, 1].axis("off")

    plt.suptitle(results_path.parent.name)
    plt.tight_layout()
    plt.show()


# Find and plot all available YOLO training result files
result_files = find_yolo_results_csv()

for result_file in result_files:
    plot_training_results(result_file)

# 12. Save Model for Future Use

In [ ]:
# Save the best model to a file
best_model.export(format='onnx')  # Export to ONNX
best_model.export(format='tflite')  # Export to TFLite for mobile

# Also save as PyTorch model
torch.save(best_model.model.state_dict(), 'insect_detector_model.pth')
print("Model saved successfully!")

# Summary

You now have:
1. **Dataset** prepared in YOLO format
2. **Trained YOLOv8 model** for insect detection
3. **Detection function** that draws bounding boxes and counts insects
4. **Batch processing** capability
5. **Model exported** for deployment

The model can detect insects in new images and provide both the count and visual bounding boxes.

# **SECTION 2:**

# # We now start the synthetic data generation***

In [ ]:
# ==============================================================================
#  RESTORE SYNTHETIC DATA — SKIP SECTION 2
#  ─────────────────────────────────────────────────────────────────────────────
#  Place this cell IMMEDIATELY BEFORE the "SECTION 2" markdown cell.
#  Copies generated_dataset_v6 directly from the Kaggle input dataset
#  "synthetic-gen-data" (already unzipped, as shown in the dataset panel).
#
#  After this cell runs successfully, SKIP cells 34-40 (Section 2) AND
#  skip cell 43 (redundant — this cell already does the same copy) —
#  jump straight to cell 44 (CONFIG).
# ==============================================================================

import shutil
from pathlib import Path

BASE = Path("/kaggle/working")

SRC_SYNTH = Path("/kaggle/input/datasets/your-kaggle-username/synthetic-gen-data/generated_dataset_v6")
DST_SYNTH = BASE / "generated_dataset_v6"

# ── Check source exists ──────────────────────────────────────────────────────
if not SRC_SYNTH.exists():
    raise FileNotFoundError(
        f"Source synthetic dataset not found:\n{SRC_SYNTH}\n\n"
        "Check the exact Kaggle input path from the left dataset panel — "
        "the dataset slug 'synthetic-gen-data' may differ if you re-uploaded it."
    )

# ── Remove any stale working copy ────────────────────────────────────────────
if DST_SYNTH.exists():
    shutil.rmtree(DST_SYNTH)
    print("Removed old working folder:", DST_SYNTH)

# ── Copy dataset ──────────────────────────────────────────────────────────────
print("Copying generated_dataset_v6 from Kaggle input... (2000 images, may take a minute)")
shutil.copytree(SRC_SYNTH, DST_SYNTH)

print("Synthetic dataset copied successfully.")
print("From:", SRC_SYNTH)
print("To:  ", DST_SYNTH)

# ── Verify ──────────────────────────────────────────────────────────────────
image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    image_files.extend(DST_SYNTH.rglob(ext))
label_files = list(DST_SYNTH.rglob("*.txt"))

print(f"\n{'='*60}")
print("SYNTHETIC DATA RESTORE COMPLETE")
print(f"{'='*60}")
print("Images found:", len(image_files))
print("Labels found:", len(label_files))
print("dataset.yaml exists:  ", (DST_SYNTH / "dataset.yaml").exists())
print("stats.json exists:    ", (DST_SYNTH / "stats.json").exists())
print("images/train exists:  ", (DST_SYNTH / "images" / "train").exists())
print("images/val exists:    ", (DST_SYNTH / "images" / "val").exists())
print("labels/train exists:  ", (DST_SYNTH / "labels" / "train").exists())
print("labels/val exists:    ", (DST_SYNTH / "labels" / "val").exists())

if len(image_files) == 2000:
    print("\n✓ Full dataset restored (2000 images). Safe to skip Section 2.")
else:
    print(f"\n⚠ Expected 2000 images, found {len(image_files)}. "
          "Check the dataset before skipping Section 2.")

print("\nNEXT STEPS:")
print("  - SKIP Section 2 (synthetic data generation, all cells under that header)")
print("  - SKIP the 'COPY SYNTHETIC DATASET V6 FROM KAGGLE INPUT' cell "
      "(first cell under Section 3 — redundant, this cell already did that copy)")
print("  - Continue with the CONFIG cell (second cell under Section 3) onward")

In [ ]:
 
import subprocess, sys, os
 
# Block CuPy / GPU onnxruntime before any other import
os.environ["REMBG_PROVIDERS"]       = "CPUExecutionProvider"
os.environ["ONNXRUNTIME_PROVIDERS"] = "CPUExecutionProvider"
os.environ["OMP_NUM_THREADS"]       = "4"
 
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr:
        print(f"  WARN: {r.stderr[-200:]}")
    return r.returncode == 0
 
print("[1/4] Installing rembg (CPU only)...")
run("pip uninstall -y rembg onnxruntime onnxruntime-gpu 2>/dev/null")
run("pip install -q 'onnxruntime>=1.18.0' --no-deps")
run("pip install -q 'rembg==2.0.57' --no-deps")
run("pip install -q 'pillow>=10.0.0' 'scipy>=1.11.0' 'tqdm>=4.66.0' "
    "'pymatting>=1.1.8' 'pooch>=1.6.0' 'aiohttp' 'aiofiles'")
 
print("[2/4] Installing ML stack...")
run("pip install -q 'timm==0.9.16' 'albumentations==1.3.1' "
    "'peft==0.8.0' 'psutil'")
 
print("[3/4] Installing image processing extras...")
run("pip install -q 'opencv-contrib-python-headless' 2>/dev/null || "
    "pip install -q 'opencv-contrib-python' 2>/dev/null || true")
 
print("[4/4] Done.")

In [ ]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  KAGGLE CELL A — Install dependencies & define all paths                   ║
# ║  Run this first. Does NOT restart kernel.                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
 

 
# ── All paths — edit ONLY this section ────────────────────────────────────────
import os
from pathlib import Path
 
BASE = Path("/kaggle/working")
 
# ── INPUT: your Aspilanta raw photos ─────────────────────────────────────────
# These are the original (un-cut) photos with backgrounds
PATH_ASPILANTA_RAW = Path(
    "/kaggle/input/datasets/your-kaggle-username/ashpilanta-3wtop/"
    "data gen-3/clean_insect(top)")
 
# ── INPUT: EU Moths dataset root ──────────────────────────────────────────────
# Must contain:  images/  images.txt  labels.txt  class_names.txt
PATH_EU_MOTHS_ROOT = Path("/kaggle/input/datasets/your-kaggle-username/distractor-insect-1/Distractor insect")      # ← adjust to your dataset slug
 
# ── WORKING DIRS (created automatically) ─────────────────────────────────────
PATH_CUTOUTS_ASPILANTA  = BASE / "cutouts" / "aspilanta"   # extracted targets
PATH_CUTOUTS_DISTRACTOR = BASE / "cutouts" / "eu_moths"    # extracted distractors
PATH_BACKGROUNDS        = Path(
    "/kaggle/input/datasets/your-kaggle-username/ashpilanta-3wtop/"
    "data gen-3/Many Backgrounds")
PATH_OUTPUT             = BASE / "generated_dataset_v5"
 
for p in [PATH_CUTOUTS_ASPILANTA, PATH_CUTOUTS_DISTRACTOR,
          PATH_OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)
 
# ── Quick verify ──────────────────────────────────────────────────────────────
import torch, cv2, numpy as np
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
print("\n" + "=" * 55)
print("CELL A — Environment")
print("=" * 55)
print(f"  numpy   : {np.__version__}")
print(f"  torch   : {torch.__version__}")
print(f"  cv2     : {cv2.__version__}")
print(f"  device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
print(f"\n  Aspilanta raw  : {PATH_ASPILANTA_RAW}")
print(f"  EU Moths root  : {PATH_EU_MOTHS_ROOT}")
print(f"  Backgrounds    : {PATH_BACKGROUNDS}")
print(f"  Output         : {PATH_OUTPUT}")
 
asp_count = len(list(PATH_ASPILANTA_RAW.glob("*.*"))) \
            if PATH_ASPILANTA_RAW.exists() else 0
eum_count = len(list(PATH_EU_MOTHS_ROOT.rglob("*.jpg"))) \
            if PATH_EU_MOTHS_ROOT.exists() else 0
bg_count  = len(list(PATH_BACKGROUNDS.rglob("*.jpg"))) \
            if PATH_BACKGROUNDS.exists() else 0
 
print(f"\n  Aspilanta images : {asp_count}")
print(f"  EU Moths images  : {eum_count}")
print(f"  Background images: {bg_count}")
print("=" * 55)
print("\n✓ Cell A complete — run Cell B next")
 
 

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  KAGGLE CELL B (UPDATED) — Extract insects from both datasets              ║
# ║  Distractor source: your hand-picked folder (no index files needed)        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import io, gc, json, math, subprocess, sys, os, warnings
import cv2, numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display as ipy_display, Image as IPyImage

os.environ["REMBG_PROVIDERS"]       = "CPUExecutionProvider"
os.environ["ONNXRUNTIME_PROVIDERS"] = "CPUExecutionProvider"
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE                    = Path("/kaggle/working")
PATH_ASPILANTA_RAW      = Path(
    "/kaggle/input/datasets/your-kaggle-username/ashpilanta-3wtop/"
    "data gen-3/clean_insect(top)")
PATH_DISTRACTOR_RAW     = Path(
    "/kaggle/input/datasets/your-kaggle-username/distractor-insect-1/"
    "Distractor insect")                    # ← your hand-picked folder
PATH_CUTOUTS_ASPILANTA  = BASE / "cutouts" / "aspilanta"
PATH_CUTOUTS_DISTRACTOR = BASE / "cutouts" / "distractor"
OUTPUTS                 = BASE / "outputs"

for p in [PATH_CUTOUTS_ASPILANTA, PATH_CUTOUTS_DISTRACTOR, OUTPUTS]:
    p.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}


# ══════════════════════════════════════════════════════════════════════════════
#  DIAGNOSTIC
# ══════════════════════════════════════════════════════════════════════════════

def diagnose(label: str, path: Path):
    print(f"\n  [{label}]  {path}")
    if not path.exists():
        print(f"    ✗ PATH DOES NOT EXIST")
        return 0
    imgs = [f for f in path.rglob("*")
            if f.suffix.lower() in SUPPORTED_EXTS]
    print(f"    ✓ {len(imgs)} images found")
    for f in imgs[:5]:
        print(f"      {f.name}")
    if len(imgs) > 5:
        print(f"      ... and {len(imgs)-5} more")
    return len(imgs)

print("=" * 60)
print("DIAGNOSTIC")
print("=" * 60)
n_asp  = diagnose("Aspilanta raw",   PATH_ASPILANTA_RAW)
n_dist = diagnose("Distractor raw",  PATH_DISTRACTOR_RAW)
print("=" * 60)


# ══════════════════════════════════════════════════════════════════════════════
#  SHARED EXTRACTION UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def get_ram_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1e9
    except Exception:
        return -1.0


def colour_distance_mask(img_bgr, margin_frac=0.08):
    h, w   = img_bgr.shape[:2]
    margin = max(5, int(min(h, w) * margin_frac))
    border = np.vstack([
        img_bgr[:margin,  :].reshape(-1, 3),
        img_bgr[-margin:, :].reshape(-1, 3),
        img_bgr[:, :margin].reshape(-1, 3),
        img_bgr[:, -margin:].reshape(-1, 3),
    ]).astype(np.float32)
    bg_mean = border.mean(0)
    bg_std  = border.std(0) + 10.0
    dist    = np.sqrt(
        ((img_bgr.astype(np.float32) - bg_mean) ** 2
         / bg_std ** 2).sum(2))
    return ((dist > 2.5) * 255).astype(np.uint8)


def guided_filter_alpha(img_bgr, rough_mask, radius=4, eps=1e-3):
    guide = (cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
               .astype(np.float32) / 255.0)
    alpha = rough_mask.astype(np.float32) / 255.0
    try:
        gf      = cv2.ximgproc.createGuidedFilter(
            guide[:, :, np.newaxis], radius, eps)
        refined = gf.filter(alpha[:, :, np.newaxis])
        refined = np.clip(refined[:, :, 0], 0, 1)
    except Exception:
        refined = cv2.bilateralFilter(alpha, 9, 0.1, 15)
    return (refined * 255).astype(np.uint8)


def clean_mask(mask):
    k = np.ones((3, 3), np.uint8)
    m = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=3)
    m = cv2.morphologyEx(m,    cv2.MORPH_OPEN,  k, iterations=1)
    flood = m.copy()
    h, w  = m.shape
    fm    = np.zeros((h+2, w+2), np.uint8)
    cv2.floodFill(flood, fm, (0, 0), 255)
    m     = cv2.bitwise_or(m, cv2.bitwise_not(flood))
    n_lbl, labels, sts, _ = cv2.connectedComponentsWithStats(m)
    if n_lbl <= 1: return m
    areas = sts[1:, cv2.CC_STAT_AREA]
    if len(areas) == 0: return m
    best  = int(np.argmax(areas)) + 1
    out   = np.zeros_like(m)
    out[labels == best] = 255
    return out


def grabcut_with_hint(img_bgr, hint, iters=10):
    h, w    = img_bgr.shape[:2]
    gc_mask = np.zeros((h, w), np.uint8)
    gc_mask[hint > 0]  = cv2.GC_PR_FGD
    gc_mask[hint == 0] = cv2.GC_PR_BGD
    border  = max(3, min(h, w) // 20)
    gc_mask[:border,  :] = cv2.GC_BGD
    gc_mask[-border:, :] = cv2.GC_BGD
    gc_mask[:, :border]  = cv2.GC_BGD
    gc_mask[:, -border:] = cv2.GC_BGD
    k       = np.ones((5, 5), np.uint8)
    gc_mask[cv2.erode(hint, k, iterations=3) > 0] = cv2.GC_FGD
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(img_bgr, gc_mask, None, bgd, fgd,
                    iters, cv2.GC_INIT_WITH_MASK)
    except cv2.error:
        rect = (border, border, w-2*border, h-2*border)
        gc_mask2 = np.zeros((h, w), np.uint8)
        cv2.grabCut(img_bgr, gc_mask2, rect, bgd, fgd,
                    iters, cv2.GC_INIT_WITH_RECT)
        gc_mask = gc_mask2
    return np.where(
        (gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD),
        255, 0).astype(np.uint8)


def full_extraction(img_bgr):
    """GrabCut + guided filter foreground extraction. Returns (bgra, coverage)."""
    h, w     = img_bgr.shape[:2]
    col_mask = colour_distance_mask(img_bgr)
    k        = np.ones((5, 5), np.uint8)
    hint     = cv2.dilate(col_mask, k, iterations=2)
    binary   = grabcut_with_hint(img_bgr, hint, iters=10)
    alpha    = guided_filter_alpha(img_bgr, binary, radius=4, eps=1e-3)
    _, ab    = cv2.threshold(alpha, 30, 255, cv2.THRESH_BINARY)
    ab       = clean_mask(ab)
    alpha    = guided_filter_alpha(img_bgr, ab, radius=3, eps=5e-4)
    coverage = float((alpha > 30).sum() / (h * w))
    # Fallback if coverage is extreme
    if coverage < 0.03 or coverage > 0.93:
        border = max(3, min(h, w) // 20)
        gc_fb  = np.zeros((h, w), np.uint8)
        bgd    = np.zeros((1, 65), np.float64)
        fgd    = np.zeros((1, 65), np.float64)
        rect   = (border, border, w-2*border, h-2*border)
        cv2.grabCut(img_bgr, gc_fb, rect, bgd, fgd, 8, cv2.GC_INIT_WITH_RECT)
        fb       = np.where(
            (gc_fb==cv2.GC_FGD)|(gc_fb==cv2.GC_PR_FGD),
            255, 0).astype(np.uint8)
        alpha    = guided_filter_alpha(img_bgr, fb, radius=4, eps=1e-3)
        coverage = float((alpha > 30).sum() / (h * w))
    bgra          = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2BGRA)
    bgra[:, :, 3] = alpha
    return bgra, coverage


# ══════════════════════════════════════════════════════════════════════════════
#  PART A — ASPILANTA EXTRACTION  (rembg subprocess, GrabCut fallback)
# ══════════════════════════════════════════════════════════════════════════════

_REMBG_WORKER = """
import sys, os, io, gc
os.environ["REMBG_PROVIDERS"]       = "CPUExecutionProvider"
os.environ["ONNXRUNTIME_PROVIDERS"] = "CPUExecutionProvider"
import cv2, numpy as np
from PIL import Image

input_path  = sys.argv[1]
output_path = sys.argv[2]
img_bgr = cv2.imread(input_path)
if img_bgr is None: sys.exit(1)

coverage = 0.0
bgra     = None

try:
    from rembg import remove as rembg_remove, new_session
    session  = new_session("u2netp", providers=["CPUExecutionProvider"])
    with open(input_path, "rb") as f: raw = f.read()
    result   = rembg_remove(raw, session=session)
    img_pil  = Image.open(io.BytesIO(result)).convert("RGBA")
    bgra     = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGBA2BGRA)
    alpha    = np.array(img_pil.split()[3])
    coverage = float((alpha > 30).sum() / alpha.size)
    del session, raw, result; gc.collect()
except Exception as e:
    sys.stderr.write(f"rembg: {e}\\n")

if bgra is None or coverage < 0.05:
    h, w    = img_bgr.shape[:2]
    border  = max(3, min(h,w)//20)
    gc_mask = np.zeros((h,w), np.uint8)
    bgd     = np.zeros((1,65), np.float64)
    fgd     = np.zeros((1,65), np.float64)
    rect    = (border, border, w-2*border, h-2*border)
    cv2.grabCut(img_bgr, gc_mask, rect, bgd, fgd, 8, cv2.GC_INIT_WITH_RECT)
    m       = np.where((gc_mask==2)|(gc_mask==0),0,255).astype(np.uint8)
    bgra    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2BGRA)
    bgra[:,:,3] = m
    coverage    = float((m > 30).sum() / m.size)

import json
cv2.imwrite(output_path, bgra)
print(json.dumps({"coverage": round(coverage, 3)}))
"""

_WORKER_PATH = Path("/kaggle/working/_rembg_worker.py")
_WORKER_PATH.write_text(_REMBG_WORKER)


def extract_aspilanta_all(in_dir, out_dir, force=False):
    print("\n" + "═"*60)
    print("PART A — Aspilanta foreground extraction")
    print("═"*60)
    src_files = sorted(f for f in in_dir.iterdir()
                       if f.suffix.lower() in SUPPORTED_EXTS)
    print(f"  Source : {in_dir}  ({len(src_files)} files)\n")
    saved = []
    for i, src in enumerate(src_files, 1):
        out = out_dir / (src.stem + "_cutout.png")
        if not force and out.exists():
            alpha = np.array(Image.open(out).split()[-1])
            cov   = float((alpha > 30).sum() / alpha.size)
            saved.append(out)
            print(f"  ↷ [{i:02d}] {src.name[:50]}  ({cov:.0%}) cached")
            continue
        print(f"  [{i:02d}/{len(src_files)}] {src.name[:50]}",
              end=" ... ", flush=True)
        result = subprocess.run(
            [sys.executable, str(_WORKER_PATH), str(src), str(out)],
            capture_output=True, text=True, timeout=120,
            env={**os.environ,
                 "REMBG_PROVIDERS":       "CPUExecutionProvider",
                 "ONNXRUNTIME_PROVIDERS": "CPUExecutionProvider"})
        if result.returncode == 0 and result.stdout.strip():
            try:
                info = json.loads(result.stdout.strip().split("\n")[-1])
                cov  = info.get("coverage", 0)
                print(f"{'✓' if cov > 0.15 else '⚠'} cov={cov:.0%}")
                saved.append(out)
            except Exception:
                print("✓"); saved.append(out)
        else:
            print("⚠ subprocess failed → GrabCut fallback")
            img = cv2.imread(str(src))
            if img is not None:
                bgra, cov = full_extraction(img)
                if cov > 0.03:
                    cv2.imwrite(str(out), bgra)
                    saved.append(out)
        gc.collect()
    print(f"\n  ✓ {len(saved)}/{len(src_files)} Aspilanta cutouts saved")
    return saved


# ══════════════════════════════════════════════════════════════════════════════
#  PART B — DISTRACTOR EXTRACTION  (your hand-picked folder, no index files)
#
#  Scans PATH_DISTRACTOR_RAW recursively for all images.
#  Searches one level of subfolders too (e.g. if you organised by species).
#  Applies the same GrabCut + guided filter extraction as Aspilanta.
# ══════════════════════════════════════════════════════════════════════════════

def extract_distractors_from_folder(in_dir:  Path,
                                     out_dir: Path,
                                     force:   bool = False,
                                     max_images: int = 500) -> list:
    """
    Extract foreground from every image in your hand-picked distractor folder.

    Works with any folder structure:
      flat/        all images directly in the folder
      organised/   images in species subfolders  (e.g. Vanessa_atalanta/*.jpg)

    No index files, no labels.txt — just grab everything and extract.
    """
    print("\n" + "═"*60)
    print("PART B — Distractor foreground extraction")
    print("═"*60)
    print(f"  Source : {in_dir}")

    if not in_dir.exists():
        print(f"  ✗ Folder does not exist: {in_dir}")
        print("  Check PATH_DISTRACTOR_RAW")
        return []

    # Collect all image files recursively
    all_images = []
    for ext in SUPPORTED_EXTS:
        all_images.extend(in_dir.rglob(f"*{ext}"))
        all_images.extend(in_dir.rglob(f"*{ext.upper()}"))
    all_images = sorted(set(all_images))   # deduplicate

    print(f"  Found  : {len(all_images)} images")
    if not all_images:
        print("  ✗ No images found. Check the folder path.")
        return []

    # Limit if needed
    if len(all_images) > max_images:
        import random as _r
        _r.shuffle(all_images)
        all_images = all_images[:max_images]
        print(f"  Using  : {len(all_images)} (capped at {max_images})")

    out_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = []
    bad         = 0
    skipped     = 0

    for i, src in enumerate(all_images, 1):
        # Build output filename — include parent dir name for uniqueness
        parent = src.parent.name if src.parent != in_dir else "dist"
        stem   = f"{parent}_{src.stem}"[:80]   # cap length
        out    = out_dir / (stem + "_cutout.png")

        # Skip cached
        if not force and out.exists():
            try:
                alpha = np.array(Image.open(out).split()[-1])
                cov   = float((alpha > 30).sum() / alpha.size)
                if cov > 0.03:
                    saved_paths.append(out)
                    skipped += 1
                    continue
            except Exception:
                pass   # re-extract if file is corrupt

        # Load image
        img = cv2.imread(str(src))
        if img is None:
            bad += 1
            continue

        # Resize very large images (speeds up GrabCut dramatically)
        h, w = img.shape[:2]
        if max(h, w) > 1024:
            scale = 1024 / max(h, w)
            img   = cv2.resize(img,
                               (int(w*scale), int(h*scale)),
                               interpolation=cv2.INTER_AREA)

        # Extract foreground
        bgra, coverage = full_extraction(img)

        if coverage < 0.03:
            bad += 1
            if i % 20 == 0:
                print(f"  [{i:03d}/{len(all_images)}] "
                      f"✗ {src.name[:40]}  cov={coverage:.0%} too low")
            continue

        cv2.imwrite(str(out), bgra)
        saved_paths.append(out)

        # Progress every 20 images or at the end
        if i % 20 == 0 or i == len(all_images):
            tag = "✓" if coverage > 0.15 else "⚠"
            print(f"  [{i:03d}/{len(all_images)}] {tag} "
                  f"{src.parent.name}/{src.name[:30]:<30}  "
                  f"cov={coverage:.0%}  "
                  f"saved={len(saved_paths)}  "
                  f"RAM={get_ram_gb():.1f}GB")

        gc.collect()

    print(f"\n  ✓ {len(saved_paths)} distractor cutouts saved  "
          f"({skipped} cached,  {bad} failed/skipped)")
    return saved_paths


# ══════════════════════════════════════════════════════════════════════════════
#  PART C — VISUAL QC GRIDS
# ══════════════════════════════════════════════════════════════════════════════

def qc_grid(cutout_paths, title, out_path, cols=8, max_show=48):
    import random as _r
    files = list(cutout_paths)
    if len(files) > max_show:
        files = _r.sample(files, max_show)
    n = len(files)
    if n == 0:
        print(f"  No cutouts to display for: {title}")
        return
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols,
                              figsize=(cols*2, rows*2.5))
    axes = np.array(axes).flatten()
    for ax, p in zip(axes, files):
        try:
            img      = Image.open(p).convert("RGBA")
            alpha_np = np.array(img.split()[3])
            cov      = float((alpha_np > 30).sum() / alpha_np.size)
            wi, hi   = img.size
            sz       = max(4, min(wi, hi) // 12)
            checker  = np.zeros((hi, wi, 3), np.uint8)
            for r in range(0, hi, sz):
                for c in range(0, wi, sz):
                    v = 200 if (r//sz + c//sz) % 2 == 0 else 240
                    checker[r:r+sz, c:c+sz] = v
            comp = Image.fromarray(checker)
            comp.paste(img, (0, 0), img.split()[3])
            ax.imshow(np.array(comp))
            color = ("green"  if cov > 0.15 else
                     "orange" if cov > 0.05 else "red")
            ax.set_title(f"{p.stem[:12]}\n{cov:.0%}",
                         fontsize=5, color=color)
        except Exception:
            ax.set_facecolor("lightcoral")
            ax.set_title("err", fontsize=5)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    fig.legend(handles=[
        mpatches.Patch(color="green",  label=">15% good"),
        mpatches.Patch(color="orange", label="5-15% check"),
        mpatches.Patch(color="red",    label="<5% bad"),
    ], loc="lower center", ncol=3, fontsize=8)
    plt.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(str(out_path), dpi=100, bbox_inches="tight")
    plt.close()
    print(f"  QC grid saved: {out_path}")
    ipy_display(IPyImage(str(out_path)))


# ══════════════════════════════════════════════════════════════════════════════
#  RUN
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*60)
print("CELL B — Insect Extraction")
print("═"*60)

# Part A: Aspilanta
asp_cutouts = extract_aspilanta_all(
    PATH_ASPILANTA_RAW,
    PATH_CUTOUTS_ASPILANTA,
    force=False)

# Part B: Your hand-picked distractors
dist_cutouts = extract_distractors_from_folder(
    PATH_DISTRACTOR_RAW,
    PATH_CUTOUTS_DISTRACTOR,
    force=False,
    max_images=500)        # raise this if you have more images

# Part C: QC grids
print("\nGenerating QC grids...")
OUTPUTS.mkdir(exist_ok=True)

qc_grid(asp_cutouts,
        f"Aspilanta cutouts  ({len(asp_cutouts)} total)",
        OUTPUTS / "qc_aspilanta.png",  cols=8)

qc_grid(dist_cutouts,
        f"Distractor cutouts  ({len(dist_cutouts)} total)",
        OUTPUTS / "qc_distractor.png", cols=8)

# Save manifests for Cell C
asp_manifest  = OUTPUTS / "aspilanta_cutouts.json"
dist_manifest = OUTPUTS / "distractor_cutouts.json"   # ← new name

asp_manifest.write_text(
    json.dumps([str(p) for p in asp_cutouts],  indent=2))
dist_manifest.write_text(
    json.dumps([str(p) for p in dist_cutouts], indent=2))

print(f"\n{'='*60}")
print("CELL B COMPLETE")
print(f"{'='*60}")
print(f"  Aspilanta cutouts  : {len(asp_cutouts)}")
print(f"  Distractor cutouts : {len(dist_cutouts)}")
print(f"  Manifests saved    : {OUTPUTS}")
print("\n✓ Run Cell C to generate the synthetic dataset")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IMPROVED CELL C — High-Quality Synthetic Dataset Generator  v6            ║
# ║  Replaces the existing Cell C in master-thesis__6_.ipynb                  ║
# ║  Key improvements:                                                         ║
# ║    1. Multi-scale Poisson blending (seamlessClone MIXED + NORMAL fallback) ║
# ║    2. Per-insect color temperature & brightness jitter                     ║
# ║    3. Realistic motion blur on fast-moving insects                         ║
# ║    4. Pad-aware lighting gradient matching                                 ║
# ║    5. Improved overlap rejection with soft IoU                             ║
# ║    6. Sticky-trap glue sheen simulation                                   ║
# ║    7. Better YOLO label validation before writing                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import cv2
import numpy as np
import random
import json
import warnings
import os
import math

from pathlib import Path
from scipy.stats import truncnorm

import torch
import torch.nn.functional as TF
import torchvision.transforms.functional as TVF

os.environ["REMBG_PROVIDERS"]       = "CPUExecutionProvider"
os.environ["ONNXRUNTIME_PROVIDERS"] = "CPUExecutionProvider"
warnings.filterwarnings("ignore")

# ── Re-declare all paths (safe after restart) ─────────────────────────────────
BASE                    = Path("/kaggle/working")
PATH_BACKGROUNDS        = Path(
    "/kaggle/input/datasets/your-kaggle-username/ashpilanta-3wtop/"
    "data gen-3/Many Backgrounds")
PATH_OUTPUT             = BASE / "generated_dataset_v6"        # <-- v6 folder
OUTPUTS                 = BASE / "outputs"
asp_manifest            = OUTPUTS / "aspilanta_cutouts.json"
dist_manifest           = OUTPUTS / "distractor_cutouts.json"

# ── Configuration ─────────────────────────────────────────────────────────────
INSECT_MEAN_MM   = 5.0
INSECT_STD_MM    = 0.8
INSECT_MIN_MM    = 2.5      # smaller insects = harder = better training
INSECT_MAX_MM    = 8.0
PAD_WIDTH_MM     = 140.0
NUM_IMAGES       = 2000
TRAIN_RATIO      = 0.85
CLS_ASPILANTA    = 0
CLS_DISTRACTOR   = 1
NAME_ASPILANTA   = "Aspilanta"
NAME_DISTRACTOR  = "eu_moth"
MAX_ASPILANTA    = 10       # up from 8 — denser traps = better counting training
MAX_DISTRACTOR   = 5        # up from 3
MAX_ATTEMPTS     = 40
EXTRACT_PAD      = True
PAD_MIN_FRAC     = 0.05
PAD_MAX_FRAC     = 0.95

# Blur config (organic post-paste blur)
ORGANIC_K        = (3, 3)
ORGANIC_S        = 0.5

# Sheen probability (sticky pad glue shimmer)
SHEEN_PROB       = 0.30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[GPU] Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"[GPU] Name   : {torch.cuda.get_device_name(0)}")


# ══════════════════════════════════════════════════════════════════════════════
#  LOAD CUTOUTS FROM MANIFESTS
# ══════════════════════════════════════════════════════════════════════════════

def load_bgra_images(paths: list, label: str) -> list:
    imgs = []
    for p in paths:
        img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        if img is None:
            continue
        if img.ndim == 3 and img.shape[2] == 4:
            imgs.append(img)
        elif img.ndim == 3 and img.shape[2] == 3:
            bgra        = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
            bgra[:,:,3] = 255
            imgs.append(bgra)
    print(f"  {label:<25} {len(imgs)} cutouts loaded")
    return imgs


# ══════════════════════════════════════════════════════════════════════════════
#  GPU RESIZE
# ══════════════════════════════════════════════════════════════════════════════

def bgr_to_tensor(img):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return (torch.from_numpy(rgb.copy())
            .permute(2,0,1).float().div(255.0)
            .unsqueeze(0).to(DEVICE))

def tensor_to_bgr(t):
    arr = (t.squeeze(0).permute(1,2,0)
            .clamp(0,1).mul(255).byte().cpu().numpy())
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

def gpu_resize_bgra(img, new_w, new_h):
    try:
        t   = bgr_to_tensor(img[:,:,:3])
        r   = TF.interpolate(t, size=(new_h, new_w),
                             mode='bilinear', align_corners=False)
        bgr = tensor_to_bgr(r)
    except Exception:
        bgr = cv2.resize(img[:,:,:3], (new_w, new_h),
                         interpolation=cv2.INTER_AREA)
    alpha       = cv2.resize(img[:,:,3], (new_w, new_h),
                             interpolation=cv2.INTER_AREA)
    bgra        = cv2.cvtColor(bgr, cv2.COLOR_BGR2BGRA)
    bgra[:,:,3] = alpha
    return bgra


# ══════════════════════════════════════════════════════════════════════════════
#  SIZE SAMPLING  (Truncated Gaussian 2.5–8 mm)
# ══════════════════════════════════════════════════════════════════════════════

def sample_size_px(pad_width_px):
    px_per_mm = pad_width_px / PAD_WIDTH_MM
    a = (INSECT_MIN_MM - INSECT_MEAN_MM) / INSECT_STD_MM
    b = (INSECT_MAX_MM - INSECT_MEAN_MM) / INSECT_STD_MM
    mm = truncnorm.rvs(a, b, loc=INSECT_MEAN_MM, scale=INSECT_STD_MM)
    return max(4, int(round(mm * px_per_mm)))


# ══════════════════════════════════════════════════════════════════════════════
#  STICKY PAD DETECTION & EXTRACTION  (unchanged from v5)
# ══════════════════════════════════════════════════════════════════════════════

def detect_pad_bbox(bg):
    hsv  = cv2.cvtColor(bg, cv2.COLOR_BGR2HSV)
    H, W = bg.shape[:2]
    mask = (
        cv2.inRange(hsv, ( 18, 70,  80), ( 38,255,255)) |
        cv2.inRange(hsv, (  0,  0, 170), (180, 50,255)) |
        cv2.inRange(hsv, ( 90, 50,  60), (135,255,255)) |
        cv2.inRange(hsv, ( 40, 50,  50), ( 90,255,220))
    )
    k    = cv2.getStructuringElement(cv2.MORPH_RECT,(20,20))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k, iterations=2)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return (0,0,W,H), False
    largest = max(cnts, key=cv2.contourArea)
    area    = cv2.contourArea(largest)
    if not (H*W*PAD_MIN_FRAC < area < H*W*PAD_MAX_FRAC):
        return (0,0,W,H), False
    x,y,w,h = cv2.boundingRect(largest)
    inset    = max(8, min(w,h)//25)
    return (x+inset, y+inset, w-2*inset, h-2*inset), True


def extract_clean_pad(bg):
    (px,py,pw,ph), found = detect_pad_bbox(bg)
    if not found or pw<50 or ph<50:
        H,W   = bg.shape[:2]
        mx,my = int(W*0.10), int(H*0.10)
        return (bg[my:my+int(H*0.80), mx:mx+int(W*0.80)].copy(),
                (mx,my,int(W*0.80),int(H*0.80)), False)
    canvas = bg[py:py+ph, px:px+pw].copy()
    gray   = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    blur   = cv2.GaussianBlur(gray,(51,51),0)
    diff   = cv2.subtract(blur,gray)
    _,dm   = cv2.threshold(diff,18,255,cv2.THRESH_BINARY)
    k2     = np.ones((3,3),np.uint8)
    dm     = cv2.morphologyEx(dm,cv2.MORPH_CLOSE,k2,iterations=2)
    dm     = cv2.morphologyEx(dm,cv2.MORPH_OPEN, k2,iterations=1)
    n_lbl,labels,sts,_ = cv2.connectedComponentsWithStats(dm)
    clean  = np.zeros_like(dm)
    for lid in range(1,n_lbl):
        if 8 <= sts[lid,cv2.CC_STAT_AREA] <= 2000:
            clean[labels==lid] = 255
    if clean.any():
        clean  = cv2.dilate(clean,k2,iterations=2)
        canvas = cv2.inpaint(canvas,clean,5,cv2.INPAINT_TELEA)
    return canvas,(px,py,pw,ph),True


# ══════════════════════════════════════════════════════════════════════════════
#  NEW: PAD-AWARE LIGHTING GRADIENT MATCHING
#  Samples local background mean color around the paste position
#  and slightly shifts the insect hue toward it — makes pasted insects
#  look more naturally lit and less "copy-pasted".
# ══════════════════════════════════════════════════════════════════════════════

def match_local_lighting(ins_bgr: np.ndarray,
                          bg: np.ndarray,
                          x: int, y: int, w: int, h: int,
                          strength: float = 0.20) -> np.ndarray:
    """Blend insect color tone toward local background mean."""
    bh, bw = bg.shape[:2]
    # Sample a slightly larger patch
    margin  = max(10, w // 2)
    rx1     = max(0, x - margin)
    ry1     = max(0, y - margin)
    rx2     = min(bw, x + w + margin)
    ry2     = min(bh, y + h + margin)
    patch   = bg[ry1:ry2, rx1:rx2]
    if patch.size == 0:
        return ins_bgr
    local_mean = patch.mean(axis=(0,1))          # BGR means
    ins_f      = ins_bgr.astype(np.float32)
    blended    = ins_f * (1 - strength) + local_mean * strength
    return np.clip(blended, 0, 255).astype(np.uint8)


# ══════════════════════════════════════════════════════════════════════════════
#  NEW: PER-INSECT COLOR / BRIGHTNESS JITTER
# ══════════════════════════════════════════════════════════════════════════════

def color_jitter_insect(ins_bgr: np.ndarray) -> np.ndarray:
    """Random brightness, contrast, and slight hue shift per insect."""
    out   = ins_bgr.astype(np.float32)
    alpha = random.uniform(0.80, 1.20)    # brightness
    beta  = random.uniform(-15,  15)      # contrast offset
    out   = np.clip(out * alpha + beta, 0, 255)

    # Slight hue shift in HSV
    if random.random() < 0.50:
        hsv       = cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,0] = (hsv[:,:,0] + random.uniform(-8, 8)) % 180
        hsv[:,:,1] = np.clip(hsv[:,:,1] * random.uniform(0.85, 1.15), 0, 255)
        out        = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR).astype(np.float32)

    return np.clip(out, 0, 255).astype(np.uint8)


# ══════════════════════════════════════════════════════════════════════════════
#  NEW: MOTION BLUR  (applied to ~15% of insects to simulate live trap)
# ══════════════════════════════════════════════════════════════════════════════

def apply_motion_blur(img_bgr: np.ndarray, max_kernel: int = 5) -> np.ndarray:
    k    = random.choice([3, 5]) if max_kernel >= 5 else 3
    ang  = random.uniform(0, 180)
    M    = cv2.getRotationMatrix2D((k//2, k//2), ang, 1)
    kern = np.zeros((k, k), np.float32)
    kern[k//2, :] = 1.0
    kern          = cv2.warpAffine(kern, M, (k, k))
    kern         /= kern.sum() + 1e-9
    return cv2.filter2D(img_bgr, -1, kern)


# ══════════════════════════════════════════════════════════════════════════════
#  CONTACT SHADOW  (unchanged from v5 — works well)
# ══════════════════════════════════════════════════════════════════════════════

def add_shadow(bg, x, y, w, h, alpha_mask):
    out      = bg.copy()
    dx       = random.randint(1, max(2, w//8))
    dy       = random.randint(1, max(2, h//8))
    strength = random.uniform(0.06, 0.18)
    shadow   = cv2.dilate(alpha_mask, np.ones((5,5),np.uint8), iterations=1)
    shadow   = cv2.GaussianBlur(shadow, (0,0), float(max(1, w//6)))
    shadow   = shadow.astype(np.float32)/255.0 * strength
    sx1,sy1  = max(0,x+dx), max(0,y+dy)
    sx2,sy2  = min(bg.shape[1],x+dx+w), min(bg.shape[0],y+dy+h)
    smx1,smy1 = sx1-(x+dx), sy1-(y+dy)
    smx2,smy2 = smx1+(sx2-sx1), smy1+(sy2-sy1)
    if (sx2>sx1 and sy2>sy1
            and smx2<=shadow.shape[1] and smy2<=shadow.shape[0]):
        roi = out[sy1:sy2,sx1:sx2].astype(np.float32)
        sm  = shadow[smy1:smy2,smx1:smx2]
        if roi.shape[:2] == sm.shape:
            out[sy1:sy2,sx1:sx2] = np.clip(
                roi*(1-sm[...,np.newaxis]),0,255).astype(np.uint8)
    return out


# ══════════════════════════════════════════════════════════════════════════════
#  NEW: STICKY-TRAP GLUE SHEEN SIMULATION
#  Adds semi-transparent highlight ellipses to simulate the reflective
#  glue surface that makes real sticky-trap photos look different from
#  plain paper composites.
# ══════════════════════════════════════════════════════════════════════════════

def add_sheen(canvas: np.ndarray) -> np.ndarray:
    """Add a subtle specular highlight blob to the canvas."""
    h, w   = canvas.shape[:2]
    overlay = canvas.copy()
    cx      = random.randint(w//6, 5*w//6)
    cy      = random.randint(h//6, 5*h//6)
    rx      = random.randint(w//8, w//3)
    ry      = random.randint(h//8, h//3)
    angle   = random.randint(0, 180)
    # Draw a soft white ellipse
    cv2.ellipse(overlay, (cx,cy), (rx,ry), angle, 0, 360,
                (255,255,255), -1)
    alpha_sheen = random.uniform(0.03, 0.09)
    return cv2.addWeighted(canvas, 1 - alpha_sheen, overlay, alpha_sheen, 0)


# ══════════════════════════════════════════════════════════════════════════════
#  IMPROVED PASTE INSECT PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def paste_insect(background, insect_bgra, target_w, pos):
    """
    Improved 8-step paste pipeline:
      1. GPU resize
      2. Color/brightness jitter (NEW)
      3. Local lighting match (NEW)
      4. Organic blur
      5. Optional motion blur (NEW)
      6. 1px alpha anti-alias
      7. Contact shadow
      8. Alpha composite + Poisson seamlessClone (MIXED → NORMAL fallback)
    """
    bg_h, bg_w    = background.shape[:2]
    orig_h, orig_w = insect_bgra.shape[:2]
    scale   = target_w / max(orig_w, 1)
    new_w   = max(8, int(orig_w*scale))
    new_h   = max(8, int(orig_h*scale))
    if new_w >= bg_w or new_h >= bg_h:
        return background, None

    resized   = gpu_resize_bgra(insect_bgra, new_w, new_h)
    ins_bgr   = resized[:,:,:3].copy()
    ins_alpha = resized[:,:,3].copy()

    # Step 2 — Color jitter
    ins_bgr = color_jitter_insect(ins_bgr)

    x, y = pos
    x    = max(0, min(x, bg_w - new_w))
    y    = max(0, min(y, bg_h - new_h))
    x2, y2 = x + new_w, y + new_h
    if background[y:y2, x:x2].shape[:2] != (new_h, new_w):
        return background, None

    # Step 3 — Local lighting match
    ins_bgr = match_local_lighting(ins_bgr, background, x, y, new_w, new_h,
                                    strength=random.uniform(0.10, 0.25))

    # Step 4 — Organic blur (same as v5)
    ins_bgr = cv2.GaussianBlur(ins_bgr, ORGANIC_K, ORGANIC_S)

    # Step 5 — Optional motion blur
    if random.random() < 0.15:
        ins_bgr = apply_motion_blur(ins_bgr, max_kernel=5)

    # Step 6 — Alpha anti-alias
    _, ins_alpha = cv2.threshold(ins_alpha, 30, 255, cv2.THRESH_BINARY)
    ins_alpha    = cv2.GaussianBlur(ins_alpha, (3,3), 0.5)

    # Step 7 — Contact shadow
    bg_work  = add_shadow(background, x, y, new_w, new_h, ins_alpha)
    alpha_f  = ins_alpha.astype(np.float32) / 255.0
    roi_f    = bg_work[y:y2, x:x2].astype(np.float32)
    ins_f    = ins_bgr.astype(np.float32)
    blended  = ins_f * alpha_f[...,np.newaxis] + roi_f * (1 - alpha_f[...,np.newaxis])
    result   = bg_work.copy()
    result[y:y2, x:x2] = np.clip(blended, 0, 255).astype(np.uint8)

    # Step 8 — Poisson clone (MIXED → NORMAL → skip)
    if new_w > 25 and new_h > 25:
        try:
            _, pmask = cv2.threshold(ins_alpha, 50, 255, cv2.THRESH_BINARY)
            if pmask.sum() > 0:
                center = (x + new_w//2, y + new_h//2)
                try:
                    result = cv2.seamlessClone(
                        result[y:y2, x:x2], result, pmask,
                        center, cv2.MIXED_CLONE)
                except cv2.error:
                    result = cv2.seamlessClone(
                        result[y:y2, x:x2], result, pmask,
                        center, cv2.NORMAL_CLONE)
        except Exception:
            pass

    bbox = ((x + new_w/2) / bg_w,
            (y + new_h/2) / bg_h,
            new_w / bg_w,
            new_h / bg_h)
    return result, bbox


# ══════════════════════════════════════════════════════════════════════════════
#  BACKGROUND AUGMENTATION  (same as v5 + stronger random crop)
# ══════════════════════════════════════════════════════════════════════════════

def augment_bg(bg):
    bg    = bg.copy()
    alpha = random.uniform(0.85, 1.15)
    beta  = random.uniform(-15, 15)
    bg    = np.clip(bg.astype(np.float32)*alpha+beta, 0, 255).astype(np.uint8)
    if random.random() < 0.40:
        noise = np.random.normal(0, random.uniform(2, 7), bg.shape).astype(np.float32)
        bg    = np.clip(bg.astype(np.float32)+noise, 0, 255).astype(np.uint8)
    # Vignette (new)
    if random.random() < 0.30:
        H, W = bg.shape[:2]
        Y, X = np.ogrid[:H, :W]
        cx, cy = W/2, H/2
        dist   = np.sqrt(((X-cx)/cx)**2 + ((Y-cy)/cy)**2)
        vign   = 1.0 - np.clip(dist*random.uniform(0.25, 0.55), 0, 0.35)
        bg     = np.clip(bg.astype(np.float32)*vign[...,np.newaxis], 0, 255).astype(np.uint8)
    if random.random() < 0.25:
        H, W   = bg.shape[:2]
        gx,gy  = random.randint(0,W), random.randint(0,H)
        gr     = random.randint(15, min(W,H)//4)
        m      = np.zeros((H,W), np.float32)
        cv2.circle(m, (gx,gy), gr, 1.0, -1)
        m      = cv2.GaussianBlur(m, (0,0), max(1, gr//2))
        bg     = np.clip(bg.astype(np.float32) +
                         m[...,np.newaxis]*255*random.uniform(0.04, 0.14),
                         0, 255).astype(np.uint8)
    return bg


# ══════════════════════════════════════════════════════════════════════════════
#  NEW: SOFT IoU overlap check (rejects placements that overlap > threshold)
# ══════════════════════════════════════════════════════════════════════════════

def soft_iou(b1, b2):
    """b = [x1,y1,x2,y2]"""
    ix1 = max(b1[0], b2[0]); iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2]); iy2 = min(b1[3], b2[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0:
        return 0.0
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    union = a1 + a2 - inter
    return inter / max(union, 1)


# ══════════════════════════════════════════════════════════════════════════════
#  NEW: YOLO LABEL VALIDATOR
#  Ensures all bounding box values are in [0,1] range before writing.
# ══════════════════════════════════════════════════════════════════════════════

def validate_label(cls_id, cx, cy, bw, bh,
                   min_size=0.005, max_size=0.90) -> bool:
    if not (0.0 < cx < 1.0 and 0.0 < cy < 1.0):
        return False
    if not (min_size < bw < max_size and min_size < bh < max_size):
        return False
    # Check corners are in frame
    x1 = cx - bw/2; y1 = cy - bh/2
    x2 = cx + bw/2; y2 = cy + bh/2
    if x1 < -0.05 or y1 < -0.05 or x2 > 1.05 or y2 > 1.05:
        return False
    return True


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN GENERATOR  (v6)
# ══════════════════════════════════════════════════════════════════════════════

def generate():
    print("=" * 68)
    print("ASPILANTA SYNTHETIC DATASET GENERATOR  v6")
    print("Improvements: lighting match, color jitter, motion blur,")
    print("              sheen, soft IoU, label validation, vignette")
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print("=" * 68)

    if not asp_manifest.exists() or not dist_manifest.exists():
        raise RuntimeError(
            "Manifests not found — run Cell B first.\n"
            f"  Expected: {asp_manifest}\n"
            f"           {dist_manifest}")

    asp_paths = json.loads(asp_manifest.read_text())
    eu_paths  = json.loads(dist_manifest.read_text())

    print("\nLoading cutouts...")
    aspilanta   = load_bgra_images(asp_paths, "Aspilanta")
    distractors = load_bgra_images(eu_paths,  "EU Moths")

    if not aspilanta:
        raise RuntimeError("No Aspilanta cutouts found. Re-run Cell B.")
    if not distractors:
        print("⚠ No EU Moths cutouts found. Distractors will be skipped.")

    # Backgrounds
    print("\nLoading backgrounds...")
    bg_exts = ('*.jpg','*.JPG','*.jpeg','*.JPEG','*.png','*.PNG')
    bg_paths_all = []
    for ext in bg_exts:
        bg_paths_all.extend(PATH_BACKGROUNDS.glob(ext))
        bg_paths_all.extend(PATH_BACKGROUNDS.glob(f'**/{ext}'))
    bg_paths_all = list(set(bg_paths_all))
    if not bg_paths_all:
        raise RuntimeError(f"No backgrounds in {PATH_BACKGROUNDS}")
    print(f"  {len(bg_paths_all)} backgrounds found")

    # Output dirs
    for split in ["train","val"]:
        Path(PATH_OUTPUT,"images",split).mkdir(parents=True,exist_ok=True)
        Path(PATH_OUTPUT,"labels",split).mkdir(parents=True,exist_ok=True)

    stats = {
        "total":0,"with_aspilanta":0,"with_distractor":0,
        "empty":0,"n_aspilanta":0,"n_distractor":0,
        "pads_found":0,"pads_fallback":0,
        "labels_rejected":0,"size_px_list":[],
    }

    print(f"\nGenerating {NUM_IMAGES} images...\n")
    generated = 0
    attempts  = 0

    while generated < NUM_IMAGES:
        attempts += 1
        if attempts > NUM_IMAGES * 6:
            print("Warning: stopping early (too many attempts)")
            break

        split  = ("train" if generated < int(NUM_IMAGES*TRAIN_RATIO)
                  else "val")
        bg_raw = cv2.imread(str(random.choice(bg_paths_all)))
        if bg_raw is None:
            continue
        bg_raw = augment_bg(bg_raw)

        if EXTRACT_PAD:
            canvas,(_,_,cw,ch),pad_found = extract_clean_pad(bg_raw)
            stats["pads_found" if pad_found else "pads_fallback"] += 1
        else:
            canvas = bg_raw.copy()
            cw, ch = bg_raw.shape[1], bg_raw.shape[0]

        if cw < 50 or ch < 50:
            continue

        # Optional sheen on the pad surface
        if random.random() < SHEEN_PROB:
            canvas = add_sheen(canvas)

        r     = random.random()
        n_asp = (0                       if r < 0.10 else
                 random.randint(1,3)     if r < 0.45 else
                 random.randint(4,7)     if r < 0.80 else
                 random.randint(8, MAX_ASPILANTA))
        n_dist = (0 if not distractors else
                  random.choices([0,1,2,3,MAX_DISTRACTOR],
                                  weights=[0.50,0.22,0.14,0.09,0.05])[0])

        placed = []   # list of [x1,y1,x2,y2]
        labels = []

        # ── Aspilanta ──────────────────────────────────────────────────────────
        for _ in range(n_asp):
            src = random.choice(aspilanta)
            for _ in range(MAX_ATTEMPTS):
                tw  = sample_size_px(cw)
                tw  = max(4, min(tw, cw // 3))
                ang = random.uniform(0, 360)
                M   = cv2.getRotationMatrix2D(
                    (src.shape[1]//2, src.shape[0]//2), ang, 1.0)
                rot = cv2.warpAffine(
                    src, M, (src.shape[1], src.shape[0]),
                    flags=cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_CONSTANT,
                    borderValue=(0,0,0,0))
                margin = max(5, tw)
                if cw <= 2*margin or ch <= 2*margin:
                    continue
                px = random.randint(margin, cw - margin)
                py = random.randint(margin, ch - margin)
                th  = int(tw * rot.shape[0] / max(rot.shape[1], 1))
                box = [px, py, px+tw, py+th]

                # Soft IoU overlap check (max 30% overlap allowed)
                if any(soft_iou(box, pb) > 0.30 for pb in placed):
                    continue

                canvas, bbox = paste_insect(canvas, rot, tw, (px, py))
                if bbox is None:
                    continue

                # Validate label before keeping
                if not validate_label(CLS_ASPILANTA, *bbox):
                    stats["labels_rejected"] += 1
                    continue

                placed.append(box)
                labels.append(
                    f"{CLS_ASPILANTA} "
                    f"{bbox[0]:.6f} {bbox[1]:.6f} "
                    f"{bbox[2]:.6f} {bbox[3]:.6f}")
                stats["n_aspilanta"]    += 1
                stats["size_px_list"].append(tw)
                break

        # ── EU Moths distractors ───────────────────────────────────────────────
        for _ in range(n_dist):
            src  = random.choice(distractors)
            dw   = int(sample_size_px(cw) * random.uniform(0.8, 2.5))
            dw   = max(10, min(dw, cw // 3))
            for _ in range(MAX_ATTEMPTS // 2):
                margin = max(5, dw)
                if cw <= 2*margin or ch <= 2*margin:
                    break
                px  = random.randint(margin, cw - margin)
                py  = random.randint(margin, ch - margin)
                box = [px, py, px+dw, py+dw]
                if any(soft_iou(box, pb) > 0.30 for pb in placed):
                    continue
                ang = random.uniform(0, 360)
                M   = cv2.getRotationMatrix2D(
                    (src.shape[1]//2, src.shape[0]//2), ang, 1.0)
                rot = cv2.warpAffine(
                    src, M, (src.shape[1], src.shape[0]),
                    flags=cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_CONSTANT,
                    borderValue=(0,0,0,0))
                canvas, bbox = paste_insect(canvas, rot, dw, (px, py))
                if bbox is None:
                    continue
                if not validate_label(CLS_DISTRACTOR, *bbox):
                    stats["labels_rejected"] += 1
                    continue
                placed.append(box)
                labels.append(
                    f"{CLS_DISTRACTOR} "
                    f"{bbox[0]:.6f} {bbox[1]:.6f} "
                    f"{bbox[2]:.6f} {bbox[3]:.6f}")
                stats["n_distractor"] += 1
                break

        has_insects = bool(labels)
        allow_empty = (generated % 10 == 0)   # 10% empty — matches real traps
        if not has_insects and not allow_empty:
            continue

        fname = f"synth_{generated:06d}"
        cv2.imwrite(
            str(Path(PATH_OUTPUT)/"images"/split/f"{fname}.jpg"),
            canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        (Path(PATH_OUTPUT)/"labels"/split/f"{fname}.txt").write_text(
            '\n'.join(labels))

        stats["total"]           += 1
        stats["with_aspilanta"]  += int(any(
            l.startswith(str(CLS_ASPILANTA)) for l in labels))
        stats["with_distractor"] += int(any(
            l.startswith(str(CLS_DISTRACTOR)) for l in labels))
        stats["empty"]           += int(not labels)
        generated                += 1

        if generated % 200 == 0:
            msz = (np.mean(stats["size_px_list"])
                   if stats["size_px_list"] else 0)
            print(f"  [{generated:4d}/{NUM_IMAGES}]  "
                  f"Asp={stats['n_aspilanta']}  "
                  f"Dist={stats['n_distractor']}  "
                  f"Pads={stats['pads_found']}  "
                  f"MeanPx={msz:.1f}  "
                  f"Rejected={stats['labels_rejected']}")

    # dataset.yaml
    mean_mm = 5.0
    if stats["size_px_list"]:
        try:
            bg_s = cv2.imread(str(bg_paths_all[0]))
            _,(_,_,apw,_),_ = extract_clean_pad(bg_s)
        except Exception:
            apw = 600
        mean_mm = np.mean(stats["size_px_list"]) / max(apw / PAD_WIDTH_MM, 1)

    yaml_txt = f"""# YOLOv8/YOLO11 — Aspilanta Detection  v6
path: {Path(PATH_OUTPUT).absolute()}
train: images/train
val:   images/val
nc: 2
names:
  0: {NAME_ASPILANTA}
  1: {NAME_DISTRACTOR}
# v6 improvements: lighting match, color jitter, motion blur,
#                  sheen, soft-IoU, label validation, vignette
# Size: Truncated Gaussian [{INSECT_MIN_MM},{INSECT_MAX_MM}]mm
# Generated mean ≈ {mean_mm:.2f} mm  |  GPU: {DEVICE}
"""
    (Path(PATH_OUTPUT)/'dataset.yaml').write_text(yaml_txt)

    s = {k:v for k,v in stats.items() if k != "size_px_list"}
    if stats["size_px_list"]:
        s["mean_px"]   = round(float(np.mean(stats["size_px_list"])),2)
        s["median_px"] = round(float(np.median(stats["size_px_list"])),2)
        s["std_px"]    = round(float(np.std(stats["size_px_list"])),2)
        s["mean_mm"]   = round(mean_mm, 2)
    (Path(PATH_OUTPUT)/'stats.json').write_text(json.dumps(s, indent=2))

    print(f"\n{'='*68}")
    print("COMPLETE — v6 (High-Quality Synthetic Generator)")
    print(f"{'='*68}")
    print(f"  Total             : {stats['total']}")
    print(f"  With Aspilanta    : {stats['with_aspilanta']}")
    print(f"  With EU Moths     : {stats['with_distractor']}")
    print(f"  Empty             : {stats['empty']}")
    print(f"  Labels rejected   : {stats['labels_rejected']}")
    print(f"  Aspilanta placed  : {stats['n_aspilanta']}")
    print(f"  Distractors       : {stats['n_distractor']}")
    print(f"  Pads detected     : {stats['pads_found']}")
    if stats["size_px_list"]:
        print(f"  Mean size         : {s['mean_mm']:.2f} mm ({s['mean_px']:.1f} px)")
    print(f"  GPU               : {DEVICE}")
    print(f"\n  Output            : {PATH_OUTPUT}/")
    print(f"  Train command     :")
    print(f"    yolo train model=yolov8s.pt "
          f"data={PATH_OUTPUT}/dataset.yaml epochs=100 imgsz=640")


# Verification grid (identical to v5)
def verify(n=16):
    img_dir = Path(PATH_OUTPUT)/"images"/"train"
    lbl_dir = Path(PATH_OUTPUT)/"labels"/"train"
    imgs    = sorted(img_dir.glob("*.jpg"))
    if not imgs:
        print("No images found")
        return
    sample  = random.sample(imgs, min(n, len(imgs)))
    cols    = 4
    rows    = (len(sample)+cols-1)//cols
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows*4))
    axes    = np.array(axes).flatten()
    for ax, img_path in zip(axes, sample):
        img = cv2.imread(str(img_path))
        H, W = img.shape[:2]
        lbl = lbl_dir / (img_path.stem+".txt")
        na = nd = 0
        if lbl.exists():
            for line in lbl.read_text().strip().split('\n'):
                if not line:
                    continue
                parts = line.split()
                cid   = int(parts[0])
                cx,cy,bw,bh = map(float, parts[1:])
                x1=int((cx-bw/2)*W); y1=int((cy-bh/2)*H)
                x2=int((cx+bw/2)*W); y2=int((cy+bh/2)*H)
                cv2.rectangle(img,(x1,y1),(x2,y2),
                              (0,220,0) if cid==CLS_ASPILANTA else (0,60,220),2)
                if cid == CLS_ASPILANTA: na+=1
                else:                    nd+=1
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Asp={na}  EUMoth={nd}", fontsize=8)
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    plt.suptitle(
        "v6 QC — High-Quality Composite\n"
        "Green=Aspilanta (0)   Blue=EU Moth (1)", fontsize=10)
    plt.tight_layout()
    out = Path(PATH_OUTPUT)/"verification_grid_v6.png"
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"QC grid: {out}")
    try:
        from IPython.display import display, Image as IPyImage
        display(IPyImage(str(out)))
    except ImportError:
        pass


# ── RUN ───────────────────────────────────────────────────────────────────────
generate()
verify(n=16)

# End of synthetic data generation***

# **SECTION 3:**

In [ ]:
# ============================================================
# AFTER SYNTHETIC DATA GENERATION
# Full Kaggle code for thesis experiments
# Project: Few-shot Ashpilanta detection on sticky traps
# Experiments:
#   A) YOLOv8 baseline: original 3 classes + few real Ashpilanta
#   B) YOLOv8 proposed: original 3 classes + few real Ashpilanta + synthetic sticky-trap data
#   C) Optional VLM baseline: CLIP few-shot classification on validation object crops
# ============================================================

# ============================================================
# CELL 1 — Install and imports
# ============================================================
# Run in Kaggle once. If already installed, this is safe.


from pathlib import Path
import os
import shutil
import random
import json
import math
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

warnings.filterwarnings("ignore")

BASE = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ============================================================
# COPY SYNTHETIC DATASET V6 FROM KAGGLE INPUT TO WORKING
# Put this AFTER imports and BEFORE dataset preparation/training
# ============================================================

from pathlib import Path
import shutil

BASE = Path("/kaggle/working")

SRC_SYNTH = Path("/kaggle/input/datasets/your-kaggle-username/synthetic-gen-data/generated_dataset_v6")
DST_SYNTH = BASE / "generated_dataset_v6"

# Check source exists
if not SRC_SYNTH.exists():
    raise FileNotFoundError(
        f"Source synthetic dataset not found:\n{SRC_SYNTH}\n\n"
        "Check the exact Kaggle input path from the left dataset panel."
    )

# Remove old working copy if it exists
if DST_SYNTH.exists():
    shutil.rmtree(DST_SYNTH)
    print("Removed old working folder:", DST_SYNTH)

# Copy dataset
shutil.copytree(SRC_SYNTH, DST_SYNTH)

print("Synthetic dataset copied successfully.")
print("From:", SRC_SYNTH)
print("To:", DST_SYNTH)

# Quick verification
image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    image_files.extend(DST_SYNTH.rglob(ext))

label_files = list(DST_SYNTH.rglob("*.txt"))

print("\nVerification:")
print("Images:", len(image_files))
print("Labels:", len(label_files))
print("dataset.yaml exists:", (DST_SYNTH / "dataset.yaml").exists())
print("images/train exists:", (DST_SYNTH / "images" / "train").exists())
print("images/val exists:", (DST_SYNTH / "images" / "val").exists())
print("labels/train exists:", (DST_SYNTH / "labels" / "train").exists())
print("labels/val exists:", (DST_SYNTH / "labels" / "val").exists())

In [ ]:
# ==============================================================================
#  REPLACEMENT CELL 2 — CONFIG
#  Replace the existing CELL 2 block with this one.
# ==============================================================================
 
from pathlib import Path
import os
import shutil
import random
import json
import math
import warnings
 
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
 
warnings.filterwarnings("ignore")
 
BASE  = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
 
# ── Class definitions ──────────────────────────────────────────────────────────
# Experiment 1 (base 3-class model)
CLASSES_3 = ["MR", "NC", "WF"]
 
# Experiments 2 & 3 (4-class model with Ashpilanta as class 3)
CLASSES_4  = ["MR", "NC", "WF", "Ashpilanta"]
AP_CLASS_ID = 3
 
# ── Few-shot setting ───────────────────────────────────────────────────────────
# None = use ALL real Ashpilanta images available (recommended for Exp 3)
# 5 or 10 = strict few-shot ablation
K_REAL_SHOTS = None
 
# ── Data split ratios ──────────────────────────────────────────────────────────
REAL_VAL_RATIO  = 0.20
SYNTH_VAL_RATIO = 0.20
RANDOM_SEED     = 42
 
# ── Training hyperparameters (tuned for small-insect detection) ────────────────
# Experiment 1 — 3-class base model
EXP1_EPOCHS       = 100
EXP1_IMGSZ        = 640
EXP1_BATCH        = 16
EXP1_LR0          = 0.01
EXP1_LRF          = 0.01        # final LR = LR0 * LRF
EXP1_WARMUP_EPOCHS= 3
EXP1_PATIENCE     = 20
EXP1_FREEZE       = 0           # no freeze: train all layers from scratch
EXP1_WORKERS      = 2
EXP1_MODEL        = "yolov8s.pt" # 's' gives better AP than 'n' with same speed
 
# Experiment 3 — 4-class fine-tuned model
EXP3_EPOCHS       = 120         # longer because we have more data
EXP3_IMGSZ        = 640
EXP3_BATCH        = 16
EXP3_LR0          = 0.005       # lower LR for fine-tuning
EXP3_LRF          = 0.01
EXP3_WARMUP_EPOCHS= 5
EXP3_PATIENCE     = 25
EXP3_FREEZE       = 10          # freeze first 10 layers (backbone), fine-tune head
EXP3_WORKERS      = 2
 
# Mosaic and augmentation overrides for Experiment 3
# Small-insect datasets benefit from mosaic=1.0 and copy_paste augmentation
EXP3_MOSAIC       = 1.0
EXP3_COPY_PASTE   = 0.3         # copy-paste augmentation: pastes extra AP instances
EXP3_DEGREES      = 15.0        # rotation augmentation (insects appear at all angles)
EXP3_FLIPLR       = 0.5
EXP3_FLIPUD       = 0.2
EXP3_SCALE        = 0.5         # scale jitter
EXP3_TRANSLATE    = 0.1
EXP3_HSV_H        = 0.015
EXP3_HSV_S        = 0.7
EXP3_HSV_V        = 0.4
 
# ── Dataset paths ──────────────────────────────────────────────────────────────
PATH_REAL_AP_CANDIDATES = [
    BASE / "real_ashpilanta",
    BASE / "asphilanta-1",
    INPUT / "asphilanta-1",
    INPUT / "real-ashpilanta",
    INPUT / "datasets/your-kaggle-username/asphilanta-1",
]
 
# Output folders
PATH_EXP1_DATASET = BASE / "exp1_dataset"      # 3-class base model data
PATH_EXP3_DATASET = BASE / "exp3_dataset"      # 4-class merged data
PATH_EXP2_TESTSET = BASE / "exp2_testset"      # synthetic images for zero-shot eval
PATH_TARGET_VAL   = BASE / "target_domain_val_dataset"
PATH_RESULTS      = BASE / "thesis_results"
PATH_RESULTS.mkdir(parents=True, exist_ok=True)
 
# Possible 3-class weights to use as Experiment 3 starting point
POSSIBLE_EXP1_WEIGHTS = [
    BASE / "runs/detect/exp1_base_3class/weights/best.pt",
    BASE / "runs/exp_A_yolov8_real_only_fewshot/weights/best.pt",
    BASE / "runs/detect/insect_detector/yolov8n_insects/weights/best.pt",
]
 
DEVICE = 0 if torch.cuda.is_available() else "cpu"
 
print("Torch    :", torch.__version__)
print("CUDA     :", torch.cuda.is_available())
print("Device   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Classes 3:", CLASSES_3)
print("Classes 4:", CLASSES_4)
print("AP id    :", AP_CLASS_ID)
 

In [ ]:

# ============================================================
# CELL 3 — Utility functions
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".JPG", ".JPEG", ".PNG"}


def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in {e.lower() for e in IMG_EXTS}])


def safe_clear_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def ensure_yolo_dirs(root):
    root = Path(root)
    for split in ["train", "val"]:
        (root / "images" / split).mkdir(parents=True, exist_ok=True)
        (root / "labels" / split).mkdir(parents=True, exist_ok=True)


def get_split_dirs(root, split):
    root = Path(root)
    split_names = [split]
    if split == "val":
        split_names = ["val", "valid", "validation"]

    for sp in split_names:
        img = root / "images" / sp
        lbl = root / "labels" / sp
        if img.exists() and lbl.exists():
            return img, lbl
    return None, None


def is_yolo_root(root):
    root = Path(root)
    img_train, lbl_train = get_split_dirs(root, "train")
    return img_train is not None and lbl_train is not None and len(list_images(img_train)) > 0


def find_yolo_dataset(candidates, name):
    checked = []

    for c in candidates:
        c = Path(c)
        checked.append(str(c))
        if c.exists() and is_yolo_root(c):
            print(f"Found {name}: {c}")
            return c

    # Automatic search in common places
    for root in [BASE, INPUT]:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_dir() and p.name in ["yolo_dataset", "dataset", "original_dataset", "generated_dataset", "generated_dataset_v7", "synthetic_dataset"]:
                checked.append(str(p))
                if is_yolo_root(p):
                    print(f"Found {name}: {p}")
                    return p

    raise FileNotFoundError(
        f"Could not find {name}. It must contain images/train and labels/train.\n"
        f"Checked examples:\n" + "\n".join(checked[:80])
    )


def find_real_ap_root(candidates):
    checked = []

    for c in candidates:
        c = Path(c)
        checked.append(str(c))
        imgs = list_images(c)
        if c.exists() and len(imgs) > 0:
            print(f"Found real Ashpilanta folder: {c} ({len(imgs)} images)")
            return c

    # Search folder names containing ash/asph/ap
    for root in [BASE, INPUT]:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_dir():
                continue
            low = str(p).lower()
            if any(k in low for k in ["ash", "asph", "ashpilanta", "aspilanta"]):
                imgs = list_images(p)
                checked.append(str(p))
                if 1 <= len(imgs) <= 1000:
                    print(f"Found real Ashpilanta folder: {p} ({len(imgs)} images)")
                    return p

    raise FileNotFoundError(
        "Could not find real Ashpilanta images.\n"
        "Edit PATH_REAL_AP_CANDIDATES in the CONFIG section.\n"
        "Checked:\n" + "\n".join(checked[:80])
    )


def find_label_for_image(img_path, possible_roots=None):
    img_path = Path(img_path)
    candidates = [
        img_path.with_suffix(".txt"),
        img_path.parent / f"{img_path.stem}.txt",
        img_path.parent.parent / "labels" / f"{img_path.stem}.txt",
        img_path.parent.parent / "labels" / "train" / f"{img_path.stem}.txt",
        img_path.parent.parent / "labels" / "val" / f"{img_path.stem}.txt",
        img_path.parent.parent / "labels" / "valid" / f"{img_path.stem}.txt",
    ]

    if possible_roots:
        for root in possible_roots:
            root = Path(root)
            if root.exists():
                candidates.extend(root.rglob(f"{img_path.stem}.txt"))

    for c in candidates:
        if c.exists():
            return c
    return None


def read_yolo_lines(label_path):
    label_path = Path(label_path)
    if not label_path.exists():
        return []
    text = label_path.read_text().strip()
    return text.splitlines() if text else []


def remap_label_lines(lines, class_id_map=None, force_class_id=None):
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cid = int(float(parts[0]))
            if force_class_id is not None:
                parts[0] = str(force_class_id)
            elif class_id_map is not None and cid in class_id_map:
                parts[0] = str(class_id_map[cid])
            new_lines.append(" ".join(parts[:5]))
        except Exception:
            continue
    return new_lines


def copy_one_image_and_label(img_path, dst_images, dst_labels, prefix="", label_path=None, class_id_map=None, force_class_id=None, label_roots=None):
    img_path = Path(img_path)
    dst_images = Path(dst_images)
    dst_labels = Path(dst_labels)
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    new_name = prefix + img_path.name
    shutil.copy2(img_path, dst_images / new_name)

    if label_path is None:
        label_path = find_label_for_image(img_path, possible_roots=label_roots)

    new_label = dst_labels / f"{Path(new_name).stem}.txt"
    if label_path and Path(label_path).exists():
        lines = read_yolo_lines(label_path)
        lines = remap_label_lines(lines, class_id_map=class_id_map, force_class_id=force_class_id)
        new_label.write_text("\n".join(lines))
    else:
        new_label.write_text("")
        return False
    return True


def copy_folder_yolo(src_images, src_labels, dst_images, dst_labels, prefix="", class_id_map=None, force_class_id=None):
    src_images = Path(src_images)
    src_labels = Path(src_labels)

    if not src_images.exists():
        print(f"WARNING: missing image folder, skipped: {src_images}")
        return 0, 0

    imgs = [p for p in sorted(src_images.iterdir()) if p.suffix.lower() in {e.lower() for e in IMG_EXTS}]
    copied = 0
    missing = 0
    for img in imgs:
        lbl = src_labels / f"{img.stem}.txt"
        ok = copy_one_image_and_label(
            img, dst_images, dst_labels, prefix=prefix,
            label_path=lbl, class_id_map=class_id_map, force_class_id=force_class_id
        )
        copied += 1
        if not ok:
            missing += 1
    if missing:
        print(f"WARNING: {missing}/{copied} images from {src_images} had missing labels.")
    return copied, missing


def write_yaml(dataset_root, yaml_path, names):
    dataset_root = Path(dataset_root).resolve()
    yaml_path = Path(yaml_path)
    content = f"""path: {dataset_root}
train: images/train
val: images/val
nc: {len(names)}
names: {names}
"""
    yaml_path.write_text(content)
    print(f"Wrote YAML: {yaml_path}")
    print(content)
    return yaml_path


def count_images(folder):
    return len(list_images(folder))


def count_nonempty_labels(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for f in folder.glob("*.txt") if f.read_text().strip())


def print_dataset_counts(name, root):
    root = Path(root)
    print(f"\n{name}: {root}")
    for split in ["train", "val"]:
        print(f"  {split} images:", count_images(root / "images" / split))
        print(f"  {split} labels with objects:", count_nonempty_labels(root / "labels" / split))


def yolo_xywh_to_xyxy(box, w, h, padding=0.05):
    # box = [x_center, y_center, width, height] normalized
    xc, yc, bw, bh = box
    x1 = (xc - bw / 2) * w
    y1 = (yc - bh / 2) * h
    x2 = (xc + bw / 2) * w
    y2 = (yc + bh / 2) * h

    pad_x = (x2 - x1) * padding
    pad_y = (y2 - y1) * padding
    x1 = max(0, int(x1 - pad_x))
    y1 = max(0, int(y1 - pad_y))
    x2 = min(w, int(x2 + pad_x))
    y2 = min(h, int(y2 + pad_y))
    return x1, y1, x2, y2


In [ ]:
# ==============================================================================
# REPLACEMENT CELL 4 — FIND DATASETS AND SPLIT
# FIXED: copies yolo_dataset and generated_dataset_v6 from /kaggle/input if needed
# ==============================================================================

from pathlib import Path
import shutil
import random
import numpy as np

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

BASE = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

# ==============================================================================
# Helper functions
# ==============================================================================

def count_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0

    total = 0
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        total += len(list(folder.glob(ext)))
    return total


def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []

    imgs = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        imgs.extend(folder.rglob(ext))

    return sorted(imgs)


def copy_dataset_if_missing(src, dst, required_subfolder):
    src = Path(src)
    dst = Path(dst)

    if (dst / required_subfolder).exists():
        print("Already exists:", dst)
        return dst

    if not src.exists():
        raise FileNotFoundError(
            f"Source dataset not found:\n{src}\n\n"
            f"Destination also missing:\n{dst}"
        )

    if dst.exists():
        shutil.rmtree(dst)

    print("Copying dataset...")
    print("From:", src)
    print("To:", dst)

    shutil.copytree(src, dst)

    print("Copied successfully:", dst)

    if not (dst / required_subfolder).exists():
        raise FileNotFoundError(
            f"Copy finished, but required folder is still missing:\n"
            f"{dst / required_subfolder}"
        )

    return dst


# ==============================================================================
# 1. Original 3-class YOLO dataset
# ==============================================================================

PATH_ORIGINAL = BASE / "yolo_dataset"

# Your backup input path based on your Kaggle panel
SRC_ORIGINAL = Path("/kaggle/input/datasets/your-kaggle-username/backup-1/yolo_dataset")

PATH_ORIGINAL = copy_dataset_if_missing(
    src=SRC_ORIGINAL,
    dst=PATH_ORIGINAL,
    required_subfolder=Path("images/train")
)

orig_train_img = PATH_ORIGINAL / "images" / "train"
orig_train_lbl = PATH_ORIGINAL / "labels" / "train"
orig_val_img   = PATH_ORIGINAL / "images" / "val"
orig_val_lbl   = PATH_ORIGINAL / "labels" / "val"

if not orig_train_img.exists() or not orig_train_lbl.exists():
    raise FileNotFoundError(
        f"Original YOLO dataset is incomplete:\n{PATH_ORIGINAL}"
    )

if not orig_val_img.exists() or not orig_val_lbl.exists():
    raise FileNotFoundError(
        f"Original YOLO validation folders are missing:\n{PATH_ORIGINAL}"
    )

print("\nOriginal 3-class YOLO dataset ready:")
print("PATH_ORIGINAL:", PATH_ORIGINAL)
print("Train images:", count_images(orig_train_img))
print("Val images:", count_images(orig_val_img))


# ==============================================================================
# 2. Real Ashpilanta images
# ==============================================================================

PATH_REAL_AP = Path("/kaggle/input/datasets/your-kaggle-username/asphilanta-1")

if not PATH_REAL_AP.exists():
    # fallback if your old helper exists
    if "find_real_ap_root" in globals():
        PATH_REAL_AP = find_real_ap_root(PATH_REAL_AP_CANDIDATES)
    else:
        raise FileNotFoundError(
            "Real Ashpilanta dataset not found at:\n"
            "/kaggle/input/datasets/your-kaggle-username/asphilanta-1"
        )

real_imgs_all = list_images(PATH_REAL_AP)

real_imgs_all = [
    p for p in real_imgs_all
    if "exp1_dataset" not in str(p)
    and "exp3_dataset" not in str(p)
    and "real_only_dataset" not in str(p)
    and "merged_dataset" not in str(p)
    and "generated_dataset" not in str(p)
]

if len(real_imgs_all) == 0:
    raise RuntimeError(f"No real Ashpilanta images found in: {PATH_REAL_AP}")

random.shuffle(real_imgs_all)

print("\nReal Ashpilanta dataset ready:")
print("PATH_REAL_AP:", PATH_REAL_AP)
print("Real AP images:", len(real_imgs_all))


# ==============================================================================
# 3. Synthetic dataset V6
# ==============================================================================

PATH_SYNTH = BASE / "generated_dataset_v6"

# Your uploaded synthetic dataset input path
SRC_SYNTH = Path("/kaggle/input/datasets/your-kaggle-username/synthetic-gen-data/generated_dataset_v6")

PATH_SYNTH = copy_dataset_if_missing(
    src=SRC_SYNTH,
    dst=PATH_SYNTH,
    required_subfolder=Path("images/train")
)

synth_train_img = PATH_SYNTH / "images" / "train"
synth_train_lbl = PATH_SYNTH / "labels" / "train"
synth_val_img_ex = PATH_SYNTH / "images" / "val"
synth_val_lbl_ex = PATH_SYNTH / "labels" / "val"

if not synth_train_img.exists() or not synth_train_lbl.exists():
    raise FileNotFoundError(
        f"Synthetic dataset is incomplete:\n{PATH_SYNTH}"
    )

print("\nSynthetic dataset ready:")
print("PATH_SYNTH:", PATH_SYNTH)
print("Synthetic train images:", count_images(synth_train_img))
print("Synthetic train labels:", len(list(synth_train_lbl.glob('*.txt'))))
print("Synthetic val images:", count_images(synth_val_img_ex) if synth_val_img_ex.exists() else 0)
print("Synthetic val labels:", len(list(synth_val_lbl_ex.glob('*.txt'))) if synth_val_lbl_ex.exists() else 0)


# ==============================================================================
# 4. Real AP split
# ==============================================================================

real_val_count  = max(1, int(len(real_imgs_all) * REAL_VAL_RATIO))
real_val_imgs   = real_imgs_all[:real_val_count]
real_train_pool = real_imgs_all[real_val_count:]

if K_REAL_SHOTS is None:
    real_train_imgs = real_train_pool
else:
    real_train_imgs = real_train_pool[:min(K_REAL_SHOTS, len(real_train_pool))]


# ==============================================================================
# 5. Synthetic split
# ==============================================================================

synth_train_all = list_images(synth_train_img)
random.shuffle(synth_train_all)

if len(synth_train_all) == 0:
    raise RuntimeError(f"No synthetic images found in: {synth_train_img}")

if synth_val_img_ex.exists() and count_images(synth_val_img_ex) > 0:
    synth_train_imgs = synth_train_all
    synth_val_imgs = list_images(synth_val_img_ex)
    synth_val_lbl_root = synth_val_lbl_ex
else:
    synth_val_count = max(1, int(len(synth_train_all) * SYNTH_VAL_RATIO))
    synth_val_imgs = synth_train_all[:synth_val_count]
    synth_train_imgs = synth_train_all[synth_val_count:]
    synth_val_lbl_root = synth_train_lbl


# ==============================================================================
# 6. Final summary
# ==============================================================================

print("\n── Dataset split summary ──────────────────────────────────────────")
print(f"  Original 3-class train  : {count_images(orig_train_img)}")
print(f"  Original 3-class val    : {count_images(orig_val_img)}")
print(f"  Real AP total           : {len(real_imgs_all)}")
print(f"    → train               : {len(real_train_imgs)}")
print(f"    → val                 : {len(real_val_imgs)}")
print(f"  Synthetic train         : {len(synth_train_imgs)}")
print(f"  Synthetic val           : {len(synth_val_imgs)}")
print("──────────────────────────────────────────────────────────────────")

print("\nImportant paths:")
print("PATH_ORIGINAL:", PATH_ORIGINAL)
print("PATH_REAL_AP :", PATH_REAL_AP)
print("PATH_SYNTH   :", PATH_SYNTH)

In [ ]:

# ==============================================================================
#  REPLACEMENT CELL 5 — BUILD EXPERIMENT 1 DATASET  (3-class base model)
#  Replaces old "CELL 5 — Build Experiment A dataset"
# ==============================================================================
# Experiment 1 trains ONLY on the original 3 classes: MR, NC, WF.
# Ashpilanta is NOT included. This creates the baseline model that
# Experiment 2 will use for zero-shot evaluation.
 
print("Building Experiment 1 dataset (3-class, no Ashpilanta)")
safe_clear_dir(PATH_EXP1_DATASET)
ensure_yolo_dirs(PATH_EXP1_DATASET)
 
# Copy original 3-class train split — class IDs 0,1,2 stay as-is
copy_folder_yolo(
    orig_train_img, orig_train_lbl,
    PATH_EXP1_DATASET / "images/train",
    PATH_EXP1_DATASET / "labels/train",
    prefix="orig_train_"
)
 
if orig_val_img and orig_val_lbl:
    copy_folder_yolo(
        orig_val_img, orig_val_lbl,
        PATH_EXP1_DATASET / "images/val",
        PATH_EXP1_DATASET / "labels/val",
        prefix="orig_val_"
    )
 
YAML_EXP1 = write_yaml(PATH_EXP1_DATASET, BASE / "exp1.yaml", CLASSES_3)
print_dataset_counts("Experiment 1 (3-class base)", PATH_EXP1_DATASET)
 

In [ ]:
# ==============================================================================
#  BRIDGE — Define PATH_MODEL_EXP1 / PATH_MODEL_EXP3 from restored weights
#  ─────────────────────────────────────────────────────────────────────────────
#  Run this cell INSTEAD of Cell 55 (TRAIN EXPERIMENT 1) and Cell 57
#  (TRAIN EXPERIMENT 3) whenever the weights were already restored via
#  Cell 7. Those training cells normally define these two variables as a
#  side effect of finishing training -- restoring the .pt files alone does
#  NOT recreate them, which is why Cells 56/58/59/61 failed with NameError.
# ==============================================================================
from pathlib import Path

BASE = Path("/kaggle/working")

PATH_MODEL_EXP1 = BASE / "runs/detect/exp1_base_3class/weights/best.pt"
PATH_MODEL_EXP3 = BASE / "runs/detect/exp3_4class_synthetic_finetuned/weights/best.pt"

for name, p in [("PATH_MODEL_EXP1", PATH_MODEL_EXP1), ("PATH_MODEL_EXP3", PATH_MODEL_EXP3)]:
    status = "✓ found" if p.exists() else "✗ MISSING"
    print(f"{name}: {status} -> {p}")

if not (PATH_MODEL_EXP1.exists() and PATH_MODEL_EXP3.exists()):
    raise FileNotFoundError(
        "One or both restored weight files are missing -- re-check Cell 7's "
        "output above before continuing."
    )

print("\nBoth model paths defined without retraining.")
print("Safe to now run: Cell 56 (Exp 2), Cell 58 (per-class AP), "
      "Cell 59 (evaluate all), Cell 61 (comparison table).")

In [ ]:

# ==============================================================================
#  NEW CELL 4b — BUILD EXPERIMENT 2 TEST SET
#  (zero-shot evaluation of Exp 1 model on synthetic Ashpilanta images)
#  ─────────────────────────────────────────────────────────────────────────────
#  No training happens in Experiment 2.
#  We just need a small set of synthetic images to run the Exp 1 model on.
#  We use the synthetic VAL split (already held out from training).
# ==============================================================================
 
print("Building Experiment 2 test set (synthetic-only, no Ashpilanta training)")
safe_clear_dir(PATH_EXP2_TESTSET)
ensure_yolo_dirs(PATH_EXP2_TESTSET)
 
# For Experiment 2 we evaluate WITH the 3-class model.
# The synthetic labels use class 0 = Aspilanta in the generated YAML,
# but since the 3-class model does NOT know Ashpilanta at all,
# we intentionally leave them as-is so we can count missed/wrong detections.
 
# Copy the synthetic val images (already generated with Ashpilanta bugs on traps)
exp2_count = 0
for img in synth_val_imgs[:200]:    # cap at 200 for a focused eval set
    ok = copy_one_image_and_label(
        img,
        PATH_EXP2_TESTSET / "images/val",
        PATH_EXP2_TESTSET / "labels/val",
        prefix="exp2_",
        # Keep original class 0 label from the synthetic generator
        # so we can measure how many annotations the 3-class model "misses"
        label_roots=[PATH_SYNTH]
    )
    exp2_count += int(ok)
 
# YOLO val() needs a train split too — use one placeholder image
placeholder = list_images(PATH_EXP2_TESTSET / "images/val")
if placeholder:
    ph = placeholder[0]
    ph_lbl = PATH_EXP2_TESTSET / "labels/val" / f"{ph.stem}.txt"
    shutil.copy2(ph, PATH_EXP2_TESTSET / "images/train" / ph.name)
    if ph_lbl.exists():
        shutil.copy2(ph_lbl, PATH_EXP2_TESTSET / "labels/train" / ph_lbl.name)
 
# Write a 1-class YAML for this evaluation only
# (Ashpilanta is class 0 in the synthetic labels)
EXP2_YAML = write_yaml(
    PATH_EXP2_TESTSET,
    BASE / "exp2_eval.yaml",
    ["Ashpilanta"]   # single-class YAML for the zero-shot eval dataset
)
print(f"Experiment 2 test images: {exp2_count}")
print_dataset_counts("Experiment 2 test set", PATH_EXP2_TESTSET)
 
 

In [ ]:

# ==============================================================================
#  REPLACEMENT CELL 6 — BUILD EXPERIMENT 3 DATASET  (4-class merged)
#  Replaces old "CELL 6 — Build Experiment B dataset"
# ==============================================================================
# Experiment 3 adds Ashpilanta as class 3, using:
#   - all original 3-class images (train + val)
#   - real Ashpilanta images (few-shot)
#   - 2000 synthetic sticky-trap Ashpilanta images
 
print("Building Experiment 3 dataset (4-class: MR, NC, WF, Ashpilanta)")
safe_clear_dir(PATH_EXP3_DATASET)
ensure_yolo_dirs(PATH_EXP3_DATASET)
 
# Original 3-class train — class IDs 0,1,2 stay unchanged
copy_folder_yolo(
    orig_train_img, orig_train_lbl,
    PATH_EXP3_DATASET / "images/train",
    PATH_EXP3_DATASET / "labels/train",
    prefix="orig_train_"
)
 
if orig_val_img and orig_val_lbl:
    copy_folder_yolo(
        orig_val_img, orig_val_lbl,
        PATH_EXP3_DATASET / "images/val",
        PATH_EXP3_DATASET / "labels/val",
        prefix="orig_val_"
    )
 
# Real Ashpilanta images — force to class 3
for img in real_train_imgs:
    copy_one_image_and_label(
        img,
        PATH_EXP3_DATASET / "images/train",
        PATH_EXP3_DATASET / "labels/train",
        prefix="real_train_",
        force_class_id=AP_CLASS_ID,
        label_roots=[PATH_REAL_AP]
    )
 
for img in real_val_imgs:
    copy_one_image_and_label(
        img,
        PATH_EXP3_DATASET / "images/val",
        PATH_EXP3_DATASET / "labels/val",
        prefix="real_val_",
        force_class_id=AP_CLASS_ID,
        label_roots=[PATH_REAL_AP]
    )
 
# Synthetic images — remap class 0 (Aspilanta in generator) → class 3 (Ashpilanta)
for img in synth_train_imgs:
    copy_one_image_and_label(
        img,
        PATH_EXP3_DATASET / "images/train",
        PATH_EXP3_DATASET / "labels/train",
        prefix="synth_train_",
        class_id_map={0: AP_CLASS_ID, 1: 0},  # eu_moth distractor → class 0 is dropped
        label_roots=[PATH_SYNTH]
    )
 
for img in synth_val_imgs:
    copy_one_image_and_label(
        img,
        PATH_EXP3_DATASET / "images/val",
        PATH_EXP3_DATASET / "labels/val",
        prefix="synth_val_",
        class_id_map={0: AP_CLASS_ID, 1: 0},
        label_roots=[PATH_SYNTH]
    )
 
YAML_EXP3 = write_yaml(PATH_EXP3_DATASET, BASE / "exp3.yaml", CLASSES_4)
print_dataset_counts("Experiment 3 (4-class + synthetic)", PATH_EXP3_DATASET)
 
 

In [ ]:

# ==============================================================================
#  REPLACEMENT CELL 7 — BUILD COMMON TARGET-DOMAIN VALIDATION SET
# ==============================================================================
# This val set is used to fairly compare ALL three experiments:
#   - Experiment 1: will detect 0 Ashpilanta (wrong class output expected)
#   - Experiment 2: used for zero-shot detection analysis
#   - Experiment 3: main evaluation set
 
print("Building common target-domain validation set")
safe_clear_dir(PATH_TARGET_VAL)
ensure_yolo_dirs(PATH_TARGET_VAL)
 
for img in real_val_imgs:
    copy_one_image_and_label(
        img,
        PATH_TARGET_VAL / "images/val",
        PATH_TARGET_VAL / "labels/val",
        prefix="real_val_",
        force_class_id=AP_CLASS_ID,
        label_roots=[PATH_REAL_AP]
    )
 
for img in synth_val_imgs:
    copy_one_image_and_label(
        img,
        PATH_TARGET_VAL / "images/val",
        PATH_TARGET_VAL / "labels/val",
        prefix="synth_val_",
        class_id_map={0: AP_CLASS_ID},
        label_roots=[PATH_SYNTH]
    )
 
# Placeholder train split required by YOLO
val_ph = list_images(PATH_TARGET_VAL / "images/val")
if val_ph:
    ph = val_ph[0]
    ph_lbl = PATH_TARGET_VAL / "labels/val" / f"{ph.stem}.txt"
    shutil.copy2(ph, PATH_TARGET_VAL / "images/train" / ph.name)
    if ph_lbl.exists():
        shutil.copy2(ph_lbl, PATH_TARGET_VAL / "labels/train" / ph_lbl.name)
 
YAML_TARGET_VAL = write_yaml(PATH_TARGET_VAL, BASE / "target_domain_val.yaml", CLASSES_4)
print_dataset_counts("Common target-domain validation", PATH_TARGET_VAL)
 
 

In [ ]:
# ============================================================
# CELL 8 — Visual sanity check: show random labels from merged/proposed dataset
# FIXED VERSION: defines PATH_MERGED automatically if missing
# ============================================================

from pathlib import Path
import random
import matplotlib.pyplot as plt
import cv2

BASE = Path("/kaggle/working")

# ------------------------------------------------------------
# 1. Find merged/proposed dataset automatically
# ------------------------------------------------------------

if "PATH_MERGED" not in globals():
    possible_merged_paths = [
        BASE / "merged_dataset",
        BASE / "exp3_dataset",
        BASE / "real_plus_synthetic_dataset",
    ]

    PATH_MERGED = None

    for p in possible_merged_paths:
        if (p / "images" / "train").exists() and (p / "labels" / "train").exists():
            PATH_MERGED = p
            break

    if PATH_MERGED is None:
        raise FileNotFoundError(
            "PATH_MERGED is not defined and no merged dataset was found.\n"
            "Expected one of:\n"
            "/kaggle/working/merged_dataset\n"
            "/kaggle/working/exp3_dataset\n\n"
            "Run the dataset-building cell first."
        )

print("Using dataset for sanity check:", PATH_MERGED)

# ------------------------------------------------------------
# 2. Define class names if missing
# ------------------------------------------------------------

if "CLASSES" not in globals():
    CLASSES = ["MR", "NC", "WF", "Ashpilanta"]

print("Classes:", CLASSES)

# ------------------------------------------------------------
# 3. Helper functions if missing
# ------------------------------------------------------------

def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []

    imgs = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        imgs.extend(folder.glob(ext))

    return sorted(imgs)


def read_yolo_lines(label_path):
    label_path = Path(label_path)
    if not label_path.exists():
        return []

    text = label_path.read_text().strip()
    if not text:
        return []

    return [line.strip() for line in text.splitlines() if line.strip()]


def yolo_xywh_to_xyxy(box, w, h, padding=0.0):
    xc, yc, bw, bh = box

    x1 = (xc - bw / 2) * w
    y1 = (yc - bh / 2) * h
    x2 = (xc + bw / 2) * w
    y2 = (yc + bh / 2) * h

    pad_x = (x2 - x1) * padding
    pad_y = (y2 - y1) * padding

    x1 = int(max(0, x1 - pad_x))
    y1 = int(max(0, y1 - pad_y))
    x2 = int(min(w - 1, x2 + pad_x))
    y2 = int(min(h - 1, y2 + pad_y))

    return x1, y1, x2, y2


def draw_yolo_boxes(image_path, label_path, names):
    img = cv2.imread(str(image_path))

    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    lines = read_yolo_lines(label_path)

    for line in lines:
        parts = line.split()

        if len(parts) < 5:
            continue

        cid = int(float(parts[0]))

        if cid >= len(names):
            label_name = f"class_{cid}"
        else:
            label_name = names[cid]

        xc, yc, bw, bh = map(float, parts[1:5])
        x1, y1, x2, y2 = yolo_xywh_to_xyxy([xc, yc, bw, bh], w, h, padding=0.0)

        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(
            img,
            label_name,
            (x1, max(20, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2,
        )

    return img

# ------------------------------------------------------------
# 4. Show random examples
# ------------------------------------------------------------

train_img_dir = PATH_MERGED / "images" / "train"
train_lbl_dir = PATH_MERGED / "labels" / "train"

sample_imgs = list_images(train_img_dir)

print("Training images found:", len(sample_imgs))

if len(sample_imgs) == 0:
    raise RuntimeError(f"No images found in: {train_img_dir}")

random.shuffle(sample_imgs)
sample_imgs = sample_imgs[:4]

for img_path in sample_imgs:
    lbl_path = train_lbl_dir / f"{img_path.stem}.txt"

    img = draw_yolo_boxes(img_path, lbl_path, CLASSES)

    plt.figure(figsize=(7, 7))
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")
    plt.show()

In [ ]:
# ==============================================================================
#  REPLACEMENT CELL 9 — CHOOSE STARTING WEIGHTS
# ==============================================================================
 
print("\n── Starting weights ──────────────────────────────────────────────────")
 
# Experiment 1 always starts from public YOLOv8s pretrained weights.
# YOLOv8s is strictly better than YOLOv8n for small insects:
#   - 11.2M vs 3.2M parameters
#   - typically +5-8 mAP50 on detection tasks
#   - still fast enough on Kaggle P100
EXP1_INIT_WEIGHTS = "yolov8s.pt"
print(f"Experiment 1 starting weights : {EXP1_INIT_WEIGHTS} (public pretrained)")
 
# Experiment 3 fine-tunes FROM the Experiment 1 best.pt whenever possible.
# This is the correct transfer-learning chain:
#   ImageNet → YOLOv8s → (Exp 1) 3-class insect model → (Exp 3) 4-class + AP
EXP3_INIT_WEIGHTS = None
for w in POSSIBLE_EXP1_WEIGHTS:
    if Path(w).exists():
        EXP3_INIT_WEIGHTS = str(w)
        break
 
if EXP3_INIT_WEIGHTS is None:
    EXP3_INIT_WEIGHTS = "yolov8s.pt"
    print("Experiment 1 weights not found yet — Exp 3 will start from yolov8s.pt")
    print("Run Experiment 1 training first, then re-run this cell before Exp 3.")
else:
    print(f"Experiment 3 fine-tuning from : {EXP3_INIT_WEIGHTS}")
 
print("──────────────────────────────────────────────────────────────────────")
 

In [ ]:
# ==============================================================================
#  NEW CELL 9b — AUGMENTATION CONFIG CHECK (before Experiment 3 training)
# ==============================================================================
# This cell prints a summary of the augmentation strategy used in Experiment 3.
# Good augmentation is the single biggest lever for improving mAP on
# small insects under few-shot conditions.
 
print("\n── Experiment 3 Augmentation Strategy ─────────────────────────────────")
print(f"  copy_paste   : {EXP3_COPY_PASTE}  ← pastes extra AP instances per image")
print(f"  mosaic       : {EXP3_MOSAIC}  ← combines 4 images, exposes model to more AP locations")
print(f"  degrees      : {EXP3_DEGREES}  ← insects appear at all angles on traps")
print(f"  scale        : {EXP3_SCALE}  ← simulates different camera distances")
print(f"  fliplr       : {EXP3_FLIPLR}")
print(f"  flipud       : {EXP3_FLIPUD}")
print(f"  hsv_h/s/v    : {EXP3_HSV_H}/{EXP3_HSV_S}/{EXP3_HSV_V}")
print(f"  freeze layers: {EXP3_FREEZE}  ← backbone frozen, head fine-tuned")
print(f"  LR0          : {EXP3_LR0}  ← lower than Exp1 for fine-tuning stability")
print(f"  epochs       : {EXP3_EPOCHS}")
print("──────────────────────────────────────────────────────────────────────")
 
# Confirm Experiment 3 init weights are ready
if not Path(EXP3_INIT_WEIGHTS).exists() and EXP3_INIT_WEIGHTS != "yolov8s.pt":
    print(f"\nWARNING: EXP3_INIT_WEIGHTS not found: {EXP3_INIT_WEIGHTS}")
    print("Falling back to yolov8s.pt")
    EXP3_INIT_WEIGHTS = "yolov8s.pt"
else:
    print(f"\nExperiment 3 will start from: {EXP3_INIT_WEIGHTS}")
 
 

In [ ]:
 
# ==============================================================================
#  REPLACEMENT CELL 10 — TRAIN EXPERIMENT 1  (3-class base model)
#  Replaces old "CELL 10 — Train Experiment A"
# ==============================================================================
# This is the baseline 3-class model used by Experiment 2 (zero-shot eval)
# and as the starting point for Experiment 3 fine-tuning.
 
print("\n" + "=" * 70)
print("TRAINING EXPERIMENT 1 — Base 3-class YOLO detector")
print("Classes: MR, NC, WF  |  No Ashpilanta")
print("=" * 70)
 
model_exp1 = YOLO(EXP1_INIT_WEIGHTS)
 
results_exp1 = model_exp1.train(
    data    = str(YAML_EXP1),
    epochs  = EXP1_EPOCHS,
    imgsz   = EXP1_IMGSZ,
    batch   = EXP1_BATCH,
    device  = DEVICE,
    workers = EXP1_WORKERS,
    patience= EXP1_PATIENCE,
    freeze  = EXP1_FREEZE,
    lr0     = EXP1_LR0,
    lrf     = EXP1_LRF,
    warmup_epochs = EXP1_WARMUP_EPOCHS,
    # Augmentation — standard settings for a clean 3-class detector
    mosaic  = 1.0,
    degrees = 10.0,
    fliplr  = 0.5,
    flipud  = 0.0,
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    scale   = 0.5,
    translate = 0.1,
    # Saving
    project = str(BASE / "runs/detect"),
    name    = "exp1_base_3class",
    exist_ok= True,
    save    = True,
    plots   = True,
)
 
PATH_MODEL_EXP1 = BASE / "runs/detect/exp1_base_3class/weights/best.pt"
print("\nExperiment 1 training complete.")
print("Best model:", PATH_MODEL_EXP1)
 
# After Exp1 training, update EXP3_INIT_WEIGHTS immediately
if PATH_MODEL_EXP1.exists():
    EXP3_INIT_WEIGHTS = str(PATH_MODEL_EXP1)
    print(f"Experiment 3 will now fine-tune from: {EXP3_INIT_WEIGHTS}")
 
 


In [ ]:
# ==============================================================================
#  NEW CELL 4b — EXPERIMENT 2: ZERO-SHOT EVALUATION
#  Run the Experiment 1 model on synthetic Ashpilanta images WITHOUT fine-tuning.
#  This answers: "Can a 3-class model detect Ashpilanta without training on it?"
# ==============================================================================
 
print("\n" + "=" * 70)
print("EXPERIMENT 2 — Zero-shot / few-shot evaluation (no fine-tuning)")
print("Model  : Experiment 1 (3-class: MR, NC, WF)")
print("Test   : Synthetic Ashpilanta sticky-trap images")
print("Goal   : Measure whether the 3-class model confuses or misses Ashpilanta")
print("=" * 70)
 
if not PATH_MODEL_EXP1.exists():
    raise FileNotFoundError(
        f"Experiment 1 model not found at {PATH_MODEL_EXP1}.\n"
        "Run Experiment 1 training first."
    )
 
model_exp2 = YOLO(str(PATH_MODEL_EXP1))
 
# ── Part A: raw inference on synthetic images ──────────────────────────────────
# Run the 3-class model over the synthetic test set and record what classes
# it fires on — this is the "confusion analysis" for Experiment 2.
 
import cv2
 
exp2_images = list_images(PATH_EXP2_TESTSET / "images/val")
print(f"\nRunning Exp 1 model on {len(exp2_images)} synthetic Ashpilanta images...")
 
exp2_rows  = []
class_fire_counts = {0: 0, 1: 0, 2: 0}   # MR / NC / WF
total_annot = 0
total_detected_any = 0
 
for img_path in exp2_images:
    results = model_exp2.predict(
        source    = str(img_path),
        conf      = 0.15,           # low threshold to catch any spurious hits
        iou       = 0.45,
        verbose   = False,
        save      = False,
    )
    r = results[0]
 
    # Count ground-truth Ashpilanta annotations for this image
    lbl_path = PATH_EXP2_TESTSET / "labels/val" / f"{img_path.stem}.txt"
    gt_lines  = read_yolo_lines(lbl_path) if lbl_path.exists() else []
    n_gt_ap   = sum(1 for l in gt_lines if l.split()[0] == "0")  # class 0 = Aspilanta in synth
    total_annot += n_gt_ap
 
    if r.boxes is not None and len(r.boxes) > 0:
        cls_ids = r.boxes.cls.cpu().numpy().astype(int)
        total_detected_any += len(cls_ids)
        for cid in cls_ids:
            class_fire_counts[cid] = class_fire_counts.get(cid, 0) + 1
 
    exp2_rows.append({
        "image"          : img_path.name,
        "gt_ashpilanta"  : n_gt_ap,
        "detected_MR"    : int(np.sum(r.boxes.cls.cpu().numpy() == 0)) if r.boxes else 0,
        "detected_NC"    : int(np.sum(r.boxes.cls.cpu().numpy() == 1)) if r.boxes else 0,
        "detected_WF"    : int(np.sum(r.boxes.cls.cpu().numpy() == 2)) if r.boxes else 0,
        "total_detected" : len(r.boxes) if r.boxes else 0,
    })
 
exp2_df = pd.DataFrame(exp2_rows)
exp2_csv = PATH_RESULTS / "exp2_zeroshot_detections.csv"
exp2_df.to_csv(exp2_csv, index=False)
 
print("\n── Experiment 2 Results (Zero-shot) ──────────────────────────────────")
print(f"  Total synthetic Ashpilanta annotations : {total_annot}")
print(f"  Total detections by 3-class model      : {total_detected_any}")
print(f"  Fired as MR (class 0)                  : {class_fire_counts.get(0,0)}")
print(f"  Fired as NC (class 1)                  : {class_fire_counts.get(1,0)}")
print(f"  Fired as WF (class 2)                  : {class_fire_counts.get(2,0)}")
miss_rate = 1.0 - (total_detected_any / max(total_annot, 1))
print(f"  Approximate miss rate                  : {miss_rate:.1%}")
print("──────────────────────────────────────────────────────────────────────")
print(f"\nConclusion: The 3-class model cannot label Ashpilanta as 'Ashpilanta'.")
print("It either confuses it with an existing class, or misses it entirely.")
print("This demonstrates why Experiment 3 fine-tuning on synthetic data is needed.")
print(f"\nDetailed CSV saved: {exp2_csv}")
 
# ── Part B: save a few annotated sample images for the thesis ──────────────────
exp2_vis_dir = PATH_RESULTS / "exp2_visualizations"
exp2_vis_dir.mkdir(parents=True, exist_ok=True)
 
sample_imgs = exp2_images[:6]
for img_path in sample_imgs:
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    results = model_exp2.predict(source=str(img_path), conf=0.15,
                                  verbose=False, save=False)
    r = results[0]
    # Draw 3-class predictions in RED
    if r.boxes:
        for box, cid, conf in zip(
                r.boxes.xyxy.cpu().numpy(),
                r.boxes.cls.cpu().numpy().astype(int),
                r.boxes.conf.cpu().numpy()):
            x1,y1,x2,y2 = map(int, box)
            cv2.rectangle(img_bgr, (x1,y1),(x2,y2), (0,0,220), 2)
            cv2.putText(img_bgr, f"{CLASSES_3[cid]} {conf:.2f}",
                        (x1, max(15,y1-4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,220), 1)
    # Draw ground-truth Ashpilanta in GREEN
    lbl_path = PATH_EXP2_TESTSET / "labels/val" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        h, w = img_bgr.shape[:2]
        for line in read_yolo_lines(lbl_path):
            parts = line.split()
            if len(parts) < 5:
                continue
            cx,cy,bw,bh = map(float, parts[1:5])
            x1 = int((cx-bw/2)*w); y1 = int((cy-bh/2)*h)
            x2 = int((cx+bw/2)*w); y2 = int((cy+bh/2)*h)
            cv2.rectangle(img_bgr,(x1,y1),(x2,y2),(0,220,0),2)
            cv2.putText(img_bgr,"GT:Ashpilanta",(x1,y2+15),
                        cv2.FONT_HERSHEY_SIMPLEX,0.4,(0,220,0),1)
    out_vis = exp2_vis_dir / img_path.name
    cv2.imwrite(str(out_vis), img_bgr)
 
print(f"Experiment 2 visualizations saved to: {exp2_vis_dir}")
 
# Save Experiment 2 summary
exp2_summary = {
    "experiment"                   : "Experiment 2 — Zero-shot evaluation",
    "model"                        : "Experiment 1 (3-class: MR, NC, WF)",
    "no_fine_tuning"               : True,
    "total_gt_ashpilanta_instances": int(total_annot),
    "total_detections_by_3class_model": int(total_detected_any),
    "fired_as_MR"                  : int(class_fire_counts.get(0,0)),
    "fired_as_NC"                  : int(class_fire_counts.get(1,0)),
    "fired_as_WF"                  : int(class_fire_counts.get(2,0)),
    "approximate_miss_rate"        : round(miss_rate, 4),
    "conclusion"                   : (
        "The 3-class model cannot detect Ashpilanta by name since it was "
        "never trained on it. It either misfires as MR/NC/WF or misses entirely. "
        "This motivates Experiment 3 fine-tuning with synthetic data."
    ),
}
(PATH_RESULTS / "exp2_summary.json").write_text(json.dumps(exp2_summary, indent=2))
print("Experiment 2 summary saved:", PATH_RESULTS / "exp2_summary.json")
 

In [ ]:

# ==============================================================================
#  REPLACEMENT CELL 11 — TRAIN EXPERIMENT 3  (4-class fine-tuned model)
#  Replaces old "CELL 11 — Train Experiment B"
# ==============================================================================
# This is the proposed model.  It fine-tunes the 3-class model into a 4-class
# detector using 2000 synthetic sticky-trap images + real Ashpilanta images.
#
# Key differences from the old CELL 11:
#   1. Uses yolov8s.pt (not yolov8n.pt) — better feature capacity
#   2. Fine-tunes from Exp 1 weights — correct transfer chain
#   3. copy_paste augmentation — especially helpful for rare classes
#   4. Longer training (120 epochs) with cosine LR decay
#   5. Proper freeze strategy: freeze backbone, train neck + head
 
print("\n" + "=" * 70)
print("TRAINING EXPERIMENT 3 — Synthetic-data fine-tuned 4-class detector")
print("Classes : MR, NC, WF, Ashpilanta")
print(f"Init    : {EXP3_INIT_WEIGHTS}")
print(f"Data    : Original 3-class + {len(real_train_imgs)} real AP"
      f" + {len(synth_train_imgs)} synthetic AP")
print("=" * 70)
 
model_exp3 = YOLO(EXP3_INIT_WEIGHTS)
 
results_exp3 = model_exp3.train(
    data    = str(YAML_EXP3),
    epochs  = EXP3_EPOCHS,
    imgsz   = EXP3_IMGSZ,
    batch   = EXP3_BATCH,
    device  = DEVICE,
    workers = EXP3_WORKERS,
    patience= EXP3_PATIENCE,
    freeze  = EXP3_FREEZE,
    lr0     = EXP3_LR0,
    lrf     = EXP3_LRF,
    warmup_epochs = EXP3_WARMUP_EPOCHS,
    # Augmentation — tuned for rare-class small-insect detection
    mosaic      = EXP3_MOSAIC,
    copy_paste  = EXP3_COPY_PASTE,   # key for Ashpilanta class coverage
    degrees     = EXP3_DEGREES,
    fliplr      = EXP3_FLIPLR,
    flipud      = EXP3_FLIPUD,
    scale       = EXP3_SCALE,
    translate   = EXP3_TRANSLATE,
    hsv_h       = EXP3_HSV_H,
    hsv_s       = EXP3_HSV_S,
    hsv_v       = EXP3_HSV_V,
    # Training strategy
    cos_lr      = True,             # cosine LR schedule: smoother convergence
    close_mosaic= 15,               # disable mosaic in last 15 epochs for cleaner final predictions
    label_smoothing = 0.05,         # prevents overconfidence on synthetic labels
    # Saving
    project = str(BASE / "runs/detect"),
    name    = "exp3_4class_synthetic_finetuned",
    exist_ok= True,
    save    = True,
    plots   = True,
)
 
PATH_MODEL_EXP3 = BASE / "runs/detect/exp3_4class_synthetic_finetuned/weights/best.pt"
print("\nExperiment 3 training complete.")
print("Best model:", PATH_MODEL_EXP3)
 
 

In [ ]:
 
# ==============================================================================
#  NEW CELL 11b — PER-CLASS AP REPORT (after Experiment 3 training)
# ==============================================================================
# Prints and saves the per-class AP table immediately after training.
# This is the most important result for the thesis.
 
print("\n── Experiment 3 Per-Class AP Report ───────────────────────────────────")
 
if PATH_MODEL_EXP3.exists():
    _val_model = YOLO(str(PATH_MODEL_EXP3))
    _val_res   = _val_model.val(
        data     = str(YAML_EXP3),
        split    = "val",
        verbose  = True,
        plots    = False,
        project  = str(BASE / "runs/detect"),
        name     = "exp3_val_perclass",
        exist_ok = True,
    )
    box = _val_res.box
 
    print(f"\n  Overall mAP@50     : {float(box.map50):.4f}")
    print(f"  Overall mAP@50-95  : {float(box.map):.4f}")
    print(f"  Precision (mean)   : {float(box.mp):.4f}")
    print(f"  Recall (mean)      : {float(box.mr):.4f}")
 
    ap_report = {}
    if hasattr(box, "ap_class_index") and hasattr(box, "maps"):
        class_indices = list(map(int, box.ap_class_index))
        maps_arr      = np.array(box.maps, dtype=float)
        print("\n  Per-class AP@50-95:")
        for pos, cid in enumerate(class_indices):
            name = CLASSES_4[cid] if cid < len(CLASSES_4) else f"class{cid}"
            ap   = float(maps_arr[pos]) if pos < len(maps_arr) else None
            mark = " ← Ashpilanta" if cid == AP_CLASS_ID else ""
            print(f"    {name:<15} : {ap:.4f}{mark}" if ap is not None
                  else f"    {name:<15} : N/A{mark}")
            ap_report[name] = ap
 
    print("──────────────────────────────────────────────────────────────────────")
    (PATH_RESULTS / "exp3_per_class_ap.json").write_text(
        json.dumps(ap_report, indent=2))
    print("Per-class AP saved:", PATH_RESULTS / "exp3_per_class_ap.json")
else:
    print("Model not found — skipping per-class report.")
 

In [ ]:

# ==============================================================================
#  REPLACEMENT CELL 12 — EVALUATE ALL EXPERIMENTS ON COMMON VALIDATION SET
# ==============================================================================
 
print("\n" + "=" * 70)
print("EVALUATING EXPERIMENTS 1, 2, AND 3 ON COMMON TARGET-DOMAIN VAL SET")
print("=" * 70)
 
def safe_float(x):
    try:
        return float(x) if x is not None else None
    except Exception:
        return None
 
def get_class_ap(metrics, class_id, metric="map"):
    """Extract AP@50-95 (metric='map') or AP@50 (metric='map50') for class_id.

    FIX: box.maps is indexed by CLASS ID, whereas box.all_ap is indexed by
    POSITION within ap_class_index. The previous version used the positional
    index for both, which silently returned the wrong class (or 0.0) whenever
    a class was absent from the evaluation.
    """
    box = metrics.box
    try:
        idx = list(map(int, getattr(box, "ap_class_index", [])))

        if metric == "map" and hasattr(box, "maps"):
            arr = np.array(box.maps, dtype=float)
            # maps is indexed by class id directly
            if class_id < len(arr):
                return float(arr[class_id])
            return None

        if metric == "map50" and hasattr(box, "all_ap"):
            arr = np.array(box.all_ap, dtype=float)
            # all_ap is indexed by position within ap_class_index
            if class_id in idx and arr.ndim == 2:
                pos = idx.index(class_id)
                return float(arr[pos, 0]) if pos < arr.shape[0] else None
            return None

    except Exception as e:
        print(f"AP extraction warning (class {class_id}): {e}")
    return None
 
# ── Experiment 1 on target val ─────────────────────────────────────────────────
# NOTE: Exp 1 is a 3-class model evaluated on a 4-class val set.
# It will score 0 on Ashpilanta class. This is the expected/correct result
# and is the point of Experiment 2 — the model genuinely cannot detect AP.
# We use a separate 3-class YAML for Exp 1's own domain val.
 
YAML_EXP1_VAL = write_yaml(PATH_EXP1_DATASET, BASE / "exp1_val.yaml", CLASSES_3)
m_exp1 = YOLO(str(PATH_MODEL_EXP1)).val(
    data     = str(YAML_EXP1_VAL),
    split    = "val",
    verbose  = True,
    plots    = False,
    project  = str(BASE / "runs/detect"),
    name     = "val_exp1_3class",
    exist_ok = True,
)
 
# ── Experiment 3 on target val ─────────────────────────────────────────────────
m_exp3 = YOLO(str(PATH_MODEL_EXP3)).val(
    data     = str(YAML_TARGET_VAL),
    split    = "val",
    verbose  = True,
    plots    = True,
    project  = str(BASE / "runs/detect"),
    name     = "val_exp3_target_domain",
    exist_ok = True,
)
 
# ── Collect results ────────────────────────────────────────────────────────────
all_results = {
    "Experiment_1_Base_3class": {
        "description"     : "Base 3-class YOLO (MR/NC/WF only)",
        "mAP50"           : safe_float(m_exp1.box.map50),
        "mAP50_95"        : safe_float(m_exp1.box.map),
        "precision"       : safe_float(m_exp1.box.mp),
        "recall"          : safe_float(m_exp1.box.mr),
        "AP_Ashpilanta"   : None,          # cannot detect AP at all
        "note"            : "Cannot output Ashpilanta class — 3-class model only",
    },
    "Experiment_2_Zeroshot": exp2_summary,
    "Experiment_3_Synthetic_Finetuned": {
        "description"     : "4-class YOLO fine-tuned with 2000 synthetic images",
        "mAP50"           : safe_float(m_exp3.box.map50),
        "mAP50_95"        : safe_float(m_exp3.box.map),
        "precision"       : safe_float(m_exp3.box.mp),
        "recall"          : safe_float(m_exp3.box.mr),
        "AP_Ashpilanta_map50_95" : get_class_ap(m_exp3, AP_CLASS_ID, "map"),
        "AP_Ashpilanta_map50"    : get_class_ap(m_exp3, AP_CLASS_ID, "map50"),
    },
}
 
print("\n── All results ───────────────────────────────────────────────────────")
print(json.dumps(
    {k: {kk: (f"{vv:.4f}" if isinstance(vv, float) else vv)
         for kk, vv in v.items()}
     for k, v in all_results.items()},
    indent=2))
 
(PATH_RESULTS / "all_experiment_results.json").write_text(
    json.dumps(all_results, indent=2))
print("\nSaved:", PATH_RESULTS / "all_experiment_results.json")
 

In [ ]:
# ============================================================
# CELL 13 — Optional VLM experiment: CLIP few-shot on object crops
# FIXED VERSION (v3 — self-contained PATH_MERGED, no longer depends on
# another cell having run earlier in the same session)
# ============================================================

RUN_CLIP_VLM = True
# EXPANDED for the K-shot ablation (was [1, 5]) -- cheap to run since the
# loop below already iterates over this list; only the list itself changes.
CLIP_K_VALUES = [1, 2, 5, 10, 20]

clip_results = {}

if RUN_CLIP_VLM:
    try:
        from transformers import CLIPModel, CLIPProcessor
        import torch.nn.functional as F
        from sklearn.metrics import precision_recall_fscore_support, accuracy_score
        from pathlib import Path

        print("\n" + "=" * 70)
        print("RUNNING OPTIONAL VLM FEW-SHOT EXPERIMENT: CLIP")
        print("=" * 70)

        # ------------------------------------------------------------
        # Self-contained PATH_MERGED resolution -- does NOT rely on any
        # other cell having already defined it earlier in this session.
        # ------------------------------------------------------------
        BASE = globals().get("BASE", Path("/kaggle/working"))

        if "PATH_MERGED" not in globals() or globals().get("PATH_MERGED") is None:
            possible_merged_paths = [
                BASE / "merged_dataset",
                BASE / "exp3_dataset",
                BASE / "real_plus_synthetic_dataset",
            ]
            PATH_MERGED = None
            for p in possible_merged_paths:
                if (p / "images" / "train").exists() and (p / "labels" / "train").exists():
                    PATH_MERGED = p
                    break
            if PATH_MERGED is None:
                raise FileNotFoundError(
                    "PATH_MERGED could not be resolved and no merged dataset was "
                    "found.\nExpected one of:\n"
                    "  /kaggle/working/merged_dataset\n"
                    "  /kaggle/working/exp3_dataset\n"
                    "  /kaggle/working/real_plus_synthetic_dataset\n\n"
                    "Run the dataset-building cell (BUILD EXPERIMENT 3 DATASET) first."
                )

        print("PATH_MERGED resolved to:", PATH_MERGED)

        clip_model_name = "openai/clip-vit-base-patch32"
        clip_device = "cuda" if torch.cuda.is_available() else "cpu"

        clip_model = CLIPModel.from_pretrained(clip_model_name).to(clip_device)
        clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
        clip_model.eval()

        def embed_pil_images(pil_images, batch_size=16):
            embeddings = []
            proj_dim = clip_model.config.projection_dim  # 512 for clip-vit-base-patch32

            with torch.no_grad():
                for i in range(0, len(pil_images), batch_size):
                    batch = pil_images[i:i + batch_size]

                    inputs = clip_processor(
                        images=batch,
                        return_tensors="pt",
                        padding=True
                    )

                    pixel_values = inputs["pixel_values"].to(clip_device)

                    feats = clip_model.get_image_features(pixel_values=pixel_values)

                    # Unwrap to a raw tensor, whatever object type came back.
                    if not torch.is_tensor(feats):
                        unwrapped = False
                        for attr in ("image_embeds", "pooler_output", "last_hidden_state"):
                            if hasattr(feats, attr):
                                candidate = getattr(feats, attr)
                                if attr == "last_hidden_state":
                                    candidate = candidate[:, 0, :]
                                feats = candidate
                                unwrapped = True
                                break
                        if not unwrapped:
                            raise TypeError(f"Unexpected CLIP output type: {type(feats)}")

                    # Only project if this tensor is still at the raw
                    # (unprojected) hidden size -- avoids matmul mismatches
                    # when a version already returns the projected embedding.
                    if feats.shape[-1] != proj_dim:
                        feats = clip_model.visual_projection(feats)

                    feats = F.normalize(feats, p=2, dim=-1)
                    embeddings.append(feats.cpu())

            return torch.cat(embeddings, dim=0)

        def crop_objects_from_dataset(dataset_root, split="val", padding=0.10):
            dataset_root = Path(dataset_root)
            img_dir = dataset_root / "images" / split
            lbl_dir = dataset_root / "labels" / split

            crops = []
            labels = []
            meta = []

            for img_path in list_images(img_dir):
                lbl_path = lbl_dir / f"{img_path.stem}.txt"

                if not lbl_path.exists():
                    continue

                try:
                    im = Image.open(img_path).convert("RGB")
                    w, h = im.size
                except Exception:
                    continue

                for line in read_yolo_lines(lbl_path):
                    parts = line.split()

                    if len(parts) < 5:
                        continue

                    cid = int(float(parts[0]))
                    box = list(map(float, parts[1:5]))

                    x1, y1, x2, y2 = yolo_xywh_to_xyxy(box, w, h, padding=padding)

                    if x2 <= x1 or y2 <= y1:
                        continue

                    crop = im.crop((x1, y1, x2, y2))
                    crops.append(crop)
                    labels.append(1 if cid == AP_CLASS_ID else 0)
                    meta.append({
                        "image": img_path.name,
                        "class_id": cid
                    })

            return crops, np.array(labels), meta

        # Support set: real Ashpilanta training images/crops
        support_images_all = []

        for img_path in real_train_imgs:
            try:
                im = Image.open(img_path).convert("RGB")
                lbl_path = find_label_for_image(img_path, possible_roots=[PATH_REAL_AP])
                lines = read_yolo_lines(lbl_path) if lbl_path else []

                if lines:
                    w, h = im.size
                    parts = lines[0].split()
                    box = list(map(float, parts[1:5]))
                    x1, y1, x2, y2 = yolo_xywh_to_xyxy(box, w, h, padding=0.15)
                    support_images_all.append(im.crop((x1, y1, x2, y2)))
                else:
                    support_images_all.append(im)

            except Exception:
                pass

        # Query set: object crops from merged validation set
        query_crops, query_binary_labels, query_meta = crop_objects_from_dataset(
            PATH_MERGED,
            split="val",
            padding=0.10
        )

        print("Support AP images/crops:", len(support_images_all))
        print("Query validation crops:", len(query_crops))
        print("Query AP crops:", int(query_binary_labels.sum()))
        print("Query non-AP crops:", int((query_binary_labels == 0).sum()))

        if len(support_images_all) == 0:
            print("CLIP skipped: no support AP images.")
        elif len(query_crops) == 0:
            print("CLIP skipped: no query crops.")
        elif len(set(query_binary_labels.tolist())) < 2:
            print("CLIP skipped: query set contains only one binary class.")
        else:
            query_emb = embed_pil_images(query_crops)

            seen_k_eff = set()

            for k in CLIP_K_VALUES:
                k_eff = min(k, len(support_images_all))
                was_capped = k_eff < k

                # FIX: key by the REQUESTED k, not k_eff. Previously two
                # different requested K values that both got capped to the
                # same k_eff (e.g. K=10 and K=20 when only 15 support images
                # exist) would silently overwrite each other in clip_results
                # under an identical dict key. Keying by requested K keeps
                # every sweep point, and num_support/was_capped record what
                # actually happened.
                support_subset = support_images_all[:k_eff]
                support_emb = embed_pil_images(support_subset)

                proto = F.normalize(
                    support_emb.mean(dim=0, keepdim=True),
                    p=2,
                    dim=-1
                )

                sims = (query_emb @ proto.T).squeeze(1).numpy()

                # Simple reproducible threshold
                threshold = float(np.mean(sims))
                pred = (sims >= threshold).astype(int)

                precision, recall, f1, _ = precision_recall_fscore_support(
                    query_binary_labels,
                    pred,
                    average="binary",
                    zero_division=0
                )

                acc = accuracy_score(query_binary_labels, pred)

                clip_results[f"CLIP_{k}_shot_crop_classifier"] = {
                    "requested_k": int(k),
                    "accuracy": float(acc),
                    "precision": float(precision),
                    "recall": float(recall),
                    "f1": float(f1),
                    "threshold": threshold,
                    "num_query_crops": int(len(query_crops)),
                    "num_support": int(k_eff),
                    "capped_by_support_pool": bool(was_capped),
                }
                seen_k_eff.add(k_eff)

            print("\nCLIP RESULTS:")
            print(json.dumps(clip_results, indent=2))

            (PATH_RESULTS / "clip_vlm_results.json").write_text(
                json.dumps(clip_results, indent=2)
            )

            print("\nSaved:", PATH_RESULTS / "clip_vlm_results.json")

            # ------------------------------------------------------------
            # K-shot ablation table + plot -- this is the actual thesis
            # artifact: how few-shot support-set size affects the CLIP
            # filter's classification quality.
            # ------------------------------------------------------------
            kshot_rows = []
            for key, v in clip_results.items():
                kshot_rows.append({
                    "K_requested": v["requested_k"],
                    "K_effective": v["num_support"],
                    "capped": v["capped_by_support_pool"],
                    "accuracy": v["accuracy"],
                    "precision": v["precision"],
                    "recall": v["recall"],
                    "f1": v["f1"],
                })

            kshot_df = pd.DataFrame(kshot_rows).sort_values("K_requested").reset_index(drop=True)
            print("\nK-SHOT ABLATION SWEEP:")
            print(kshot_df.to_string(index=False))

            kshot_csv_path = PATH_RESULTS / "clip_kshot_ablation.csv"
            kshot_df.to_csv(kshot_csv_path, index=False)
            print("\nSaved:", kshot_csv_path)

            if kshot_df["K_requested"].nunique() > 1:
                import matplotlib.pyplot as plt

                fig, ax = plt.subplots(figsize=(7, 5))
                for metric in ["accuracy", "precision", "recall", "f1"]:
                    ax.plot(kshot_df["K_requested"], kshot_df[metric], marker="o", label=metric)

                capped_rows = kshot_df[kshot_df["capped"]]
                if len(capped_rows):
                    ax.scatter(capped_rows["K_requested"], capped_rows["f1"],
                                facecolors="none", edgecolors="red", s=150,
                                linewidths=2, label="capped by support pool size",
                                zorder=5)

                ax.set_xlabel("K (requested support-set size)")
                ax.set_ylabel("score")
                ax.set_title("CLIP few-shot filter: performance vs. support-set size (K)")
                ax.set_xscale("log")
                ax.set_xticks(kshot_df["K_requested"])
                ax.set_xticklabels(kshot_df["K_requested"])
                ax.legend()
                ax.grid(alpha=0.3)
                fig.tight_layout()

                kshot_plot_path = PATH_RESULTS / "clip_kshot_ablation.png"
                fig.savefig(kshot_plot_path, dpi=150)
                plt.show()
                print("Saved plot:", kshot_plot_path)

    except Exception as e:
        print("CLIP VLM experiment skipped due to error:", repr(e))
        clip_results = {}

In [ ]:

# ==============================================================================
#  REPLACEMENT CELL 14 — COMPARISON TABLE AND PLOTS
# ==============================================================================
 
print("\n" + "=" * 70)
print("CREATING THESIS COMPARISON TABLE AND PLOTS")
print("=" * 70)
 
rows = [
    {
        "Experiment" : "Exp 1 — Base 3-class YOLO",
        "Type"       : "CNN detector",
        "Classes"    : 3,
        "Ashpilanta" : "No",
        "mAP50"      : all_results["Experiment_1_Base_3class"]["mAP50"],
        "mAP50_95"   : all_results["Experiment_1_Base_3class"]["mAP50_95"],
        "Precision"  : all_results["Experiment_1_Base_3class"]["precision"],
        "Recall"     : all_results["Experiment_1_Base_3class"]["recall"],
        "AP_AP_50_95": None,
        "AP_AP_50"   : None,
        "Note"       : "Baseline — no AP class",
    },
    {
        "Experiment" : "Exp 2 — Zero-shot (no fine-tuning)",
        "Type"       : "Zero-shot evaluation",
        "Classes"    : 3,
        "Ashpilanta" : "Zero-shot",
        "mAP50"      : None,
        "mAP50_95"   : None,
        "Precision"  : None,
        "Recall"     : None,
        "AP_AP_50_95": None,
        "AP_AP_50"   : None,
        "Note"       : (
            f"Miss rate {all_results['Experiment_2_Zeroshot'].get('approximate_miss_rate','')*100:.1f}% "
            f"— model cannot label Ashpilanta"
        ),
    },
    {
        "Experiment" : "Exp 3 — Synthetic fine-tuned 4-class",
        "Type"       : "CNN detector",
        "Classes"    : 4,
        "Ashpilanta" : "Yes",
        "mAP50"      : all_results["Experiment_3_Synthetic_Finetuned"]["mAP50"],
        "mAP50_95"   : all_results["Experiment_3_Synthetic_Finetuned"]["mAP50_95"],
        "Precision"  : all_results["Experiment_3_Synthetic_Finetuned"]["precision"],
        "Recall"     : all_results["Experiment_3_Synthetic_Finetuned"]["recall"],
        "AP_AP_50_95": all_results["Experiment_3_Synthetic_Finetuned"]["AP_Ashpilanta_map50_95"],
        "AP_AP_50"   : all_results["Experiment_3_Synthetic_Finetuned"]["AP_Ashpilanta_map50"],
        "Note"       : "Proposed model — 2000 synthetic AP images",
    },
]
 
results_df = pd.DataFrame(rows)
metric_cols = ["mAP50","mAP50_95","Precision","Recall","AP_AP_50_95","AP_AP_50"]
for col in metric_cols:
    results_df[col] = pd.to_numeric(results_df[col], errors="coerce")
 
results_df.to_csv(PATH_RESULTS / "thesis_comparison_table.csv", index=False)
print(results_df.to_string(index=False))
print("\nSaved:", PATH_RESULTS / "thesis_comparison_table.csv")
 
# ── Plot 1: mAP50 comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
 
plot_data = results_df.dropna(subset=["mAP50"])
if not plot_data.empty:
    axes[0].bar(plot_data["Experiment"], plot_data["mAP50"],
                color=["#4C72B0","#DD8452","#55A868"])
    axes[0].set_title("mAP@50 Comparison", fontweight="bold")
    axes[0].set_ylabel("mAP@50")
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis="x", rotation=25)
    for bar, val in zip(axes[0].patches, plot_data["mAP50"]):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                     f"{val:.3f}", ha="center", va="bottom", fontsize=8)
 
# ── Plot 2: Precision / Recall ─────────────────────────────────────────────────
pr_data = results_df.dropna(subset=["Precision","Recall"])
if not pr_data.empty:
    x = np.arange(len(pr_data))
    w = 0.35
    axes[1].bar(x-w/2, pr_data["Precision"], w, label="Precision", color="#4C72B0")
    axes[1].bar(x+w/2, pr_data["Recall"],    w, label="Recall",    color="#DD8452")
    axes[1].set_title("Precision / Recall", fontweight="bold")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(pr_data["Experiment"], rotation=25, ha="right")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
 
# ── Plot 3: Ashpilanta AP@50 and AP@50-95 side by side ────────────────────────
ap_data = results_df.dropna(subset=["AP_AP_50"])
if not ap_data.empty:
    x = np.arange(len(ap_data))
    w = 0.35
    axes[2].bar(x-w/2, ap_data.get("AP_AP_50", []),    w, label="AP@50",    color="#55A868")
    axes[2].bar(x+w/2, ap_data.get("AP_AP_50_95", []), w, label="AP@50-95", color="#C44E52")
    axes[2].set_title("Ashpilanta Class AP\n(Experiment 3 only)", fontweight="bold")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(ap_data["Experiment"], rotation=25, ha="right")
    axes[2].set_ylim(0, 1)
    axes[2].legend()
else:
    axes[2].text(0.5, 0.5, "Ashpilanta AP\nnot available\nfor Exp 1 & 2",
                 ha="center", va="center", transform=axes[2].transAxes)
    axes[2].set_title("Ashpilanta Class AP", fontweight="bold")
 
plt.suptitle("Experiment Comparison — Ashpilanta Detection on Sticky Traps",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plot_out = PATH_RESULTS / "thesis_experiment_comparison.png"
plt.savefig(plot_out, dpi=200, bbox_inches="tight")
plt.show()
print("Comparison plot saved:", plot_out)
 
# ── Markdown table for thesis ──────────────────────────────────────────────────
md_df = results_df.copy()
for col in metric_cols:
    if col in md_df.columns:
        md_df[col] = md_df[col].apply(
            lambda x: f"{x:.4f}" if pd.notna(x) else "—")
 
md_text = md_df.to_markdown(index=False)
(PATH_RESULTS / "thesis_comparison_table.md").write_text(md_text)
print("\nMarkdown table:")
print(md_text)
print("\nCell 14 complete.")


In [ ]:
# ============================================================
# CELL 15 — Inference function for sticky-trap images
# FIXED VERSION: automatically finds proposed model if PATH_MODEL_B is missing
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from ultralytics import YOLO

BASE = Path("/kaggle/working")

# ------------------------------------------------------------
# 1. Define paths/classes if missing
# ------------------------------------------------------------

if "PATH_RESULTS" not in globals():
    PATH_RESULTS = BASE / "thesis_results"
    PATH_RESULTS.mkdir(parents=True, exist_ok=True)

if "CLASSES" not in globals():
    CLASSES = ["MR", "NC", "WF", "Ashpilanta"]

# ------------------------------------------------------------
# 2. Find proposed Experiment 3 model automatically
# ------------------------------------------------------------

def find_proposed_model():
    """
    Finds the best.pt model for Experiment 3 / proposed 4-class model.
    Prefers model paths containing synthetic, exp3, proposed, merged, or real_plus.
    """
    candidate_models = list((BASE / "runs").rglob("best.pt"))

    if len(candidate_models) == 0:
        raise FileNotFoundError(
            "No best.pt model found under /kaggle/working/runs.\n"
            "Train Experiment 3 first or restore thesis_backup.zip."
        )

    print("Found best.pt models:")
    for i, p in enumerate(candidate_models):
        print(f"{i}: {p}")

    preferred_keywords = [
        "exp3",
        "proposed",
        "synthetic",
        "real_plus",
        "merged",
        "4class",
        "ashpilanta",
        "asphilanta",
    ]

    preferred = []
    for p in candidate_models:
        s = str(p).lower()
        if any(k in s for k in preferred_keywords):
            preferred.append(p)

    if len(preferred) > 0:
        return preferred[-1]

    # fallback: use last best.pt
    return candidate_models[-1]


if "PATH_MODEL_B" not in globals():
    PATH_MODEL_B = find_proposed_model()

print("\nUsing proposed model:")
print(PATH_MODEL_B)

# ------------------------------------------------------------
# 3. Load model once
# ------------------------------------------------------------

proposed_model = YOLO(str(PATH_MODEL_B))

print("\nModel classes:")
print(proposed_model.names)

# Update CLASSES from model if possible
CLASSES = [proposed_model.names[i] for i in sorted(proposed_model.names.keys())]

print("CLASSES:", CLASSES)

# ------------------------------------------------------------
# 4. Inference function
# ------------------------------------------------------------

def predict_sticky_trap(image_path, model_path=PATH_MODEL_B, conf=0.25, save=True):
    """
    Use the proposed model to detect and count insects in a new sticky-trap image.
    Returns YOLO result object and a count table.
    """
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    model = YOLO(str(model_path))

    results = model.predict(
        source=str(image_path),
        conf=conf,
        save=save,
        project=str(PATH_RESULTS),
        name="predictions",
        exist_ok=True
    )

    r = results[0]

    if r.boxes is None or len(r.boxes) == 0:
        count_df = pd.DataFrame(columns=["class_id", "class_name", "count"])
        print("No detections.")
        return r, count_df

    class_ids = r.boxes.cls.cpu().numpy().astype(int)
    unique, counts = np.unique(class_ids, return_counts=True)

    count_df = pd.DataFrame({
        "class_id": unique,
        "class_name": [
            model.names[int(i)] if int(i) in model.names else str(i)
            for i in unique
        ],
        "count": counts,
    })

    print("\nDetection count table:")
    print(count_df)

    return r, count_df


print("\nCell 15 ready.")
print("Best proposed model:", PATH_MODEL_B)
print("Results folder:", PATH_RESULTS)

print("\nTo predict a new image, run:")
print("result, counts = predict_sticky_trap('/kaggle/input/your-image.jpg', conf=0.25)")

In [ ]:
# ==============================================================================
# SAVE HEAVY COMPUTATION RESULTS — LIGHT BACKUP VERSION
# Use this AFTER Cell 14 and BEFORE Cell 15.
#
# This avoids "No space left on device" by NOT copying huge image datasets.
# It saves:
#   - trained YOLO weights
#   - training logs/results
#   - thesis results tables/plots/json
#   - YAML files/configs
#
# Output:
#   /kaggle/working/thesis_model_results_backup.zip
# ==============================================================================

from pathlib import Path
import shutil
import json
import zipfile
from datetime import datetime

BASE = Path("/kaggle/working")

BACKUP_DIR = BASE / "thesis_model_results_backup"
ZIP_PATH = BASE / "thesis_model_results_backup.zip"

# ----------------------------------------------------------------------
# 1. Clean old failed backup folders/zips to free space
# ----------------------------------------------------------------------

CLEAN_ITEMS = [
    BASE / "thesis_full_backup",
    BASE / "thesis_full_backup.zip",
    BACKUP_DIR,
    ZIP_PATH,
]

print("=" * 70)
print("CLEANING OLD BACKUP FILES")
print("=" * 70)

for p in CLEAN_ITEMS:
    if p.exists():
        if p.is_dir():
            shutil.rmtree(p)
            print("Deleted folder:", p)
        else:
            p.unlink()
            print("Deleted file:", p)

BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------
# 2. Save only lightweight but important folders
# ----------------------------------------------------------------------

SAVE_FOLDERS = [
    BASE / "runs",             # trained model weights: best.pt, last.pt
    BASE / "thesis_results",   # CSV, plots, JSON results
]

SAVE_FILES = [
    BASE / "exp1.yaml",
    BASE / "exp2_eval.yaml",
    BASE / "exp3.yaml",
    BASE / "target_domain_val.yaml",
    BASE / "dataset.yaml",
    BASE / "real_only.yaml",
    BASE / "merged.yaml",
]

saved_folders = []
saved_files = []
skipped = []

print("\n" + "=" * 70)
print("SAVING IMPORTANT MODEL/RESULT FILES")
print("=" * 70)

for folder in SAVE_FOLDERS:
    if folder.exists():
        dst = BACKUP_DIR / folder.name
        shutil.copytree(folder, dst)
        n_files = sum(1 for x in dst.rglob("*") if x.is_file())
        saved_folders.append(str(folder))
        print(f"Saved folder: {folder} ({n_files} files)")
    else:
        skipped.append(str(folder))
        print("Skipped missing folder:", folder)

for file in SAVE_FILES:
    if file.exists():
        dst = BACKUP_DIR / file.name
        shutil.copy2(file, dst)
        saved_files.append(str(file))
        print("Saved file:", file)
    else:
        skipped.append(str(file))
        print("Skipped missing file:", file)

# ----------------------------------------------------------------------
# 3. Save model inspection metadata
# ----------------------------------------------------------------------

best_models = list((BASE / "runs").rglob("best.pt")) if (BASE / "runs").exists() else []
last_models = list((BASE / "runs").rglob("last.pt")) if (BASE / "runs").exists() else []
results_csvs = list((BASE / "runs").rglob("results.csv")) if (BASE / "runs").exists() else []

metadata = {
    "created_at": datetime.now().isoformat(),
    "note": "Light backup: contains trained weights/results only. Synthetic images are saved separately.",
    "saved_folders": saved_folders,
    "saved_files": saved_files,
    "skipped": skipped,
    "best_pt_models": [str(p) for p in best_models],
    "last_pt_models": [str(p) for p in last_models],
    "results_csv_files": [str(p) for p in results_csvs],
    "important_note": "To restore full training environment, also attach/copy generated_dataset_v6 separately.",
}

metadata_path = BACKUP_DIR / "BACKUP_METADATA.json"
metadata_path.write_text(json.dumps(metadata, indent=2))

print("\nSaved metadata:", metadata_path)

# ----------------------------------------------------------------------
# 4. Create zip
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CREATING BACKUP ZIP")
print("=" * 70)

shutil.make_archive(
    base_name=str(ZIP_PATH).replace(".zip", ""),
    format="zip",
    root_dir=BACKUP_DIR
)

size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)

print("\n" + "=" * 70)
print("BACKUP COMPLETE")
print("=" * 70)
print("Backup file:", ZIP_PATH)
print(f"Backup size: {size_mb:.2f} MB")

print("\nFound best.pt models:")
for i, p in enumerate(best_models):
    print(f"{i}: {p}")

if len(best_models) == 0:
    print("\nWARNING: No best.pt files found. Training weights were not saved.")
else:
    print("\nOK: trained model weights are included in this backup.")

print("\nDownload this file before stopping the session:")
print(ZIP_PATH)

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/working")

print("Model/results backup exists:", (BASE / "thesis_model_results_backup.zip").exists())
print("Model/results backup size MB:", (BASE / "thesis_model_results_backup.zip").stat().st_size / (1024*1024) if (BASE / "thesis_model_results_backup.zip").exists() else 0)

print("\nTrained model weights found:")
for p in (BASE / "runs").rglob("best.pt"):
    print(p)

print("\nSynthetic dataset exists in working:", (BASE / "generated_dataset_v6").exists())
print("Synthetic dataset input exists:", Path("/kaggle/input/datasets/your-kaggle-username/synthetic-gen-data/generated_dataset_v6").exists())

In [ ]:
# ============================================================
# EXPERIMENT 4: Diffusion-Augmented Cutouts + Pseudo-Labeling Self-Training
# ============================================================
# Two NEW few-shot components added on top of the existing Exp 1-3 pipeline:
#   (A) Diffusion-based cutout augmentation  -> expands the 44 real Ashpilanta
#       cutouts into many appearance variants BEFORE they go into the v6
#       copy-paste compositor. Targets the box-precision / AP@50-95 gap.
#   (B) Pseudo-labeling self-training loop   -> uses the trained Exp 3 model to
#       generate labels on UNLABELED real full-trap photos, then retrains on
#       real+synthetic combined. Targets the synthetic/real domain gap.
#
# Everything else (generator v6 compositing, Exp 3 training config, CLIP
# K-shot filter) is REUSED UNCHANGED from your existing notebook. This file
# assumes those cells have already been run and the following variables
# already exist in your session:
#   PATH_AP_CUTOUTS, PATH_MODEL_EXP1, EXP3_INIT_WEIGHTS, YAML_EXP3,
#   CLASSES_4, BASE, generate_synthetic_dataset() [your v6 generator entrypoint]
# ============================================================

# %% [markdown]
# ## Setup

# %%
import os
import shutil
import random
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ------------------------------------------------------------------
# Paths for the new components (add these alongside your existing PATH_* vars)
# ------------------------------------------------------------------
PATH_AP_CUTOUTS_AUGMENTED = BASE / "ashpilanta_cutouts_augmented"
PATH_AP_CUTOUTS_AUGMENTED.mkdir(parents=True, exist_ok=True)

PATH_UNLABELED_REAL = BASE / "unlabeled_real_traps"          # <-- point this at
                                                               #     your pool of
                                                               #     unlabeled real
                                                               #     full-trap photos
PATH_PSEUDO_LABELED = BASE / "pseudo_labeled_dataset"
(PATH_PSEUDO_LABELED / "images").mkdir(parents=True, exist_ok=True)
(PATH_PSEUDO_LABELED / "labels").mkdir(parents=True, exist_ok=True)

PATH_EXP4_DATASET = BASE / "exp4_dataset"
YAML_EXP4 = BASE / "exp4.yaml"


# ============================================================
# (A) DIFFUSION-BASED CUTOUT AUGMENTATION
# ============================================================
# Strategy: run img2img at moderate strength on the RGB channels of each
# real cutout to generate appearance variants (lighting, texture, minor
# color variation), then REUSE the original alpha mask on the output so
# the silhouette stays valid for the existing pad-aware compositor. This
# avoids the much harder problem of generating novel silhouettes/poses
# with correct alpha mattes.

# %%
def install_diffusers_if_needed():
    """Run once per Kaggle session."""
    os.system("pip install -q diffusers accelerate safetensors --no-index 2>/dev/null "
              "|| pip install -q diffusers accelerate safetensors")


def load_img2img_pipeline():
    from diffusers import StableDiffusionImg2ImgPipeline
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        safety_checker=None,
    ).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe


AUGMENT_PROMPT = (
    "close-up macro photograph of a small insect, natural lighting, "
    "sharp focus, realistic texture, entomology reference photo"
)
NEGATIVE_PROMPT = "cartoon, illustration, painting, blurry, low quality, deformed"


def augment_cutout(pipe, rgba_path: Path, n_variants: int = 5,
                    strength_range=(0.25, 0.45)) -> list[Path]:
    """
    Generate n_variants appearance variants of a single RGBA cutout.
    The alpha channel (silhouette) is preserved from the original;
    only RGB appearance is diffusion-augmented.
    """
    rgba = cv2.imread(str(rgba_path), cv2.IMREAD_UNCHANGED)
    if rgba is None or rgba.shape[2] != 4:
        print(f"Skipping {rgba_path.name}: not a valid RGBA cutout")
        return []

    bgr, alpha = rgba[:, :, :3], rgba[:, :, 3]
    rgb_pil = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)).convert("RGB")

    # SD models want dims divisible by 8
    w, h = rgb_pil.size
    w8, h8 = (w // 8) * 8 or 8, (h // 8) * 8 or 8
    rgb_pil_resized = rgb_pil.resize((w8, h8))

    out_paths = []
    for i in range(n_variants):
        strength = random.uniform(*strength_range)
        with torch.autocast(DEVICE) if DEVICE == "cuda" else torch.no_grad():
            result = pipe(
                prompt=AUGMENT_PROMPT,
                negative_prompt=NEGATIVE_PROMPT,
                image=rgb_pil_resized,
                strength=strength,
                guidance_scale=6.5,
                num_inference_steps=30,
            ).images[0]

        result = result.resize((w, h))
        result_bgr = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)

        # Reapply original alpha mask -> keeps silhouette valid for the compositor
        rgba_out = np.dstack([result_bgr, alpha])

        out_name = f"{rgba_path.stem}_diff{i:02d}.png"
        out_path = PATH_AP_CUTOUTS_AUGMENTED / out_name
        cv2.imwrite(str(out_path), rgba_out)
        out_paths.append(out_path)

    return out_paths


def run_cutout_augmentation(n_variants_per_cutout: int = 5):
    install_diffusers_if_needed()
    pipe = load_img2img_pipeline()

    cutout_paths = sorted(PATH_AP_CUTOUTS.glob("*.png"))
    print(f"Augmenting {len(cutout_paths)} real cutouts "
          f"x{n_variants_per_cutout} variants each...")

    all_new = []
    for p in cutout_paths:
        new_paths = augment_cutout(pipe, p, n_variants=n_variants_per_cutout)
        all_new.extend(new_paths)

    print(f"Generated {len(all_new)} diffusion-augmented cutout variants "
          f"-> {PATH_AP_CUTOUTS_AUGMENTED}")

    del pipe
    torch.cuda.empty_cache()
    return all_new


def build_expanded_cutout_pool():
    """
    Combine original + diffusion-augmented cutouts into a single pool the
    v6 generator draws from. Point your generator's cutout-loading step at
    this combined directory instead of PATH_AP_CUTOUTS alone.
    """
    pool_dir = BASE / "ashpilanta_cutouts_pool"
    pool_dir.mkdir(parents=True, exist_ok=True)

    for p in PATH_AP_CUTOUTS.glob("*.png"):
        shutil.copy(p, pool_dir / p.name)
    for p in PATH_AP_CUTOUTS_AUGMENTED.glob("*.png"):
        shutil.copy(p, pool_dir / p.name)

    n_total = len(list(pool_dir.glob("*.png")))
    print(f"Expanded cutout pool: {n_total} total cutouts "
          f"(44 real + diffusion variants) at {pool_dir}")
    return pool_dir


# Usage (run once):
# run_cutout_augmentation(n_variants_per_cutout=5)         # 44 -> ~264 cutouts
# expanded_pool_dir = build_expanded_cutout_pool()
# Then re-run your EXISTING v6 generator, pointing its cutout source at
# expanded_pool_dir instead of PATH_AP_CUTOUTS, to produce a new
# generated_dataset_v7 with far more source appearance diversity.


# ============================================================
# (B) PSEUDO-LABELING SELF-TRAINING LOOP
# ============================================================
# Uses the trained Exp 3 model to label real, currently-unlabeled full-trap
# photos. Only high-confidence, non-overlapping detections are kept, to
# avoid reinforcing the model's own mistakes.

# %%
PSEUDO_CONF_THRESHOLD = 0.55   # deliberately stricter than the 0.25 eval
                                # default / 0.35-0.45 deployment range, since
                                # these labels become ground truth for retraining
PSEUDO_IOU_NMS = 0.45
MAX_PSEUDO_LABELS_PER_IMAGE = 60   # sanity cap; adjust to your trap density


def yolo_label_line(cls_id: int, xc: float, yc: float, w: float, h: float) -> str:
    return f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"


def generate_pseudo_labels(model_path: Path, unlabeled_dir: Path,
                            out_dir: Path, conf: float = PSEUDO_CONF_THRESHOLD):
    """
    Run the Exp 3 model on unlabeled real images and write YOLO-format
    pseudo-labels for high-confidence detections only.
    """
    model = YOLO(str(model_path))
    img_paths = sorted(list(unlabeled_dir.glob("*.jpg")) +
                        list(unlabeled_dir.glob("*.jpeg")) +
                        list(unlabeled_dir.glob("*.png")))

    print(f"Running pseudo-labeling on {len(img_paths)} unlabeled real images "
          f"(conf >= {conf})...")

    kept_images = 0
    kept_boxes_by_class = {i: 0 for i in range(len(CLASSES_4))}

    for img_path in img_paths:
        results = model.predict(
            source=str(img_path), conf=conf, iou=PSEUDO_IOU_NMS,
            verbose=False, save=False,
        )
        r = results[0]
        if r.boxes is None or len(r.boxes) == 0:
            continue

        boxes_xywhn = r.boxes.xywhn.cpu().numpy()
        cls_ids = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()

        if len(cls_ids) > MAX_PSEUDO_LABELS_PER_IMAGE:
            # keep only the most confident detections up to the cap
            top_idx = np.argsort(-confs)[:MAX_PSEUDO_LABELS_PER_IMAGE]
            boxes_xywhn, cls_ids = boxes_xywhn[top_idx], cls_ids[top_idx]

        lines = []
        for (xc, yc, w, h), cid in zip(boxes_xywhn, cls_ids):
            lines.append(yolo_label_line(int(cid), xc, yc, w, h))
            kept_boxes_by_class[int(cid)] = kept_boxes_by_class.get(int(cid), 0) + 1

        if not lines:
            continue

        # Copy image + write label
        dst_img = out_dir / "images" / img_path.name
        dst_lbl = out_dir / "labels" / f"{img_path.stem}.txt"
        shutil.copy(img_path, dst_img)
        dst_lbl.write_text("\n".join(lines))
        kept_images += 1

    print(f"Pseudo-labeled {kept_images}/{len(img_paths)} images.")
    print("Per-class pseudo-label counts:",
          {CLASSES_4[k]: v for k, v in kept_boxes_by_class.items()})
    return kept_images


def build_exp4_dataset():
    """
    Merge: Exp 3 dataset (original 3-class + synthetic Ashpilanta + real
    Ashpilanta) with the newly pseudo-labeled real images, into a new
    Exp 4 training set. Reuses your existing exp3_dataset as the base.
    """
    exp3_dir = BASE / "exp3_dataset"
    for split in ["train", "val"]:
        (PATH_EXP4_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
        (PATH_EXP4_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

    # Copy everything from Exp 3 dataset first
    for split in ["train", "val"]:
        for img in (exp3_dir / "images" / split).glob("*"):
            shutil.copy(img, PATH_EXP4_DATASET / "images" / split / img.name)
        for lbl in (exp3_dir / "labels" / split).glob("*"):
            shutil.copy(lbl, PATH_EXP4_DATASET / "labels" / split / lbl.name)

    # Add pseudo-labeled real images into the TRAIN split only
    # (never into val -- val must stay clean ground truth)
    pl_images = list((PATH_PSEUDO_LABELED / "images").glob("*"))
    for img in pl_images:
        lbl = PATH_PSEUDO_LABELED / "labels" / f"{img.stem}.txt"
        if not lbl.exists():
            continue
        shutil.copy(img, PATH_EXP4_DATASET / "images" / "train" / img.name)
        shutil.copy(lbl, PATH_EXP4_DATASET / "labels" / "train" / lbl.name)

    n_train = len(list((PATH_EXP4_DATASET / "images" / "train").glob("*")))
    n_val = len(list((PATH_EXP4_DATASET / "images" / "val").glob("*")))
    print(f"Exp 4 dataset built: {n_train} train images "
          f"({len(pl_images)} pseudo-labeled real added), {n_val} val images")

    YAML_EXP4.write_text(f"""# YOLOv8 dataset configuration for Experiment 4
path: {PATH_EXP4_DATASET}
train: images/train
val:   images/val

nc: {len(CLASSES_4)}
names: {CLASSES_4}
""")
    print(f"Wrote {YAML_EXP4}")


# ============================================================
# Exp 4 training call -- same hyperparameters as Exp 3, new dataset
# ============================================================

# %%
def train_exp4():
    model_exp4 = YOLO(str(EXP3_INIT_WEIGHTS))  # continue from Exp 3, not Exp 1,
                                                 # since Exp 3 already knows Ashpilanta

    results_exp4 = model_exp4.train(
        data=str(YAML_EXP4),
        epochs=60, imgsz=640, batch=16, device=DEVICE,
        freeze=10,
        lr0=0.002, lrf=0.01,          # lower LR: gentle refinement, not
                                        # from-scratch fine-tuning
        warmup_epochs=3, cos_lr=True,
        mosaic=1.0, copy_paste=0.2,
        degrees=15.0, fliplr=0.5, flipud=0.2, scale=0.5,
        label_smoothing=0.05, close_mosaic=10,
        project=str(BASE / "runs/detect"),
        name="exp4_diffusion_pseudolabel_finetuned",
        save=True, plots=True,
    )
    return model_exp4, results_exp4


# ============================================================
# ONE-SHOT DRIVER: run the whole new pipeline end to end
# ============================================================

# %%
def run_exp4_pipeline():
    print("=" * 60)
    print("STEP 1/4: Diffusion-augment the 44 real cutouts")
    print("=" * 60)
    run_cutout_augmentation(n_variants_per_cutout=5)
    expanded_pool_dir = build_expanded_cutout_pool()

    print("\n" + "=" * 60)
    print("STEP 2/4: Re-run v6 generator with expanded cutout pool")
    print("=" * 60)
    print("--> Point your existing generate_synthetic_dataset() call at "
          f"{expanded_pool_dir} instead of PATH_AP_CUTOUTS, save output as "
          "generated_dataset_v7, then rebuild Exp 3 dataset with v7 in place "
          "of v6 (reuse your existing Section 3 dataset-building cells).")

    print("\n" + "=" * 60)
    print("STEP 3/4: Pseudo-label unlabeled real trap photos")
    print("=" * 60)
    if not PATH_UNLABELED_REAL.exists() or not any(PATH_UNLABELED_REAL.iterdir()):
        print(f"WARNING: {PATH_UNLABELED_REAL} is empty or missing. "
              "Skipping pseudo-labeling -- populate this folder with "
              "unlabeled real full-trap photos to enable this step.")
    else:
        generate_pseudo_labels(
            model_path=EXP3_INIT_WEIGHTS,
            unlabeled_dir=PATH_UNLABELED_REAL,
            out_dir=PATH_PSEUDO_LABELED,
        )
        build_exp4_dataset()

        print("\n" + "=" * 60)
        print("STEP 4/4: Train Exp 4")
        print("=" * 60)
        model_exp4, results_exp4 = train_exp4()
        return model_exp4, results_exp4

    return None, None


# Run with:
# model_exp4, results_exp4 = run_exp4_pipeline()

# **SECTION 5:**

# MAIN SOLUTION — Deployment Pipeline for Real-Time Field Images
## (Ashpilanta-only: MR/NC/WF are filtered out immediately and never appear
## in output, since they are not part of the thesis's detection goal)

Pipeline: **tiled inference → filter to Ashpilanta → cross-tile NMS →
CLIP few-shot filter → confidence thresholding → annotated image / CSV /
JSON output.**


In [ ]:
# ============================================================
# CELL 16 — MAIN SOLUTION: deployment pipeline setup
# ============================================================

import cv2, json
import numpy as np
import pandas as pd
from pathlib import Path
import torch
from torchvision.ops import nms as torch_nms
from ultralytics import YOLO
from PIL import Image

if "BASE" not in globals():
    BASE = Path("/kaggle/working")
if "PATH_RESULTS" not in globals():
    PATH_RESULTS = BASE / "thesis_results"
    PATH_RESULTS.mkdir(parents=True, exist_ok=True)
if "CLASSES_4" not in globals():
    CLASSES_4 = ["MR", "NC", "WF", "Ashpilanta"]
if "AP_CLASS_ID" not in globals():
    AP_CLASS_ID = 3

# ── Pick the best available trained detector: prefer Exp 4, fall back to Exp 3 ──
_exp4_candidate = BASE / "runs/detect/exp4_diffusion_pseudolabel_finetuned/weights/best.pt"
if "PATH_MODEL_EXP3" not in globals():
    PATH_MODEL_EXP3 = BASE / "runs/detect/exp3_4class_synthetic_finetuned/weights/best.pt"

if _exp4_candidate.exists():
    DEPLOY_MODEL_PATH = _exp4_candidate
    print("Using Experiment 4 model (diffusion + pseudo-label refined) for deployment.")
elif PATH_MODEL_EXP3.exists():
    DEPLOY_MODEL_PATH = PATH_MODEL_EXP3
    print("Experiment 4 model not found -- using Experiment 3 model for deployment.")
else:
    raise FileNotFoundError(
        "No trained 4-class model found. Train (or restore) Experiment 3 "
        "and/or Experiment 4 before running the deployment pipeline."
    )

_deploy_yolo_model = YOLO(str(DEPLOY_MODEL_PATH))
print("Deployment detector loaded:", DEPLOY_MODEL_PATH)
print("Model classes:", _deploy_yolo_model.names)
print(f"NOTE: this pipeline only reports class {AP_CLASS_ID} (Ashpilanta). "
      f"MR/NC/WF are detected internally by the model but filtered out "
      f"immediately and never appear in output.")

# ── Deployment configuration ──────────────────────────────────────────────
TILE_SIZE     = 640
TILE_OVERLAP  = 0.25
TILE_CONF     = 0.25   # per-tile confidence; tune with Cell 24c if needed
TILE_IOU      = 0.45   # per-tile NMS
MERGE_IOU     = 0.55   # cross-tile NMS threshold
DEPLOY_CONF   = 0.40   # final reported confidence threshold
# UPDATED based on measured evidence (Cell 28/28b run, now that the CLIP
# filter actually executes): on generated_dataset_v6, CLIP filtering moves
# precision 0.931 -> 0.936 (+0.5pp) but recall 0.801 -> 0.675 (-12.6pp),
# for a net F1 drop from 0.861 to 0.785. The recall cost outweighs the
# precision gain, so CLIP filtering is now OFF by default. Flip to True
# per-call via detect_ashpilanta_field_image(use_clip_filter=True) if your
# own data shows a different trade-off (e.g. more visually-similar
# distractor insects than generated_dataset_v6 has).
USE_CLIP_FILTER_DEFAULT = False

# ── CLIP setup: SELF-CONTAINED, does NOT depend on Cell 13 having run ──────
# FIX (previous bug): the CLIP filter used to silently no-op whenever
# Cell 13 hadn't been executed earlier in *this* kernel session (e.g.
# after a restart, or when jumping straight to Section 5/6 via the
# RESTORE cells). Every "CLIP-filtered" precision/recall number reported
# below was therefore identical to the unfiltered baseline -- CLIP was
# never actually applied. This block now loads CLIP directly here, so
# the deployment pipeline is correct regardless of what else has run.

if "clip_model" not in globals():
    from transformers import CLIPModel, CLIPProcessor
    import torch.nn.functional as F

    print("\nLoading CLIP model for deployment (self-contained, independent of Cell 13)...")
    _clip_model_name = "openai/clip-vit-base-patch32"
    _clip_device = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model = CLIPModel.from_pretrained(_clip_model_name).to(_clip_device)
    clip_processor = CLIPProcessor.from_pretrained(_clip_model_name)
    clip_model.eval()

    def embed_pil_images(pil_images, batch_size=16):
        embeddings = []
        proj_dim = clip_model.config.projection_dim
        with torch.no_grad():
            for i in range(0, len(pil_images), batch_size):
                batch = pil_images[i:i + batch_size]
                inputs = clip_processor(images=batch, return_tensors="pt", padding=True)
                pixel_values = inputs["pixel_values"].to(_clip_device)
                feats = clip_model.get_image_features(pixel_values=pixel_values)
                if not torch.is_tensor(feats):
                    for attr in ("image_embeds", "pooler_output", "last_hidden_state"):
                        if hasattr(feats, attr):
                            candidate = getattr(feats, attr)
                            if attr == "last_hidden_state":
                                candidate = candidate[:, 0, :]
                            feats = candidate
                            break
                if feats.shape[-1] != proj_dim:
                    feats = clip_model.visual_projection(feats)
                feats = F.normalize(feats, p=2, dim=-1)
                embeddings.append(feats.cpu())
        return torch.cat(embeddings, dim=0)

    print("CLIP model ready:", _clip_model_name, "on", _clip_device)


def _build_deploy_clip_support_images(k=10, padding=0.15):
    """Builds a K-shot Ashpilanta support set directly from the real
    training images (PATH_REAL_AP / real_train_imgs collected earlier in
    the notebook), independent of whether Cell 13 ran in this session."""
    if "support_images_all" in globals() and len(globals()["support_images_all"]) > 0:
        return globals()["support_images_all"]

    if "real_train_imgs" not in globals() or "PATH_REAL_AP" not in globals():
        print("WARNING: real_train_imgs / PATH_REAL_AP not found -- cannot "
              "build a CLIP support set. Run the earlier data-loading cells first.")
        return []

    imgs = []
    for img_path in real_train_imgs[:max(k, 10)]:
        try:
            im = Image.open(img_path).convert("RGB")
            lbl_path = find_label_for_image(img_path, possible_roots=[PATH_REAL_AP]) \
                if "find_label_for_image" in globals() else None
            lines = read_yolo_lines(lbl_path) if (lbl_path and "read_yolo_lines" in globals()) else []
            if lines and "yolo_xywh_to_xyxy" in globals():
                w, h = im.size
                parts = lines[0].split()
                box = list(map(float, parts[1:5]))
                x1, y1, x2, y2 = yolo_xywh_to_xyxy(box, w, h, padding=padding)
                imgs.append(im.crop((x1, y1, x2, y2)))
            else:
                imgs.append(im)
        except Exception:
            pass

    globals()["support_images_all"] = imgs
    return imgs


In [ ]:
# ============================================================
# CELL 17 — Tiled inference, filtered to Ashpilanta only
# ============================================================

def make_tiles(img_w, img_h, tile=TILE_SIZE, overlap=TILE_OVERLAP):
    """Overlapping tile grid covering the full image, so no insect near a
    tile boundary is lost to a single unlucky crop."""
    stride = max(1, int(tile * (1 - overlap)))
    xs = list(range(0, max(img_w - tile, 0) + 1, stride)) or [0]
    ys = list(range(0, max(img_h - tile, 0) + 1, stride)) or [0]
    if xs[-1] + tile < img_w:
        xs.append(max(0, img_w - tile))
    if ys[-1] + tile < img_h:
        ys.append(max(0, img_h - tile))

    tiles = []
    for y in ys:
        for x in xs:
            x2, y2 = min(x + tile, img_w), min(y + tile, img_h)
            x1, y1 = max(0, x2 - tile), max(0, y2 - tile)
            tiles.append((x1, y1, x2, y2))
    seen = set()
    unique_tiles = []
    for t in tiles:
        if t not in seen:
            seen.add(t)
            unique_tiles.append(t)
    return unique_tiles


def tiled_predict(image_path, conf=TILE_CONF, iou=TILE_IOU, keep_class_id=AP_CLASS_ID):
    """Runs the detector tile-by-tile at native resolution, projects boxes
    back to full-image coordinates, and (by default) discards every
    detection that isn't Ashpilanta before it ever leaves this function --
    MR/NC/WF are not part of the thesis's detection goal and are dropped
    here rather than carried through downstream stages for no reason.
    Set keep_class_id=None to keep all classes (useful for debugging only).
    """
    image_path = Path(image_path)
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    h, w = img.shape[:2]

    tiles = make_tiles(w, h)
    print(f"{image_path.name}: {w}x{h}px -> {len(tiles)} tiles "
          f"({TILE_SIZE}px, {int(TILE_OVERLAP*100)}% overlap)")

    all_boxes, all_scores, all_classes = [], [], []

    for (x1, y1, x2, y2) in tiles:
        tile_img = img[y1:y2, x1:x2]
        results = _deploy_yolo_model.predict(
            source=tile_img, conf=conf, iou=iou, verbose=False, save=False,
        )
        r = results[0]
        if r.boxes is None or len(r.boxes) == 0:
            continue

        xyxy = r.boxes.xyxy.cpu().numpy().copy()
        scores = r.boxes.conf.cpu().numpy()
        cls_ids = r.boxes.cls.cpu().numpy().astype(int)

        if keep_class_id is not None:
            keep = cls_ids == keep_class_id
            if not keep.any():
                continue
            xyxy, scores, cls_ids = xyxy[keep], scores[keep], cls_ids[keep]

        xyxy[:, [0, 2]] += x1
        xyxy[:, [1, 3]] += y1

        all_boxes.append(xyxy)
        all_scores.append(scores)
        all_classes.append(cls_ids)

    if not all_boxes:
        return (np.empty((0, 4)), np.empty((0,)),
                np.empty((0,), dtype=int), img)

    return (np.concatenate(all_boxes, axis=0),
            np.concatenate(all_scores, axis=0),
            np.concatenate(all_classes, axis=0),
            img)


In [ ]:
# ============================================================
# CELL 18 — Cross-tile NMS
#
# UPDATED: added a center-distance merge criterion alongside IoU.
# Two adjacent tiles can each capture a different partial view of the
# same insect near a tile seam -- producing two boxes whose IoU is below
# MERGE_IOU even though they're the same physical insect. Pure IoU-based
# NMS misses these; merging by center proximity as well catches them.
# ============================================================

def cross_tile_nms(boxes, scores, classes, iou_thresh=MERGE_IOU, center_dist_frac=0.6):
    """Removes duplicate detections of the same Ashpilanta instance
    produced by overlapping tiles. Two boxes are treated as duplicates if
    EITHER:
      - their IoU >= iou_thresh (standard NMS), OR
      - their centers are within center_dist_frac * average_box_size of
        each other (catches tile-seam duplicates that standard IoU misses)
    Only the highest-confidence box among duplicates is kept."""
    if len(boxes) == 0:
        return boxes, scores, classes

    order = np.argsort(-scores)
    boxes, scores, classes = boxes[order], scores[order], classes[order]

    n = len(boxes)
    suppressed = np.zeros(n, dtype=bool)
    keep = []

    centers = np.stack([(boxes[:, 0] + boxes[:, 2]) / 2,
                         (boxes[:, 1] + boxes[:, 3]) / 2], axis=1)
    sizes = np.maximum(boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1])

    for i in range(n):
        if suppressed[i]:
            continue
        keep.append(i)

        for j in range(i + 1, n):
            if suppressed[j] or classes[j] != classes[i]:
                continue

            iou = box_iou_xyxy(boxes[i], boxes[j:j + 1])[0]
            center_dist = np.linalg.norm(centers[i] - centers[j])
            avg_size = (sizes[i] + sizes[j]) / 2
            is_duplicate = (iou >= iou_thresh) or (center_dist <= center_dist_frac * avg_size)

            if is_duplicate:
                suppressed[j] = True

    keep = sorted(keep)
    return boxes[keep], scores[keep], classes[keep]


In [ ]:
# ============================================================
# CELL 19 — CLIP few-shot filter for Ashpilanta candidates
#
# FIX: previously this ONLY worked if Cell 13 had already run earlier in
# the same session (it read clip_model / support_images_all / clip_results
# straight out of globals() and silently skipped filtering otherwise --
# see Cell 16 for the full explanation). It now falls back to building its
# own support set and self-calibrating a threshold when Cell 13's richer,
# 3,768-crop calibration isn't available, so the filter always actually runs.
# ============================================================

def _get_or_build_clip_prototype(k=5):
    """Prefers the support set collected by Cell 13 if present in this
    session; otherwise builds one directly from the real training images
    via _build_deploy_clip_support_images() (defined in Cell 16)."""
    if "_deploy_clip_prototype" in globals():
        return globals()["_deploy_clip_prototype"]

    if "clip_model" not in globals() or "embed_pil_images" not in globals():
        return None

    support = globals().get("support_images_all")
    if not support:
        support = _build_deploy_clip_support_images(k=max(k, 10))

    if not support:
        return None

    import torch.nn.functional as F
    k_eff = min(k, len(support))
    support_emb = embed_pil_images(support[:k_eff])
    proto = F.normalize(support_emb.mean(dim=0, keepdim=True), p=2, dim=-1)
    globals()["_deploy_clip_prototype"] = proto
    print(f"Built CLIP {k_eff}-shot Ashpilanta prototype for deployment filtering.")
    return proto


def _get_calibrated_clip_threshold(k=5, fallback=0.75):
    """Resolves a similarity threshold, preferring (in order):
      1. The threshold computed by the Cell 13 K-shot experiment for the
         matching K value (statistically meaningful -- derived from 3,768
         real query crops, not 2-3 per-image candidates).
      2. A cached value from a previous call in this session.
      3. A threshold self-calibrated right now from the support crops
         themselves (mean similarity minus one std-dev, i.e. conservative).
      4. A fixed fallback (0.75) only if neither of the above is possible.
    """
    key = f"CLIP_{k}_shot_crop_classifier"
    if "clip_results" in globals() and key in globals().get("clip_results", {}):
        threshold = globals()["clip_results"][key]["threshold"]
        globals()["_deploy_clip_threshold"] = threshold
        return threshold

    if "_deploy_clip_threshold" in globals():
        return globals()["_deploy_clip_threshold"]

    proto = _get_or_build_clip_prototype(k=k)
    support = globals().get("support_images_all")
    if proto is not None and support:
        import torch.nn.functional as F
        support_emb = embed_pil_images(support)
        sims = (support_emb @ proto.T).squeeze(1).numpy()
        threshold = float(np.mean(sims) - np.std(sims))
        print(f"No Cell 13 calibrated threshold found for K={k} -- "
              f"self-calibrated threshold={threshold:.3f} from "
              f"{len(support)} support crops.")
    else:
        print(f"WARNING: no calibrated CLIP threshold found for K={k} and "
              f"no support set available -- using fallback threshold={fallback}")
        threshold = fallback

    globals()["_deploy_clip_threshold"] = threshold
    return threshold


def clip_filter_ashpilanta(boxes, scores, classes, img_bgr, sim_margin=0.0,
                            fixed_threshold=None):
    """Drops Ashpilanta detections whose crop embedding is not close enough
    to the real-image CLIP prototype -- catches background artefacts and
    distractor insects that superficially resemble the target class.
    Input is assumed to already be Ashpilanta-only (filtered by
    tiled_predict); this function no longer needs to sub-select by class."""
    if len(boxes) == 0:
        return boxes, scores, classes

    proto = _get_or_build_clip_prototype()
    if proto is None:
        print("CLIP prototype unavailable -- no support images or CLIP model "
              "could be found/built. Skipping CLIP filter for this call.")
        return boxes, scores, classes

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    crops = []
    for box in boxes:
        x1, y1, x2, y2 = box.astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        crop = img_rgb[y1:y2, x1:x2]
        if crop.size == 0:
            crops.append(Image.fromarray(np.zeros((8, 8, 3), dtype=np.uint8)))
        else:
            crops.append(Image.fromarray(crop))

    crop_emb = embed_pil_images(crops)
    sims = (crop_emb @ proto.T).squeeze(1).numpy()

    threshold = fixed_threshold if fixed_threshold is not None else _get_calibrated_clip_threshold(k=5)
    threshold = threshold + sim_margin

    keep_mask = sims >= threshold
    n_dropped = int((~keep_mask).sum())
    if n_dropped:
        print(f"CLIP filter removed {n_dropped}/{len(boxes)} Ashpilanta "
              f"candidate(s) below calibrated similarity threshold {threshold:.3f}")

    return boxes[keep_mask], scores[keep_mask], classes[keep_mask]


In [ ]:
# ============================================================
# CELL 20 — Output: annotated image / CSV / JSON (Ashpilanta only)
# ============================================================

from PIL import Image

ASHPILANTA_COLOR_BGR = (196, 78, 82)   # single color, since only one class is reported

def save_annotated_image(img_bgr, boxes, scores, classes, out_path):
    vis = img_bgr.copy()
    for (x1, y1, x2, y2), s in zip(boxes.astype(int), scores):
        cv2.rectangle(vis, (x1, y1), (x2, y2), ASHPILANTA_COLOR_BGR, 2)
        cv2.putText(vis, f"Ashpilanta {s:.2f}", (x1, max(0, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, ASHPILANTA_COLOR_BGR, 1, cv2.LINE_AA)
    cv2.imwrite(str(out_path), vis)
    return out_path


def save_detection_csv(boxes, scores, classes, out_path):
    rows = [{
        "detection_id": i,
        "class_name": "Ashpilanta",
        "confidence": float(s),
        "x1": float(b[0]), "y1": float(b[1]), "x2": float(b[2]), "y2": float(b[3]),
    } for i, (b, s) in enumerate(zip(boxes, scores))]
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df


def save_count_json(classes, out_path):
    summary = {
        "total_ashpilanta": int(len(classes)),
    }
    Path(out_path).write_text(json.dumps(summary, indent=2))
    return summary


In [ ]:
# ============================================================
# CELL 21 — MAIN SOLUTION ENTRY POINT (Ashpilanta only)
#
# UPDATED based on measured evidence, now that the CLIP fix from Cell 16/19
# is confirmed actually running (see Cell 28/28b): on generated_dataset_v6,
# the tiled pipeline scores Precision=0.095 Recall=0.013 (F1=0.023), while
# plain single-pass scores Precision=0.931 Recall=0.801 (F1=0.861) --
# tiling is still dramatically worse, and CLIP filtering makes tiling even
# worse on top of that (tile crops are already out-of-distribution for the
# CLIP prototype, so the filter now discards real detections too).
#
# Also per Cell 28b: CLIP filtering on the (correctly-behaving) single-pass
# baseline itself trades recall for precision at a net F1 loss
# (0.861 -> 0.785), so USE_CLIP_FILTER_DEFAULT is now False (see Cell 16).
#
# Current default: single-pass detection, CLIP filter OFF. Tiling remains
# OPT-IN only, reserved for images so large that Ashpilanta would become
# undetectably small after a standard resize -- there is currently no
# evidence it helps on generated_dataset_v6-style images at all.
# ============================================================

TILING_SIZE_THRESHOLD = 3000  # px; only tile if BOTH dimensions exceed this
                                # AND single-pass confidence looks weak.
                                # Tune this based on your own field photos --
                                # there is currently no evidence this helps
                                # on generated_dataset_v6-style images at all.


def _single_pass_predict(image_path, conf=DEPLOY_CONF):
    """Standard whole-image resize-and-predict, matching how the model
    was actually trained and how Cell 13's CLIP threshold was calibrated."""
    img = cv2.imread(str(image_path))
    results = _deploy_yolo_model.predict(source=str(image_path), conf=conf,
                                          verbose=False, save=False)
    r = results[0]
    if r.boxes is None or len(r.boxes) == 0:
        return np.empty((0, 4)), np.empty((0,)), np.empty((0,), dtype=int), img

    all_boxes = r.boxes.xyxy.cpu().numpy()
    all_scores = r.boxes.conf.cpu().numpy()
    all_classes = r.boxes.cls.cpu().numpy().astype(int)
    ap_mask = all_classes == AP_CLASS_ID
    return all_boxes[ap_mask], all_scores[ap_mask], all_classes[ap_mask], img


def detect_ashpilanta_field_image(image_path, deploy_conf=DEPLOY_CONF,
                                   use_clip_filter=USE_CLIP_FILTER_DEFAULT,
                                   force_tiling=False, out_dir=None):
    """
    THE MAIN SOLUTION.
    Given ONE raw field photograph of a sticky trap, returns bounding
    boxes and a total count for Ashpilanta ONLY.

    Defaults to SINGLE-PASS detection (matches training-time resize
    behavior, matches the CLIP filter's calibration) -- measured to
    dramatically outperform tiled inference on generated_dataset_v6.

    Set force_tiling=True to use the tiled pipeline instead (only
    recommended for images so large that Ashpilanta would be lost to a
    standard resize -- verify this actually helps on your specific images
    before relying on it, since the evidence so far shows it doesn't).
    """
    image_path = Path(image_path)
    out_dir = Path(out_dir) if out_dir else (PATH_RESULTS / "field_predictions")
    out_dir.mkdir(parents=True, exist_ok=True)

    if force_tiling:
        boxes, scores, classes, img = tiled_predict(image_path)
        boxes, scores, classes = cross_tile_nms(boxes, scores, classes)
        keep = scores >= deploy_conf
        boxes, scores, classes = boxes[keep], scores[keep], classes[keep]
    else:
        boxes, scores, classes, img = _single_pass_predict(image_path, conf=deploy_conf)

    if use_clip_filter:
        boxes, scores, classes = clip_filter_ashpilanta(boxes, scores, classes, img)

    stem = image_path.stem
    annotated_path = save_annotated_image(img, boxes, scores, classes,
                                           out_dir / f"{stem}_annotated.jpg")
    csv_path = out_dir / f"{stem}_detections.csv"
    json_path = out_dir / f"{stem}_summary.json"

    det_df = save_detection_csv(boxes, scores, classes, csv_path)
    summary = save_count_json(classes, json_path)

    print(f"\n{'='*60}\nFIELD IMAGE RESULT: {image_path.name} "
          f"({'tiled' if force_tiling else 'single-pass'})\n{'='*60}")
    print(json.dumps(summary, indent=2))
    print("Annotated image:", annotated_path)
    print("Detections CSV: ", csv_path)
    print("Summary JSON:   ", json_path)

    return det_df, summary, annotated_path


# Run on a real field photo with:
# det_df, summary, annotated_path = detect_ashpilanta_field_image(
#     "/kaggle/input/your-field-photo/trap_photo.jpg"
# )


# **SECTION 6:**

# MAIN SOLUTION — Test Results & Performance (Ashpilanta-only)

All metrics below are computed for Ashpilanta ONLY. MR/NC/WF ground truth
and predictions are excluded entirely before matching -- they are not part
of the thesis's detection goal, and including them (as earlier versions of
this evaluation did) distorted every metric with irrelevant noise (e.g. an
MR false-positive count of 88 that had nothing to do with Ashpilanta
performance).


In [ ]:
# ============================================================
# CELL 22 — Evaluation utilities: IoU matching + ground truth loading
# (Ashpilanta only)
# ============================================================

import time
import numpy as np
import pandas as pd
import cv2
from pathlib import Path

def load_ashpilanta_gt_as_xyxy(label_path, img_w, img_h, ap_class_id=AP_CLASS_ID):
    """Reads a YOLO-format label file and returns ONLY Ashpilanta ground
    truth boxes as (boxes_xyxy). MR/NC/WF lines are skipped entirely."""
    label_path = Path(label_path)
    if not label_path.exists():
        return np.empty((0, 4))

    boxes = []
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        if not parts:
            continue
        cid = int(float(parts[0]))
        if cid != ap_class_id:
            continue
        xc, yc, w, h = map(float, parts[1:5])
        x1 = (xc - w / 2) * img_w
        y1 = (yc - h / 2) * img_h
        x2 = (xc + w / 2) * img_w
        y2 = (yc + h / 2) * img_h
        boxes.append([x1, y1, x2, y2])

    return np.array(boxes) if boxes else np.empty((0, 4))


def box_iou_xyxy(box, boxes):
    """IoU between one box [4] and an array of boxes [N,4]."""
    if len(boxes) == 0:
        return np.empty((0,))
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_box = max(0, box[2] - box[0]) * max(0, box[3] - box[1])
    area_boxes = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
    union = area_box + area_boxes - inter
    return np.where(union > 0, inter / union, 0)


def match_ashpilanta_detections(pred_boxes, gt_boxes, iou_thresh=0.5):
    """Greedy IoU matching, Ashpilanta-only (both inputs are already
    filtered to this single class, so no class comparison is needed).
    Returns (TP, FP, FN) as plain integers."""
    gt_matched = np.zeros(len(gt_boxes), dtype=bool)
    tp, fp = 0, 0

    for pb in pred_boxes:
        unmatched_idx = np.where(~gt_matched)[0]
        if len(unmatched_idx) == 0:
            fp += 1
            continue
        ious = box_iou_xyxy(pb, gt_boxes[unmatched_idx])
        best = np.argmax(ious) if len(ious) else -1
        if best >= 0 and ious[best] >= iou_thresh:
            gt_matched[unmatched_idx[best]] = True
            tp += 1
        else:
            fp += 1

    fn = int((~gt_matched).sum())
    return tp, fp, fn


In [ ]:
# ============================================================
# CELL 23 — Evaluation harness (Ashpilanta only): run a pipeline over a
# dataset, compute Ashpilanta precision/recall/F1/count-MAE/timing
#
# UPDATED: added a "baseline_clip" pipeline option -- single-pass
# detection (the crop style CLIP was actually calibrated on in Cell 13)
# with the CLIP filter applied on top, to test whether CLIP can improve
# on baseline's already-strong precision without tiling's domain-mismatch
# problems.
# ============================================================

def evaluate_field_pipeline(image_dir, label_dir, pipeline="main",
                             iou_thresh=0.5, deploy_conf=DEPLOY_CONF,
                             use_clip_filter=True, max_images=None,
                             gt_ap_class_id=None):
    """
    pipeline = "main"          -> tiled inference + cross-tile NMS + CLIP filter
                                   (measured to perform poorly -- domain mismatch,
                                   kept for comparison/documentation purposes)
    pipeline = "baseline"      -> single-pass detection only, no CLIP filter
    pipeline = "baseline_clip" -> single-pass detection + CLIP filter (crop
                                   style matches what Cell 13 calibrated on)

    gt_ap_class_id: the class ID that means "Ashpilanta" in THIS label_dir's
    files. Defaults to the global AP_CLASS_ID (3, the merged 4-class
    scheme). Pass 0 explicitly when evaluating directly against raw
    generated_dataset_v6 label files (generator's own scheme).
    """
    effective_gt_class_id = gt_ap_class_id if gt_ap_class_id is not None else AP_CLASS_ID

    image_dir, label_dir = Path(image_dir), Path(label_dir)
    img_paths = sorted(list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.jpeg")) +
                        list(image_dir.glob("*.png")))
    if max_images:
        img_paths = img_paths[:max_images]

    total_tp, total_fp, total_fn = 0, 0, 0
    ashpilanta_abs_errors = []
    per_image_times = []

    print(f"Evaluating pipeline='{pipeline}' on {len(img_paths)} images "
          f"(Ashpilanta only, ground-truth class ID={effective_gt_class_id})...")

    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        gt_label_path = label_dir / f"{img_path.stem}.txt"
        gt_boxes = load_ashpilanta_gt_as_xyxy(gt_label_path, w, h,
                                               ap_class_id=effective_gt_class_id)

        t0 = time.time()

        if pipeline == "main":
            boxes, scores, classes, img_loaded = tiled_predict(img_path)
            boxes, scores, classes = cross_tile_nms(boxes, scores, classes)
            keep = scores >= deploy_conf
            boxes, scores, classes = boxes[keep], scores[keep], classes[keep]
            if use_clip_filter:
                boxes, scores, classes = clip_filter_ashpilanta(boxes, scores, classes, img)

        elif pipeline in ("baseline", "baseline_clip"):
            results = _deploy_yolo_model.predict(source=str(img_path), conf=deploy_conf,
                                                  verbose=False, save=False)
            r = results[0]
            if r.boxes is not None and len(r.boxes) > 0:
                all_boxes = r.boxes.xyxy.cpu().numpy()
                all_scores = r.boxes.conf.cpu().numpy()
                all_classes = r.boxes.cls.cpu().numpy().astype(int)
                ap_mask = all_classes == AP_CLASS_ID
                boxes, scores, classes = all_boxes[ap_mask], all_scores[ap_mask], all_classes[ap_mask]
            else:
                boxes, scores, classes = np.empty((0, 4)), np.empty((0,)), np.empty((0,), dtype=int)

            if pipeline == "baseline_clip":
                boxes, scores, classes = clip_filter_ashpilanta(boxes, scores, classes, img)

        else:
            raise ValueError(f"Unknown pipeline: {pipeline}")

        elapsed = time.time() - t0
        per_image_times.append(elapsed)

        tp, fp, fn = match_ashpilanta_detections(boxes, gt_boxes, iou_thresh=iou_thresh)
        total_tp += tp
        total_fp += fp
        total_fn += fn

        ashpilanta_abs_errors.append(abs(len(boxes) - len(gt_boxes)))

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    summary = {
        "pipeline": pipeline,
        "n_images": len(img_paths),
        "TP": total_tp, "FP": total_fp, "FN": total_fn,
        "ashpilanta_precision": round(precision, 4),
        "ashpilanta_recall": round(recall, 4),
        "ashpilanta_f1": round(f1, 4),
        "ashpilanta_count_MAE": round(float(np.mean(ashpilanta_abs_errors)), 4) if ashpilanta_abs_errors else None,
        "avg_inference_time_sec": round(float(np.mean(per_image_times)), 4) if per_image_times else None,
        "total_inference_time_sec": round(float(np.sum(per_image_times)), 2) if per_image_times else None,
    }

    print(f"\n{pipeline} pipeline results (Ashpilanta only):")
    print(f"  TP={total_tp}  FP={total_fp}  FN={total_fn}")
    print(f"  Precision={summary['ashpilanta_precision']:.3f}  "
          f"Recall={summary['ashpilanta_recall']:.3f}  F1={summary['ashpilanta_f1']:.3f}")
    print(f"  Count MAE={summary['ashpilanta_count_MAE']}  "
          f"avg time={summary['avg_inference_time_sec']:.3f}s/image")

    return summary


In [ ]:
# ============================================================
# CELL 24 — Run both pipelines on the target-domain validation set
# (Ashpilanta only)
# ============================================================

TEST_IMAGE_DIR = PATH_TARGET_VAL / "images" / "val"
TEST_LABEL_DIR = PATH_TARGET_VAL / "labels" / "val"

print(f"Test set: {TEST_IMAGE_DIR}")
print(f"Images available: {len(list(TEST_IMAGE_DIR.glob('*.jpg'))) + len(list(TEST_IMAGE_DIR.glob('*.png')))}")

MAX_TEST_IMAGES = 20   # <-- set to None for the full validation set

main_summary = evaluate_field_pipeline(
    TEST_IMAGE_DIR, TEST_LABEL_DIR, pipeline="main",
    deploy_conf=DEPLOY_CONF, use_clip_filter=True, max_images=MAX_TEST_IMAGES,
)

baseline_summary = evaluate_field_pipeline(
    TEST_IMAGE_DIR, TEST_LABEL_DIR, pipeline="baseline",
    deploy_conf=DEPLOY_CONF, max_images=MAX_TEST_IMAGES,
)


In [ ]:
# ============================================================
# CELL 24b — DIAGNOSTIC: break down Ashpilanta-only results by image size
# ============================================================

def evaluate_field_pipeline_bucketed(image_dir, label_dir, size_threshold=800,
                                      deploy_conf=DEPLOY_CONF, use_clip_filter=True,
                                      max_images=None):
    """Splits results into 'small' (max dimension <= size_threshold, e.g.
    real close-up reference photos) vs 'large' (e.g. synthetic full-scene
    composites), Ashpilanta only."""
    image_dir, label_dir = Path(image_dir), Path(label_dir)
    img_paths = sorted(list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.jpeg")) +
                        list(image_dir.glob("*.png")))
    if max_images:
        img_paths = img_paths[:max_images]

    buckets = {
        "small": {"tp": 0, "fp": 0, "fn": 0, "n_images": 0, "tile_counts": []},
        "large": {"tp": 0, "fp": 0, "fn": 0, "n_images": 0, "tile_counts": []},
    }

    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        bucket = "small" if max(h, w) <= size_threshold else "large"

        n_tiles = len(make_tiles(w, h))
        buckets[bucket]["tile_counts"].append(n_tiles)
        buckets[bucket]["n_images"] += 1

        gt_label_path = label_dir / f"{img_path.stem}.txt"
        gt_boxes = load_ashpilanta_gt_as_xyxy(gt_label_path, w, h)

        boxes, scores, classes, _ = tiled_predict(img_path)
        boxes, scores, classes = cross_tile_nms(boxes, scores, classes)
        keep = scores >= deploy_conf
        boxes, scores, classes = boxes[keep], scores[keep], classes[keep]
        if use_clip_filter:
            boxes, scores, classes = clip_filter_ashpilanta(boxes, scores, classes, img)

        tp, fp, fn = match_ashpilanta_detections(boxes, gt_boxes, iou_thresh=0.5)
        buckets[bucket]["tp"] += tp
        buckets[bucket]["fp"] += fp
        buckets[bucket]["fn"] += fn

    print(f"\n{'='*70}")
    print(f"SIZE-BUCKETED DIAGNOSTIC (Ashpilanta only, threshold: "
          f"max(h,w) <= {size_threshold}px = 'small')")
    print(f"{'='*70}")

    for bucket_name, data in buckets.items():
        if data["n_images"] == 0:
            print(f"\n[{bucket_name}] no images in this bucket")
            continue
        precision = data["tp"] / (data["tp"] + data["fp"]) if (data["tp"] + data["fp"]) > 0 else 0.0
        recall = data["tp"] / (data["tp"] + data["fn"]) if (data["tp"] + data["fn"]) > 0 else 0.0
        avg_tiles = np.mean(data["tile_counts"]) if data["tile_counts"] else 0

        print(f"\n[{bucket_name}] n_images={data['n_images']}  avg_tiles/image={avg_tiles:.1f}")
        print(f"  TP={data['tp']}  FP={data['fp']}  FN={data['fn']}  "
              f"P={precision:.3f}  R={recall:.3f}")

    return buckets


diagnostic_results = evaluate_field_pipeline_bucketed(
    TEST_IMAGE_DIR, TEST_LABEL_DIR, size_threshold=800,
    deploy_conf=DEPLOY_CONF, use_clip_filter=True, max_images=MAX_TEST_IMAGES,
)


In [ ]:
# ============================================================
# CELL 24c — TUNING: sweep TILE_CONF on the large-image bucket
# (Ashpilanta only)
# ============================================================

TILE_CONF_CANDIDATES = [0.25, 0.35, 0.45, 0.55]


def evaluate_large_bucket_at_tile_conf(image_dir, label_dir, tile_conf,
                                        size_threshold=800, deploy_conf=DEPLOY_CONF,
                                        use_clip_filter=True, max_images=None):
    image_dir, label_dir = Path(image_dir), Path(label_dir)
    img_paths = sorted(list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.jpeg")) +
                        list(image_dir.glob("*.png")))
    if max_images:
        img_paths = img_paths[:max_images]

    tp_total, fp_total, fn_total = 0, 0, 0
    n_large = 0

    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        if max(h, w) <= size_threshold:
            continue
        n_large += 1

        gt_label_path = label_dir / f"{img_path.stem}.txt"
        gt_boxes = load_ashpilanta_gt_as_xyxy(gt_label_path, w, h)

        boxes, scores, classes, _ = tiled_predict(img_path, conf=tile_conf)
        boxes, scores, classes = cross_tile_nms(boxes, scores, classes)
        keep = scores >= deploy_conf
        boxes, scores, classes = boxes[keep], scores[keep], classes[keep]
        if use_clip_filter:
            boxes, scores, classes = clip_filter_ashpilanta(boxes, scores, classes, img)

        tp, fp, fn = match_ashpilanta_detections(boxes, gt_boxes, iou_thresh=0.5)
        tp_total += tp
        fp_total += fp
        fn_total += fn

    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0.0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0.0

    return {
        "tile_conf": tile_conf, "n_large_images": n_large,
        "TP": tp_total, "FP": fp_total, "FN": fn_total,
        "precision": round(precision, 4), "recall": round(recall, 4),
    }


sweep_results = []
for tc in TILE_CONF_CANDIDATES:
    print(f"\nTesting TILE_CONF={tc} on large-image bucket (Ashpilanta only)...")
    result = evaluate_large_bucket_at_tile_conf(
        TEST_IMAGE_DIR, TEST_LABEL_DIR, tile_conf=tc,
        size_threshold=800, deploy_conf=DEPLOY_CONF, use_clip_filter=True,
        max_images=MAX_TEST_IMAGES,
    )
    sweep_results.append(result)
    print(result)

sweep_df = pd.DataFrame(sweep_results)
print("\n" + "=" * 70)
print("TILE_CONF SWEEP RESULTS (large-image bucket, Ashpilanta only)")
print("=" * 70)
print(sweep_df.to_string(index=False))

sweep_csv = PATH_RESULTS / "tile_conf_sweep_large_bucket_ashpilanta.csv"
sweep_df.to_csv(sweep_csv, index=False)
print(f"\nSaved: {sweep_csv}")


In [ ]:
# ============================================================
# CELL 24d — DIAGNOSTIC: trace one large image stage-by-stage
# (Ashpilanta only)
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

DEBUG_IMAGE_NAME = "synth_val_synth_001700.jpg"   # <-- change if not in your folder
debug_img_path = TEST_IMAGE_DIR / DEBUG_IMAGE_NAME

img = cv2.imread(str(debug_img_path))
h, w = img.shape[:2]
gt_boxes = load_ashpilanta_gt_as_xyxy(TEST_LABEL_DIR / f"{debug_img_path.stem}.txt", w, h)

print(f"Image: {DEBUG_IMAGE_NAME}  ({w}x{h}px)")
print(f"Ashpilanta ground truth boxes: {len(gt_boxes)}")

raw_boxes, raw_scores, raw_classes, _ = tiled_predict(debug_img_path)
print(f"\nStage 1 -- raw tiled_predict (Ashpilanta only): {len(raw_boxes)} detections")
if len(raw_boxes):
    print(f"  score range: {raw_scores.min():.3f} - {raw_scores.max():.3f}")

nms_boxes, nms_scores, nms_classes = cross_tile_nms(raw_boxes, raw_scores, raw_classes)
print(f"Stage 2 -- after cross_tile_nms:   {len(nms_boxes)} detections "
      f"(removed {len(raw_boxes) - len(nms_boxes)})")

keep = nms_scores >= DEPLOY_CONF
conf_boxes, conf_scores, conf_classes = nms_boxes[keep], nms_scores[keep], nms_classes[keep]
print(f"Stage 3 -- after DEPLOY_CONF={DEPLOY_CONF}:  {len(conf_boxes)} detections "
      f"(removed {len(nms_boxes) - len(conf_boxes)})")

clip_boxes, clip_scores, clip_classes = clip_filter_ashpilanta(conf_boxes, conf_scores, conf_classes, img)
print(f"Stage 4 -- after CLIP filter:      {len(clip_boxes)} detections "
      f"(removed {len(conf_boxes) - len(clip_boxes)})")

if len(raw_boxes) and len(gt_boxes):
    overlap_found = 0
    for gt_box in gt_boxes:
        ious = box_iou_xyxy(gt_box, raw_boxes)
        if len(ious) and ious.max() >= 0.5:
            overlap_found += 1
    print(f"\nOf {len(gt_boxes)} Ashpilanta ground-truth boxes, {overlap_found} "
          f"have a raw Ashpilanta prediction overlapping at IoU>=0.5.")

scale = 1200 / max(w, h)
vis = cv2.resize(img, (int(w * scale), int(h * scale)))
vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(14, 14 * h / w))
ax.imshow(vis_rgb)

for box in gt_boxes:
    x1, y1, x2, y2 = box * scale
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                            edgecolor="lime", facecolor="none", linewidth=1.5))

for box in raw_boxes:
    x1, y1, x2, y2 = box * scale
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                            edgecolor="red", facecolor="none", linewidth=0.8, alpha=0.6))

ax.set_title(f"{DEBUG_IMAGE_NAME}  |  green=Ashpilanta GT ({len(gt_boxes)})  "
             f"red=raw Ashpilanta predictions ({len(raw_boxes)})")
ax.axis("off")
plt.tight_layout()
plt.savefig(PATH_RESULTS / "debug_large_image_overlay.png", dpi=150)
plt.show()
print(f"\nSaved: {PATH_RESULTS / 'debug_large_image_overlay.png'}")


In [ ]:
# ============================================================
# CELL 25 — Comparison table: Main Solution vs single-pass baseline
# (Ashpilanta only)
# ============================================================

comparison_df = pd.DataFrame([main_summary, baseline_summary])
comparison_df = comparison_df[["pipeline", "n_images", "TP", "FP", "FN",
                                "ashpilanta_precision", "ashpilanta_recall",
                                "ashpilanta_f1", "ashpilanta_count_MAE",
                                "avg_inference_time_sec", "total_inference_time_sec"]]

print("\n" + "=" * 70)
print("MAIN SOLUTION vs BASELINE -- Ashpilanta only")
print("=" * 70)
print(comparison_df.to_string(index=False))

out_csv = PATH_RESULTS / "main_solution_vs_baseline_ashpilanta.csv"
comparison_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")


In [ ]:
# ============================================================
# CELL 26 — Comparison plot (Ashpilanta only)
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ["ashpilanta_precision", "ashpilanta_recall", "ashpilanta_f1"]
titles = ["Precision", "Recall", "F1"]
for ax, metric, title in zip(axes, metrics, titles):
    ax.bar(comparison_df["pipeline"], comparison_df[metric], color=["#4C72B0", "#DD8452"])
    ax.set_title(f"Ashpilanta {title}")
    ax.set_ylim(0, 1)
    for i, v in enumerate(comparison_df[metric]):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plot_path = PATH_RESULTS / "main_solution_vs_baseline_ashpilanta.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved: {plot_path}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.bar(comparison_df["pipeline"], comparison_df["ashpilanta_count_MAE"], color=["#4C72B0", "#DD8452"])
ax1.set_title("Ashpilanta Count MAE (lower is better)")
ax2.bar(comparison_df["pipeline"], comparison_df["avg_inference_time_sec"], color=["#4C72B0", "#DD8452"])
ax2.set_title("Avg inference time per image (sec)")
plt.tight_layout()
plt.savefig(PATH_RESULTS / "main_solution_count_and_speed_ashpilanta.png", dpi=150)
plt.show()


In [ ]:
# ============================================================
# CELL 27 — Qualitative check: run the Main Solution on a few sample images
# ============================================================

import matplotlib.pyplot as plt
from PIL import Image as PILImage

sample_paths = sorted(list(TEST_IMAGE_DIR.glob("*.jpg")) + list(TEST_IMAGE_DIR.glob("*.png")))[:3]

for img_path in sample_paths:
    det_df, summary, annotated_path = detect_ashpilanta_field_image(img_path)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(PILImage.open(annotated_path))
    ax.set_title(f"{img_path.name}  |  Ashpilanta count: {summary['total_ashpilanta']}")
    ax.axis("off")
    plt.show()


In [ ]:
# ============================================================
# CELL 28 — Main Solution evaluated on the RAW SYNTHETIC GENERATOR
# DATASET (generated_dataset_v6) specifically -- not the mixed
# real+synthetic target_domain_val set used in Cell 24.
#
# UPDATED: raw generated_dataset_v6 label files use the synthetic
# generator's OWN internal class scheme, where Ashpilanta = class 0
# (not class 3, which is only used in the merged 4-class datasets).
# SYNTH_AP_CLASS_ID below reflects this and is passed explicitly to
# evaluate_field_pipeline via gt_ap_class_id.
# ============================================================

SYNTH_IMAGE_DIR = PATH_SYNTH / "images" / "val"
SYNTH_LABEL_DIR = PATH_SYNTH / "labels" / "val"

# Generator's own schema: Aspilanta = 0, distractor moth = 1.
# (See exp2_eval.yaml: nc=1, names=['Aspilanta'] -- confirms class 0.)
SYNTH_AP_CLASS_ID = 0

n_synth_val = (len(list(SYNTH_IMAGE_DIR.glob("*.jpg"))) +
               len(list(SYNTH_IMAGE_DIR.glob("*.png"))))
print(f"Synthetic generator dataset: {PATH_SYNTH}")
print(f"Val images available: {n_synth_val}")
print(f"Ground truth Ashpilanta class ID in this dataset: {SYNTH_AP_CLASS_ID} "
      f"(generator's own schema, NOT the merged-dataset class 3)")

if n_synth_val == 0:
    raise FileNotFoundError(
        f"No validation images found at {SYNTH_IMAGE_DIR}. "
        "Confirm generated_dataset_v6 was restored/copied "
        "(see Cell 'FIND DATASETS AND SPLIT')."
    )

MAX_SYNTH_TEST_IMAGES = 30   # set to None to run the full synthetic val split

synth_main_summary = evaluate_field_pipeline(
    SYNTH_IMAGE_DIR, SYNTH_LABEL_DIR, pipeline="main",
    deploy_conf=DEPLOY_CONF, use_clip_filter=True, max_images=MAX_SYNTH_TEST_IMAGES,
    gt_ap_class_id=SYNTH_AP_CLASS_ID,
)

synth_baseline_summary = evaluate_field_pipeline(
    SYNTH_IMAGE_DIR, SYNTH_LABEL_DIR, pipeline="baseline",
    deploy_conf=DEPLOY_CONF, max_images=MAX_SYNTH_TEST_IMAGES,
    gt_ap_class_id=SYNTH_AP_CLASS_ID,
)

synth_comparison_df = pd.DataFrame([synth_main_summary, synth_baseline_summary])
synth_comparison_df = synth_comparison_df[["pipeline", "n_images", "TP", "FP", "FN",
                                            "ashpilanta_precision", "ashpilanta_recall",
                                            "ashpilanta_f1", "ashpilanta_count_MAE",
                                            "avg_inference_time_sec"]]

print("\n" + "=" * 70)
print("MAIN SOLUTION vs BASELINE -- generated_dataset_v6 ONLY (Ashpilanta only)")
print("=" * 70)
print(synth_comparison_df.to_string(index=False))

synth_out_csv = PATH_RESULTS / "main_solution_vs_baseline_synthetic_only.csv"
synth_comparison_df.to_csv(synth_out_csv, index=False)
print(f"\nSaved: {synth_out_csv}")

In [ ]:
# ============================================================
# CELL 28b — Test baseline_clip on generated_dataset_v6: does CLIP
# filtering, correctly paired with single-pass detection, improve on
# baseline's already-strong precision without destroying recall?
# Run this AFTER Cell 28.
# ============================================================

synth_baseline_clip_summary = evaluate_field_pipeline(
    SYNTH_IMAGE_DIR, SYNTH_LABEL_DIR, pipeline="baseline_clip",
    deploy_conf=DEPLOY_CONF, max_images=MAX_SYNTH_TEST_IMAGES,
    gt_ap_class_id=SYNTH_AP_CLASS_ID,
)

three_way_df = pd.DataFrame([synth_main_summary, synth_baseline_summary, synth_baseline_clip_summary])
three_way_df = three_way_df[["pipeline", "n_images", "TP", "FP", "FN",
                              "ashpilanta_precision", "ashpilanta_recall",
                              "ashpilanta_f1", "ashpilanta_count_MAE",
                              "avg_inference_time_sec"]]

print("\n" + "=" * 70)
print("THREE-WAY COMPARISON on generated_dataset_v6 (Ashpilanta only)")
print("=" * 70)
print(three_way_df.to_string(index=False))

three_way_csv = PATH_RESULTS / "three_way_pipeline_comparison_synthetic.csv"
three_way_df.to_csv(three_way_csv, index=False)
print(f"\nSaved: {three_way_csv}")

# Resolve the trade-off automatically instead of leaving it as a
# conditional for the reader -- decide which pipeline the evidence
# actually supports, and say so explicitly.
main_row = three_way_df[three_way_df["pipeline"] == "main"].iloc[0]
base_row = three_way_df[three_way_df["pipeline"] == "baseline"].iloc[0]
clip_row = three_way_df[three_way_df["pipeline"] == "baseline_clip"].iloc[0]

precision_gain = clip_row["ashpilanta_precision"] - base_row["ashpilanta_precision"]
recall_drop = base_row["ashpilanta_recall"] - clip_row["ashpilanta_recall"]
f1_gain = clip_row["ashpilanta_f1"] - base_row["ashpilanta_f1"]

# Threshold for "worth it": CLIP must not cost more than 2x what it gains
# in precision, measured in recall points, and must not reduce F1.
clip_is_worth_it = (recall_drop <= 2 * precision_gain) and (f1_gain >= 0)

print(f"\nPrecision change (baseline_clip vs baseline): {precision_gain:+.4f}")
print(f"Recall change (baseline_clip vs baseline):    {-recall_drop:+.4f}")
print(f"F1 change (baseline_clip vs baseline):        {f1_gain:+.4f}")

if clip_is_worth_it:
    recommended_pipeline = "baseline_clip"
    print(f"\n-> CLIP filtering IS worth it here: precision improved without a "
          f"disproportionate recall cost, and F1 did not drop. "
          f"Use pipeline='baseline_clip' as the reported Main Solution result.")
else:
    recommended_pipeline = "baseline"
    print(f"\n-> CLIP filtering is NOT worth it here: it costs {recall_drop:.4f} "
          f"recall for only {precision_gain:.4f} precision, a net F1 change of "
          f"{f1_gain:+.4f}. Use pipeline='baseline' (single-pass, no CLIP filter) "
          f"as the reported Main Solution result.")

print(f"(For reference: tiled 'main' pipeline scored F1={main_row['ashpilanta_f1']:.4f} "
      f"-- confirmed worst of the three regardless of the CLIP trade-off above.)")


In [ ]:
# ============================================================
# CELL 29 — Qualitative: Ashpilanta detected among distractors + other
# insect classes, directly in the raw synthetic generator scenes
#
# UPDATED: uses SYNTH_AP_CLASS_ID (0) from Cell 28 when reading ground
# truth from generated_dataset_v6's own label files, since this raw
# dataset uses the generator's internal class scheme, not the merged
# 4-class scheme the model itself predicts in.
# ============================================================

import matplotlib.pyplot as plt
from PIL import Image as PILImage

synth_sample_paths = sorted(
    list(SYNTH_IMAGE_DIR.glob("*.jpg")) + list(SYNTH_IMAGE_DIR.glob("*.png"))
)[:4]

if not synth_sample_paths:
    raise FileNotFoundError(f"No sample images found in {SYNTH_IMAGE_DIR}")

for img_path in synth_sample_paths:
    raw_img = cv2.imread(str(img_path))
    h, w = raw_img.shape[:2]
    gt_boxes = load_ashpilanta_gt_as_xyxy(
        SYNTH_LABEL_DIR / f"{img_path.stem}.txt", w, h,
        ap_class_id=SYNTH_AP_CLASS_ID,
    )

    det_df, summary, annotated_path = detect_ashpilanta_field_image(img_path)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(PILImage.open(annotated_path))
    ax.set_title(
        f"{img_path.name}\n"
        f"Predicted Ashpilanta: {summary['total_ashpilanta']}  |  "
        f"Ground truth Ashpilanta: {len(gt_boxes)}"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"\nAll images above are pulled directly from generated_dataset_v6, "
      f"the compositor's own output -- each scene contains MR/NC/WF "
      f"instances and distractor insects alongside Ashpilanta on the "
      f"sticky pad background. Only Ashpilanta is boxed and counted, "
      f"per the Main Solution's filtering (Cell 17).")

## Section 8 — Ablation Studies: proving every part of the methodology

Four techniques make up the proposed method: **copy-paste compositing**, **YOLO transfer
learning**, **fine-tuning**, and **metric-learning (CLIP few-shot filter)**. This section adds
one controlled experiment per design choice inside each technique, isolating exactly one
variable at a time and holding everything else fixed at the main-solution recipe.

Run this after Experiment 3 (Cell 58) and its evaluation cell (Cell 60, which defines `m_exp3`,
`YAML_TARGET_VAL`, `get_class_ap`, `safe_float`) have completed — those are reused below.

**Items 10–11 (the metric-learning / CLIP ablations) are already implemented in Cells 13 and
28b of this notebook** (`CLIP_K_VALUES` sweep, and the `main` vs `baseline` vs `baseline_clip`
three-way comparison) — just run those cells; nothing new is needed for that technique.
The new cells below cover items 1–9 (compositing + transfer learning + fine-tuning).

### 8.1 — Copy-paste compositing: regenerate synthetic data with each component isolated

In [ ]:
# ==============================================================================
#  ABLATION — SELF-CONTAINED variant generator (fixed).
#  Does NOT depend on Section 2 / Cell 40 having run in this kernel -- all
#  constants and helper functions needed for compositing are redefined here,
#  identical to the values in the original v6 generator, so the ablation
#  isolates only the flag being tested.
#
#  It DOES still depend on the raw per-insect cutout manifests
#  (aspilanta_cutouts.json / distractor_cutouts.json) existing, because
#  ablating blending/distractors/pad-detection requires access to the
#  individual un-composited cutouts -- there is no way around that. If you
#  have not run the raw extraction step (Cells "KAGGLE CELL A" / "KAGGLE
#  CELL B", i.e. cells 38-39) in THIS session, run those first -- the check
#  below will tell you clearly if they are missing.
# ==============================================================================

import cv2, numpy as np, random, json, warnings, os, math
from pathlib import Path
from scipy.stats import truncnorm
import torch
import torch.nn.functional as TF

warnings.filterwarnings("ignore")

# ── Paths & constants (must match the original v6 generator exactly) ─────────
BASE                    = Path("/kaggle/working")
PATH_BACKGROUNDS        = Path(
    "/kaggle/input/datasets/your-kaggle-username/ashpilanta-3wtop/"
    "data gen-3/Many Backgrounds")
OUTPUTS                 = BASE / "outputs"
asp_manifest            = OUTPUTS / "aspilanta_cutouts.json"
dist_manifest           = OUTPUTS / "distractor_cutouts.json"

INSECT_MEAN_MM   = 5.0
INSECT_STD_MM    = 0.8
INSECT_MIN_MM    = 2.5
INSECT_MAX_MM    = 8.0
PAD_WIDTH_MM     = 140.0
NUM_IMAGES       = 2000        # <-- the missing definition that caused the NameError
TRAIN_RATIO      = 0.85
CLS_ASPILANTA    = 0
CLS_DISTRACTOR   = 1
NAME_ASPILANTA   = "Aspilanta"
NAME_DISTRACTOR  = "eu_moth"
MAX_ASPILANTA    = 10
MAX_DISTRACTOR   = 5
MAX_ATTEMPTS     = 40
PAD_MIN_FRAC     = 0.05
PAD_MAX_FRAC     = 0.95
ORGANIC_K        = (3, 3)
ORGANIC_S        = 0.5
SHEEN_PROB       = 0.30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Upfront, actionable check (instead of a deep stack trace later) ──────────
if not asp_manifest.exists() or not dist_manifest.exists():
    raise RuntimeError(
        "Raw cutout manifests not found -- needed to run the compositing "
        "ablations (Poisson blending / distractors / pad-aware placement).\n"
        f"  Expected: {asp_manifest}\n"
        f"            {dist_manifest}\n\n"
        "These are produced by the raw extraction step, NOT by the "
        "'RESTORE SYNTHETIC DATA' shortcut (which only restores the final "
        "composited images, not the individual cutouts).\n\n"
        "Fix: run cells 38 and 39 ('KAGGLE CELL A' / 'KAGGLE CELL B') in this "
        "session first, then re-run this cell. That step is slow (subprocess "
        "background removal per insect file) but only needs to happen once "
        "per session -- all three compositing ablations reuse the same manifests."
    )


def load_bgra_images(paths: list, label: str) -> list:
    imgs = []
    for p in paths:
        img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        if img is None:
            continue
        if img.ndim == 3 and img.shape[2] == 4:
            imgs.append(img)
        elif img.ndim == 3 and img.shape[2] == 3:
            bgra        = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
            bgra[:,:,3] = 255
            imgs.append(bgra)
    print(f"  {label:<25} {len(imgs)} cutouts loaded")
    return imgs


def bgr_to_tensor(img):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return (torch.from_numpy(rgb.copy())
            .permute(2,0,1).float().div(255.0)
            .unsqueeze(0).to(DEVICE))

def tensor_to_bgr(t):
    arr = (t.squeeze(0).permute(1,2,0)
            .clamp(0,1).mul(255).byte().cpu().numpy())
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

def gpu_resize_bgra(img, new_w, new_h):
    try:
        t   = bgr_to_tensor(img[:,:,:3])
        r   = TF.interpolate(t, size=(new_h, new_w),
                             mode='bilinear', align_corners=False)
        bgr = tensor_to_bgr(r)
    except Exception:
        bgr = cv2.resize(img[:,:,:3], (new_w, new_h),
                         interpolation=cv2.INTER_AREA)
    alpha       = cv2.resize(img[:,:,3], (new_w, new_h),
                             interpolation=cv2.INTER_AREA)
    bgra        = cv2.cvtColor(bgr, cv2.COLOR_BGR2BGRA)
    bgra[:,:,3] = alpha
    return bgra


def sample_size_px(pad_width_px):
    px_per_mm = pad_width_px / PAD_WIDTH_MM
    a = (INSECT_MIN_MM - INSECT_MEAN_MM) / INSECT_STD_MM
    b = (INSECT_MAX_MM - INSECT_MEAN_MM) / INSECT_STD_MM
    mm = truncnorm.rvs(a, b, loc=INSECT_MEAN_MM, scale=INSECT_STD_MM)
    return max(4, int(round(mm * px_per_mm)))


def detect_pad_bbox(bg):
    hsv  = cv2.cvtColor(bg, cv2.COLOR_BGR2HSV)
    H, W = bg.shape[:2]
    mask = (
        cv2.inRange(hsv, ( 18, 70,  80), ( 38,255,255)) |
        cv2.inRange(hsv, (  0,  0, 170), (180, 50,255)) |
        cv2.inRange(hsv, ( 90, 50,  60), (135,255,255)) |
        cv2.inRange(hsv, ( 40, 50,  50), ( 90,255,220))
    )
    k    = cv2.getStructuringElement(cv2.MORPH_RECT,(20,20))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k, iterations=2)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return (0,0,W,H), False
    largest = max(cnts, key=cv2.contourArea)
    area    = cv2.contourArea(largest)
    if not (H*W*PAD_MIN_FRAC < area < H*W*PAD_MAX_FRAC):
        return (0,0,W,H), False
    x,y,w,h = cv2.boundingRect(largest)
    inset    = max(8, min(w,h)//25)
    return (x+inset, y+inset, w-2*inset, h-2*inset), True


def extract_clean_pad(bg):
    (px,py,pw,ph), found = detect_pad_bbox(bg)
    if not found or pw<50 or ph<50:
        H,W   = bg.shape[:2]
        mx,my = int(W*0.10), int(H*0.10)
        return (bg[my:my+int(H*0.80), mx:mx+int(W*0.80)].copy(),
                (mx,my,int(W*0.80),int(H*0.80)), False)
    canvas = bg[py:py+ph, px:px+pw].copy()
    gray   = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    blur   = cv2.GaussianBlur(gray,(51,51),0)
    diff   = cv2.subtract(blur,gray)
    _,dm   = cv2.threshold(diff,18,255,cv2.THRESH_BINARY)
    k2     = np.ones((3,3),np.uint8)
    dm     = cv2.morphologyEx(dm,cv2.MORPH_CLOSE,k2,iterations=2)
    dm     = cv2.morphologyEx(dm,cv2.MORPH_OPEN, k2,iterations=1)
    n_lbl,labels,sts,_ = cv2.connectedComponentsWithStats(dm)
    clean  = np.zeros_like(dm)
    for lid in range(1,n_lbl):
        if 8 <= sts[lid,cv2.CC_STAT_AREA] <= 2000:
            clean[labels==lid] = 255
    if clean.any():
        clean  = cv2.dilate(clean,k2,iterations=2)
        canvas = cv2.inpaint(canvas,clean,5,cv2.INPAINT_TELEA)
    return canvas,(px,py,pw,ph),True


def match_local_lighting(ins_bgr: np.ndarray, bg: np.ndarray,
                          x: int, y: int, w: int, h: int,
                          strength: float = 0.20) -> np.ndarray:
    bh, bw = bg.shape[:2]
    margin  = max(10, w // 2)
    rx1     = max(0, x - margin)
    ry1     = max(0, y - margin)
    rx2     = min(bw, x + w + margin)
    ry2     = min(bh, y + h + margin)
    patch   = bg[ry1:ry2, rx1:rx2]
    if patch.size == 0:
        return ins_bgr
    local_mean = patch.mean(axis=(0,1))
    ins_f      = ins_bgr.astype(np.float32)
    blended    = ins_f * (1 - strength) + local_mean * strength
    return np.clip(blended, 0, 255).astype(np.uint8)


def color_jitter_insect(ins_bgr: np.ndarray) -> np.ndarray:
    out   = ins_bgr.astype(np.float32)
    alpha = random.uniform(0.80, 1.20)
    beta  = random.uniform(-15,  15)
    out   = np.clip(out * alpha + beta, 0, 255)
    if random.random() < 0.50:
        hsv       = cv2.cvtColor(out.astype(np.uint8), cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,0] = (hsv[:,:,0] + random.uniform(-8, 8)) % 180
        hsv[:,:,1] = np.clip(hsv[:,:,1] * random.uniform(0.85, 1.15), 0, 255)
        out        = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR).astype(np.float32)
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_motion_blur(img_bgr: np.ndarray, max_kernel: int = 5) -> np.ndarray:
    k    = random.choice([3, 5]) if max_kernel >= 5 else 3
    ang  = random.uniform(0, 180)
    M    = cv2.getRotationMatrix2D((k//2, k//2), ang, 1)
    kern = np.zeros((k, k), np.float32)
    kern[k//2, :] = 1.0
    kern          = cv2.warpAffine(kern, M, (k, k))
    kern         /= kern.sum() + 1e-9
    return cv2.filter2D(img_bgr, -1, kern)


def add_shadow(bg, x, y, w, h, alpha_mask):
    out      = bg.copy()
    dx       = random.randint(1, max(2, w//8))
    dy       = random.randint(1, max(2, h//8))
    strength = random.uniform(0.06, 0.18)
    shadow   = cv2.dilate(alpha_mask, np.ones((5,5),np.uint8), iterations=1)
    shadow   = cv2.GaussianBlur(shadow, (0,0), float(max(1, w//6)))
    shadow   = shadow.astype(np.float32)/255.0 * strength
    sx1,sy1  = max(0,x+dx), max(0,y+dy)
    sx2,sy2  = min(bg.shape[1],x+dx+w), min(bg.shape[0],y+dy+h)
    smx1,smy1 = sx1-(x+dx), sy1-(y+dy)
    smx2,smy2 = smx1+(sx2-sx1), smy1+(sy2-sy1)
    if (sx2>sx1 and sy2>sy1
            and smx2<=shadow.shape[1] and smy2<=shadow.shape[0]):
        roi = out[sy1:sy2,sx1:sx2].astype(np.float32)
        sm  = shadow[smy1:smy2,smx1:smx2]
        if roi.shape[:2] == sm.shape:
            out[sy1:sy2,sx1:sx2] = np.clip(
                roi*(1-sm[...,np.newaxis]),0,255).astype(np.uint8)
    return out


def add_sheen(canvas: np.ndarray) -> np.ndarray:
    h, w   = canvas.shape[:2]
    overlay = canvas.copy()
    cx      = random.randint(w//6, 5*w//6)
    cy      = random.randint(h//6, 5*h//6)
    rx      = random.randint(w//8, w//3)
    ry      = random.randint(h//8, h//3)
    angle   = random.randint(0, 180)
    cv2.ellipse(overlay, (cx,cy), (rx,ry), angle, 0, 360, (255,255,255), -1)
    alpha_sheen = random.uniform(0.03, 0.09)
    return cv2.addWeighted(canvas, 1 - alpha_sheen, overlay, alpha_sheen, 0)


def augment_bg(bg):
    bg    = bg.copy()
    alpha = random.uniform(0.85, 1.15)
    beta  = random.uniform(-15, 15)
    bg    = np.clip(bg.astype(np.float32)*alpha+beta, 0, 255).astype(np.uint8)
    if random.random() < 0.40:
        noise = np.random.normal(0, random.uniform(2, 7), bg.shape).astype(np.float32)
        bg    = np.clip(bg.astype(np.float32)+noise, 0, 255).astype(np.uint8)
    if random.random() < 0.30:
        H, W = bg.shape[:2]
        Y, X = np.ogrid[:H, :W]
        cx, cy = W/2, H/2
        dist   = np.sqrt(((X-cx)/cx)**2 + ((Y-cy)/cy)**2)
        vign   = 1.0 - np.clip(dist*random.uniform(0.25, 0.55), 0, 0.35)
        bg     = np.clip(bg.astype(np.float32)*vign[...,np.newaxis], 0, 255).astype(np.uint8)
    if random.random() < 0.25:
        H, W   = bg.shape[:2]
        gx,gy  = random.randint(0,W), random.randint(0,H)
        gr     = random.randint(15, min(W,H)//4)
        m      = np.zeros((H,W), np.float32)
        cv2.circle(m, (gx,gy), gr, 1.0, -1)
        m      = cv2.GaussianBlur(m, (0,0), max(1, gr//2))
        bg     = np.clip(bg.astype(np.float32) +
                         m[...,np.newaxis]*255*random.uniform(0.04, 0.14),
                         0, 255).astype(np.uint8)
    return bg


def soft_iou(b1, b2):
    ix1 = max(b1[0], b2[0]); iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2]); iy2 = min(b1[3], b2[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0:
        return 0.0
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    union = a1 + a2 - inter
    return inter / max(union, 1)


def validate_label(cls_id, cx, cy, bw, bh, min_size=0.005, max_size=0.90) -> bool:
    if not (0.0 < cx < 1.0 and 0.0 < cy < 1.0):
        return False
    if not (min_size < bw < max_size and min_size < bh < max_size):
        return False
    x1 = cx - bw/2; y1 = cy - bh/2
    x2 = cx + bw/2; y2 = cy + bh/2
    if x1 < -0.05 or y1 < -0.05 or x2 > 1.05 or y2 > 1.05:
        return False
    return True


def paste_insect_ablation(background, insect_bgra, target_w, pos, use_poisson=True):
    """Identical to the original paste_insect(), with the Poisson seamlessClone
    step (Step 8) made conditional on use_poisson."""
    bg_h, bg_w     = background.shape[:2]
    orig_h, orig_w = insect_bgra.shape[:2]
    scale = target_w / max(orig_w, 1)
    new_w = max(8, int(orig_w * scale))
    new_h = max(8, int(orig_h * scale))
    if new_w >= bg_w or new_h >= bg_h:
        return background, None

    resized   = gpu_resize_bgra(insect_bgra, new_w, new_h)
    ins_bgr   = resized[:, :, :3].copy()
    ins_alpha = resized[:, :, 3].copy()

    ins_bgr = color_jitter_insect(ins_bgr)

    x, y = pos
    x = max(0, min(x, bg_w - new_w))
    y = max(0, min(y, bg_h - new_h))
    x2, y2 = x + new_w, y + new_h
    if background[y:y2, x:x2].shape[:2] != (new_h, new_w):
        return background, None

    ins_bgr = match_local_lighting(ins_bgr, background, x, y, new_w, new_h,
                                    strength=random.uniform(0.10, 0.25))
    ins_bgr = cv2.GaussianBlur(ins_bgr, ORGANIC_K, ORGANIC_S)
    if random.random() < 0.15:
        ins_bgr = apply_motion_blur(ins_bgr, max_kernel=5)

    _, ins_alpha = cv2.threshold(ins_alpha, 30, 255, cv2.THRESH_BINARY)
    ins_alpha = cv2.GaussianBlur(ins_alpha, (3, 3), 0.5)

    bg_work = add_shadow(background, x, y, new_w, new_h, ins_alpha)
    alpha_f = ins_alpha.astype(np.float32) / 255.0
    roi_f   = bg_work[y:y2, x:x2].astype(np.float32)
    ins_f   = ins_bgr.astype(np.float32)
    blended = ins_f * alpha_f[..., np.newaxis] + roi_f * (1 - alpha_f[..., np.newaxis])
    result  = bg_work.copy()
    result[y:y2, x:x2] = np.clip(blended, 0, 255).astype(np.uint8)

    if use_poisson and new_w > 25 and new_h > 25:
        try:
            _, pmask = cv2.threshold(ins_alpha, 50, 255, cv2.THRESH_BINARY)
            if pmask.sum() > 0:
                center = (x + new_w // 2, y + new_h // 2)
                try:
                    result = cv2.seamlessClone(
                        result[y:y2, x:x2], result, pmask, center, cv2.MIXED_CLONE)
                except cv2.error:
                    result = cv2.seamlessClone(
                        result[y:y2, x:x2], result, pmask, center, cv2.NORMAL_CLONE)
        except Exception:
            pass

    bbox = ((x + new_w / 2) / bg_w, (y + new_h / 2) / bg_h, new_w / bg_w, new_h / bg_h)
    return result, bbox


def generate_v6_variant(output_path, use_poisson=True, use_distractors=True,
                         use_pad_aware=True, num_images=None, seed=None):
    """Same generation loop as the original generate(), parameterized by the
    three ablation flags and writing to output_path instead of PATH_OUTPUT."""
    if num_images is None:
        num_images = NUM_IMAGES
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    output_path = Path(output_path)
    asp_paths = json.loads(asp_manifest.read_text())
    eu_paths  = json.loads(dist_manifest.read_text())

    aspilanta   = load_bgra_images(asp_paths, "Aspilanta")
    distractors = load_bgra_images(eu_paths, "EU Moths") if use_distractors else []
    if not aspilanta:
        raise RuntimeError("No Aspilanta cutouts found.")

    bg_exts = ('*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG')
    bg_paths_all = []
    for ext in bg_exts:
        bg_paths_all.extend(PATH_BACKGROUNDS.glob(ext))
        bg_paths_all.extend(PATH_BACKGROUNDS.glob(f'**/{ext}'))
    bg_paths_all = list(set(bg_paths_all))
    if not bg_paths_all:
        raise RuntimeError(f"No backgrounds in {PATH_BACKGROUNDS}")

    for split in ["train", "val"]:
        Path(output_path, "images", split).mkdir(parents=True, exist_ok=True)
        Path(output_path, "labels", split).mkdir(parents=True, exist_ok=True)

    stats = {"total": 0, "with_aspilanta": 0, "with_distractor": 0, "empty": 0,
             "n_aspilanta": 0, "n_distractor": 0, "labels_rejected": 0, "size_px_list": []}

    print(f"\nGenerating {num_images} images -> {output_path}")
    print(f"  use_poisson={use_poisson}  use_distractors={use_distractors}  use_pad_aware={use_pad_aware}\n")
    generated, attempts = 0, 0

    while generated < num_images:
        attempts += 1
        if attempts > num_images * 6:
            print("Warning: stopping early (too many attempts)")
            break

        split  = "train" if generated < int(num_images * TRAIN_RATIO) else "val"
        bg_raw = cv2.imread(str(random.choice(bg_paths_all)))
        if bg_raw is None:
            continue
        bg_raw = augment_bg(bg_raw)

        if use_pad_aware:
            canvas, (_, _, cw, ch), _ = extract_clean_pad(bg_raw)
        else:
            canvas = bg_raw.copy()
            cw, ch = bg_raw.shape[1], bg_raw.shape[0]
        if cw < 50 or ch < 50:
            continue
        if random.random() < SHEEN_PROB:
            canvas = add_sheen(canvas)

        r = random.random()
        n_asp = (0 if r < 0.10 else
                 random.randint(1, 3) if r < 0.45 else
                 random.randint(4, 7) if r < 0.80 else
                 random.randint(8, MAX_ASPILANTA))
        n_dist = (0 if not distractors else
                  random.choices([0, 1, 2, 3, MAX_DISTRACTOR],
                                  weights=[0.50, 0.22, 0.14, 0.09, 0.05])[0])

        placed, labels = [], []

        for _ in range(n_asp):
            src = random.choice(aspilanta)
            for _ in range(MAX_ATTEMPTS):
                tw = sample_size_px(cw)
                tw = max(4, min(tw, cw // 3))
                ang = random.uniform(0, 360)
                M = cv2.getRotationMatrix2D((src.shape[1] // 2, src.shape[0] // 2), ang, 1.0)
                rot = cv2.warpAffine(src, M, (src.shape[1], src.shape[0]),
                                      flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,
                                      borderValue=(0, 0, 0, 0))
                margin = max(5, tw)
                if cw <= 2 * margin or ch <= 2 * margin:
                    continue
                px = random.randint(margin, cw - margin)
                py = random.randint(margin, ch - margin)
                th = int(tw * rot.shape[0] / max(rot.shape[1], 1))
                box = [px, py, px + tw, py + th]
                if any(soft_iou(box, pb) > 0.30 for pb in placed):
                    continue
                canvas, bbox = paste_insect_ablation(canvas, rot, tw, (px, py), use_poisson=use_poisson)
                if bbox is None:
                    continue
                if not validate_label(CLS_ASPILANTA, *bbox):
                    stats["labels_rejected"] += 1
                    continue
                placed.append(box)
                labels.append(f"{CLS_ASPILANTA} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}")
                stats["n_aspilanta"] += 1
                stats["size_px_list"].append(tw)
                break

        for _ in range(n_dist):
            src = random.choice(distractors)
            dw = int(sample_size_px(cw) * random.uniform(0.8, 2.5))
            dw = max(10, min(dw, cw // 3))
            for _ in range(MAX_ATTEMPTS // 2):
                margin = max(5, dw)
                if cw <= 2 * margin or ch <= 2 * margin:
                    break
                px = random.randint(margin, cw - margin)
                py = random.randint(margin, ch - margin)
                box = [px, py, px + dw, py + dw]
                if any(soft_iou(box, pb) > 0.30 for pb in placed):
                    continue
                ang = random.uniform(0, 360)
                M = cv2.getRotationMatrix2D((src.shape[1] // 2, src.shape[0] // 2), ang, 1.0)
                rot = cv2.warpAffine(src, M, (src.shape[1], src.shape[0]),
                                      flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,
                                      borderValue=(0, 0, 0, 0))
                canvas, bbox = paste_insect_ablation(canvas, rot, dw, (px, py), use_poisson=use_poisson)
                if bbox is None:
                    continue
                if not validate_label(CLS_DISTRACTOR, *bbox):
                    stats["labels_rejected"] += 1
                    continue
                placed.append(box)
                labels.append(f"{CLS_DISTRACTOR} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}")
                stats["n_distractor"] += 1
                break

        has_insects = bool(labels)
        allow_empty = (generated % 10 == 0)
        if not has_insects and not allow_empty:
            continue

        fname = f"synth_{generated:06d}"
        cv2.imwrite(str(output_path / "images" / split / f"{fname}.jpg"),
                    canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        (output_path / "labels" / split / f"{fname}.txt").write_text('\n'.join(labels))

        stats["total"] += 1
        stats["with_aspilanta"]  += int(any(l.startswith(str(CLS_ASPILANTA)) for l in labels))
        stats["with_distractor"] += int(any(l.startswith(str(CLS_DISTRACTOR)) for l in labels))
        stats["empty"] += int(not labels)
        generated += 1
        if generated % 400 == 0:
            print(f"  [{generated}/{num_images}]")

    yaml_txt = f"""path: {output_path.absolute()}
train: images/train
val:   images/val
nc: 2
names:
  0: {NAME_ASPILANTA}
  1: {NAME_DISTRACTOR}
"""
    (output_path / 'dataset.yaml').write_text(yaml_txt)
    s = {k: v for k, v in stats.items() if k != "size_px_list"}
    (output_path / 'stats.json').write_text(json.dumps(s, indent=2))
    print(f"Done: {output_path}  total={stats['total']}  aspilanta={stats['n_aspilanta']}  distractor={stats['n_distractor']}")
    return stats

print("Self-contained generate_v6_variant() and paste_insect_ablation() defined.")

In [ ]:
# ============================================================
# CELL 92 — Generate the 3 compositing ablation datasets
# ============================================================
#
# Ablations:
#   1. No Poisson blending
#   2. No distractor insects
#   3. No pad-aware placement
#
# Real-only does NOT require synthetic data generation.
#
# IMPORTANT:
# Cell 91 (generate_v6_variant definition) must be run first.
# ============================================================

from pathlib import Path
import glob
import os


# ------------------------------------------------------------
# 1. Basic configuration
# ------------------------------------------------------------

# Your main synthetic dataset contains 2000 images.
# We use the SAME number for every ablation so comparisons
# are controlled.
NUM_IMAGES = 100

# Keep exactly the same seed for all three variants.
# This is important so that the random sampling is as similar
# as possible across ablations.
RANDOM_SEED = 42

# All newly generated datasets go to the Kaggle working area.
BASE = Path("/kaggle/working")


# ------------------------------------------------------------
# 2. Locate the existing main synthetic dataset
# ------------------------------------------------------------

# Your Kaggle Input screenshot shows:
#
#   synthetic gen data
#       └── generated_dataset_v6
#
# We do NOT modify this dataset.
#
# Search Kaggle Input automatically instead of hard-coding
# the dataset slug.

candidate_paths = list(
    Path("/kaggle/input").glob("*/generated_dataset_v6")
)

if candidate_paths:
    PATH_SYNTH_MAIN_INPUT = candidate_paths[0]
    print("Found main synthetic dataset:")
    print(PATH_SYNTH_MAIN_INPUT)
else:
    PATH_SYNTH_MAIN_INPUT = None
    print("WARNING: Could not automatically locate generated_dataset_v6.")


# ------------------------------------------------------------
# 3. Output paths for the three ablations
# ------------------------------------------------------------

PATH_SYNTH_NO_POISSON = (
    BASE / "generated_dataset_v6_no_poisson"
)

PATH_SYNTH_NO_DISTRACTORS = (
    BASE / "generated_dataset_v6_no_distractors"
)

PATH_SYNTH_NO_PAD_AWARE = (
    BASE / "generated_dataset_v6_no_pad_aware"
)


# ------------------------------------------------------------
# 4. Check that Cell 91 was executed
# ------------------------------------------------------------

if "generate_v6_variant" not in globals():
    raise RuntimeError(
        "\nERROR: generate_v6_variant() is not defined.\n"
        "Run CELL 91 first:\n"
        "    8.1 generator patch — generate_v6_variant() definition\n"
    )


# ------------------------------------------------------------
# 5. Print configuration
# ------------------------------------------------------------

print("=" * 70)
print("COMPOSITING ABLATION DATA GENERATION")
print("=" * 70)

print(f"Number of images per variant : {NUM_IMAGES}")
print(f"Random seed                  : {RANDOM_SEED}")
print(f"Output directory             : {BASE}")

print("\nAblations:")
print("  A. No Poisson blending")
print("  B. No distractors")
print("  C. No pad-aware placement")


# ------------------------------------------------------------
# 6. Generate: NO POISSON
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("[1/3] GENERATING NO-POISSON DATASET")
print("=" * 70)

stats_no_poisson = generate_v6_variant(
    PATH_SYNTH_NO_POISSON,
    use_poisson=False,
    use_distractors=True,
    use_pad_aware=True,
    num_images=NUM_IMAGES,
    seed=RANDOM_SEED,
)

print("\n✓ No-Poisson generation complete.")
print(stats_no_poisson)


# ------------------------------------------------------------
# 7. Generate: NO DISTRACTORS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("[2/3] GENERATING NO-DISTRACTORS DATASET")
print("=" * 70)

stats_no_distractors = generate_v6_variant(
    PATH_SYNTH_NO_DISTRACTORS,
    use_poisson=True,
    use_distractors=False,
    use_pad_aware=True,
    num_images=NUM_IMAGES,
    seed=RANDOM_SEED,
)

print("\n✓ No-distractors generation complete.")
print(stats_no_distractors)


# ------------------------------------------------------------
# 8. Generate: NO PAD-AWARE PLACEMENT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("[3/3] GENERATING NO-PAD-AWARE DATASET")
print("=" * 70)

stats_no_pad_aware = generate_v6_variant(
    PATH_SYNTH_NO_PAD_AWARE,
    use_poisson=True,
    use_distractors=True,
    use_pad_aware=False,
    num_images=NUM_IMAGES,
    seed=RANDOM_SEED,
)

print("\n✓ No-pad-aware generation complete.")
print(stats_no_pad_aware)


# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALL THREE ABLATION DATASETS GENERATED")
print("=" * 70)

print("\nNo Poisson:")
print(PATH_SYNTH_NO_POISSON)

print("\nNo distractors:")
print(PATH_SYNTH_NO_DISTRACTORS)

print("\nNo pad-aware:")
print(PATH_SYNTH_NO_PAD_AWARE)

print("\nImages per dataset:", NUM_IMAGES)
print("Seed:", RANDOM_SEED)

In [ ]:
# Regenerate synthetic data for the three compositing ablations that need new
# data (Poisson off, distractors off, pad-aware off). "Real-only" (item 1)
# needs no synthetic data at all, so it is handled directly in 8.3.

PATH_SYNTH_NO_POISSON     = BASE / "generated_dataset_v6_no_poisson"
PATH_SYNTH_NO_DISTRACTORS = BASE / "generated_dataset_v6_no_distractors"
PATH_SYNTH_NO_PAD_AWARE   = BASE / "generated_dataset_v6_no_pad_aware"

stats_no_poisson = generate_v6_variant(
    PATH_SYNTH_NO_POISSON, use_poisson=False, use_distractors=True, use_pad_aware=True,
    num_images=NUM_IMAGES, seed=RANDOM_SEED)

stats_no_distractors = generate_v6_variant(
    PATH_SYNTH_NO_DISTRACTORS, use_poisson=True, use_distractors=False, use_pad_aware=True,
    num_images=NUM_IMAGES, seed=RANDOM_SEED)

stats_no_pad_aware = generate_v6_variant(
    PATH_SYNTH_NO_PAD_AWARE, use_poisson=True, use_distractors=True, use_pad_aware=False,
    num_images=NUM_IMAGES, seed=RANDOM_SEED)

### 8.2 — Generic dataset builder and trainers, reused by every ablation

In [ ]:
# ==============================================================================
#  Generic 4-class dataset builder, parallel to "REPLACEMENT CELL 6" (Experiment 3
#  dataset build), parameterized so it can build a real-only variant
#  (synth_train_imgs_var=None) or use an alternate synthetic folder.
# ==============================================================================

def build_ablation_exp3_dataset(name, synth_train_imgs_var=None, synth_val_imgs_var=None,
                                 synth_root=None):
    out_root = BASE / f"ablation_{name}_dataset"
    safe_clear_dir(out_root)
    ensure_yolo_dirs(out_root)

    copy_folder_yolo(orig_train_img, orig_train_lbl,
                      out_root / "images/train", out_root / "labels/train", prefix="orig_train_")
    if orig_val_img and orig_val_lbl:
        copy_folder_yolo(orig_val_img, orig_val_lbl,
                          out_root / "images/val", out_root / "labels/val", prefix="orig_val_")

    for img in real_train_imgs:
        copy_one_image_and_label(img, out_root / "images/train", out_root / "labels/train",
                                  prefix="real_train_", force_class_id=AP_CLASS_ID,
                                  label_roots=[PATH_REAL_AP])
    for img in real_val_imgs:
        copy_one_image_and_label(img, out_root / "images/val", out_root / "labels/val",
                                  prefix="real_val_", force_class_id=AP_CLASS_ID,
                                  label_roots=[PATH_REAL_AP])

    if synth_train_imgs_var is not None:
        for img in synth_train_imgs_var:
            copy_one_image_and_label(img, out_root / "images/train", out_root / "labels/train",
                                      prefix="synth_train_", class_id_map={0: AP_CLASS_ID, 1: 0},
                                      label_roots=[synth_root])
        for img in synth_val_imgs_var:
            copy_one_image_and_label(img, out_root / "images/val", out_root / "labels/val",
                                      prefix="synth_val_", class_id_map={0: AP_CLASS_ID, 1: 0},
                                      label_roots=[synth_root])

    yaml_path = write_yaml(out_root, BASE / f"ablation_{name}.yaml", CLASSES_4)
    print_dataset_counts(f"Ablation [{name}]", out_root)
    return yaml_path


def train_ablation_variant(run_name, data_yaml, init_weights, freeze, copy_paste=None):
    """Experiment 3 recipe (Cell 58), parameterized by init weights, freeze depth,
    dataset yaml, and copy_paste augmentation strength -- covers items 1,2,3,4,7,8,9."""
    cp = EXP3_COPY_PASTE if copy_paste is None else copy_paste
    print("\n" + "=" * 70)
    print(f"ABLATION RUN [fine-tuning recipe]: {run_name}")
    print(f"  data={data_yaml}  init={init_weights}  freeze={freeze}  copy_paste={cp}")
    print("=" * 70)

    model = YOLO(str(init_weights))
    model.train(
        data=str(data_yaml), epochs=EXP3_EPOCHS, imgsz=EXP3_IMGSZ, batch=EXP3_BATCH,
        device=DEVICE, workers=EXP3_WORKERS, patience=EXP3_PATIENCE, freeze=freeze,
        lr0=EXP3_LR0, lrf=EXP3_LRF, warmup_epochs=EXP3_WARMUP_EPOCHS,
        mosaic=EXP3_MOSAIC, copy_paste=cp, degrees=EXP3_DEGREES,
        fliplr=EXP3_FLIPLR, flipud=EXP3_FLIPUD, scale=EXP3_SCALE, translate=EXP3_TRANSLATE,
        hsv_h=EXP3_HSV_H, hsv_s=EXP3_HSV_S, hsv_v=EXP3_HSV_V,
        cos_lr=True, close_mosaic=15, label_smoothing=0.05,
        project=str(BASE / "runs/detect"), name=run_name, exist_ok=True, save=True, plots=True,
    )
    best = BASE / "runs/detect" / run_name / "weights/best.pt"
    print(f"Finished: {run_name} -> {best}")
    return best


def train_exp1_variant(run_name, init_weights):
    """Experiment 1 recipe (Cell 56), parameterized only by init weights --
    covers items 5 (scratch vs. pretrained) and 6 (YOLOv8n vs. YOLOv8s)."""
    print("\n" + "=" * 70)
    print(f"ABLATION RUN [transfer-learning recipe]: {run_name}")
    print(f"  data={YAML_EXP1}  init={init_weights}")
    print("=" * 70)

    model = YOLO(str(init_weights))
    model.train(
        data=str(YAML_EXP1), epochs=EXP1_EPOCHS, imgsz=EXP1_IMGSZ, batch=EXP1_BATCH,
        device=DEVICE, workers=EXP1_WORKERS, patience=EXP1_PATIENCE, freeze=EXP1_FREEZE,
        lr0=EXP1_LR0, lrf=EXP1_LRF, warmup_epochs=EXP1_WARMUP_EPOCHS,
        mosaic=1.0, degrees=10.0, fliplr=0.5, flipud=0.0,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, scale=0.5, translate=0.1,
        project=str(BASE / "runs/detect"), name=run_name, exist_ok=True, save=True, plots=True,
    )
    best = BASE / "runs/detect" / run_name / "weights/best.pt"
    print(f"Finished: {run_name} -> {best}")
    return best

print("build_ablation_exp3_dataset(), train_ablation_variant(), train_exp1_variant() defined.")

### 8.3 — Run all nine ablations

In [ ]:
ablation_models = {}   # fine-tuning-recipe (Exp3-style) ablations -> evaluated on YAML_TARGET_VAL
exp1_ablation_models = {}   # transfer-learning-recipe (Exp1-style) ablations -> evaluated on YAML_EXP1_VAL

# ---------- 1. Copy-paste compositing ----------

# (1) Synthetic vs. real-only
yaml_real_only = build_ablation_exp3_dataset("real_only")
ablation_models["real_only"] = train_ablation_variant(
    "ablation_real_only", yaml_real_only, EXP3_INIT_WEIGHTS, EXP3_FREEZE)

# (2) Poisson blending off
yaml_no_poisson = build_ablation_exp3_dataset(
    "no_poisson",
    synth_train_imgs_var=list_images(PATH_SYNTH_NO_POISSON / "images/train"),
    synth_val_imgs_var=list_images(PATH_SYNTH_NO_POISSON / "images/val"),
    synth_root=PATH_SYNTH_NO_POISSON)
ablation_models["no_poisson"] = train_ablation_variant(
    "ablation_no_poisson", yaml_no_poisson, EXP3_INIT_WEIGHTS, EXP3_FREEZE)

# (3) Distractors off
yaml_no_distractors = build_ablation_exp3_dataset(
    "no_distractors",
    synth_train_imgs_var=list_images(PATH_SYNTH_NO_DISTRACTORS / "images/train"),
    synth_val_imgs_var=list_images(PATH_SYNTH_NO_DISTRACTORS / "images/val"),
    synth_root=PATH_SYNTH_NO_DISTRACTORS)
ablation_models["no_distractors"] = train_ablation_variant(
    "ablation_no_distractors", yaml_no_distractors, EXP3_INIT_WEIGHTS, EXP3_FREEZE)

# (4) Pad-aware placement off
yaml_no_pad_aware = build_ablation_exp3_dataset(
    "no_pad_aware",
    synth_train_imgs_var=list_images(PATH_SYNTH_NO_PAD_AWARE / "images/train"),
    synth_val_imgs_var=list_images(PATH_SYNTH_NO_PAD_AWARE / "images/val"),
    synth_root=PATH_SYNTH_NO_PAD_AWARE)
ablation_models["no_pad_aware"] = train_ablation_variant(
    "ablation_no_pad_aware", yaml_no_pad_aware, EXP3_INIT_WEIGHTS, EXP3_FREEZE)

# ---------- 2. YOLO transfer learning ----------

# (5) Scratch init (random weights, same architecture) vs. ImageNet-pretrained (= Exp1 itself)
exp1_ablation_models["scratch_init"] = train_exp1_variant(
    "ablation_exp1_scratch_init", "yolov8s.yaml")   # .yaml = architecture only, no pretrained weights

# (6) YOLOv8n vs. YOLOv8s (Exp1 itself is the YOLOv8s reference point)
exp1_ablation_models["yolov8n_arch"] = train_exp1_variant(
    "ablation_exp1_yolov8n", "yolov8n.pt")

# ---------- 3. Fine-tuning ----------

# (7) Unfrozen backbone (freeze=0 instead of EXP3_FREEZE)
ablation_models["unfrozen_backbone"] = train_ablation_variant(
    "ablation_unfrozen_backbone", YAML_EXP3, EXP3_INIT_WEIGHTS, 0)

# (8) Fresh ImageNet init instead of fine-tuning from Exp1
ablation_models["fresh_imagenet_init"] = train_ablation_variant(
    "ablation_fresh_imagenet_init", YAML_EXP3, "yolov8s.pt", EXP3_FREEZE)

# (9) copy_paste augmentation off
ablation_models["no_copy_paste_aug"] = train_ablation_variant(
    "ablation_no_copy_paste_aug", YAML_EXP3, EXP3_INIT_WEIGHTS, EXP3_FREEZE, copy_paste=0.0)

print("\nFine-tuning-recipe ablations (evaluate on YAML_TARGET_VAL):")
for k, v in ablation_models.items():
    print(f"  {k}: {v}  exists={v.exists()}")
print("\nTransfer-learning-recipe ablations (evaluate on YAML_EXP1_VAL):")
for k, v in exp1_ablation_models.items():
    print(f"  {k}: {v}  exists={v.exists()}")

In [ ]:
# Restore all 9 ablation runs from the uploaded dataset, and rebuild the
# ablation_models / exp1_ablation_models dicts that Cell 99 expects.
import shutil, zipfile
from pathlib import Path

SRC = Path("/kaggle/input/datasets/your-kaggle-username/section8-ablations-all9")
_runs = BASE / "runs" / "detect"
_runs.mkdir(parents=True, exist_ok=True)

# If the dataset was uploaded as a zip, unzip it; if uploaded unzipped, copy.
zips = list(SRC.glob("*.zip"))
if zips:
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(_runs)
    print("Extracted", zips[0].name)
else:
    for d in SRC.iterdir():
        if d.is_dir() and d.name.startswith("ablation_"):
            shutil.copytree(d, _runs / d.name, dirs_exist_ok=True)
    print("Copied run folders")

FT_RUNS   = ["real_only","no_poisson","no_distractors","no_pad_aware",
             "unfrozen_backbone","fresh_imagenet_init","no_copy_paste_aug"]
EXP1_RUNS = ["exp1_scratch_init","exp1_yolov8n"]

def _w(name):
    p = _runs / f"ablation_{name}" / "weights" / "best.pt"
    if not p.exists():                      # older flat layout
        p = _runs / f"ablation_{name}" / "best.pt"
    return p

ablation_models      = {k: _w(k) for k in FT_RUNS}
exp1_ablation_models = {k: _w(k) for k in EXP1_RUNS}

for d, label in [(ablation_models,"fine-tuning"), (exp1_ablation_models,"transfer-learning")]:
    print(f"\n{label}:")
    for k, v in d.items():
        print(f"  {k:22s} exists={v.exists()}  {v}")

### 8.4 — Evaluate every ablation and build the master comparison table

In [ ]:
# Fine-tuning-recipe ablations: evaluate on the 4-class target-domain val set,
# same as the main Experiment 3 evaluation. Reuses get_class_ap()/safe_float()
# from the main evaluation cell (Cell 60).

rows = []

rows.append({
    "group": "reference", "variant": "Exp3 (full method)",
    "mAP50": safe_float(m_exp3.box.map50), "mAP50_95": safe_float(m_exp3.box.map),
    "AP_Ashpilanta_50": get_class_ap(m_exp3, AP_CLASS_ID, "map50"),
    "AP_Ashpilanta_50_95": get_class_ap(m_exp3, AP_CLASS_ID, "map"),
})

for key, weights_path in ablation_models.items():
    m = YOLO(str(weights_path)).val(
        data=str(YAML_TARGET_VAL), split="val", verbose=False, plots=False,
        project=str(BASE / "runs/detect"), name=f"val_{key}", exist_ok=True,
    )
    group = ("copy-paste compositing" if key in
             {"real_only", "no_poisson", "no_distractors", "no_pad_aware"}
             else "fine-tuning")
    rows.append({
        "group": group, "variant": key,
        "mAP50": safe_float(m.box.map50), "mAP50_95": safe_float(m.box.map),
        "AP_Ashpilanta_50": get_class_ap(m, AP_CLASS_ID, "map50"),
        "AP_Ashpilanta_50_95": get_class_ap(m, AP_CLASS_ID, "map"),
    })

# Transfer-learning-recipe ablations: evaluate on the 3-class Exp1 val set,
# same as m_exp1 in Cell 60.
rows.append({
    "group": "reference", "variant": "Exp1 (ImageNet-pretrained YOLOv8s)",
    "mAP50": safe_float(m_exp1.box.map50), "mAP50_95": safe_float(m_exp1.box.map),
    "AP_Ashpilanta_50": None, "AP_Ashpilanta_50_95": None,
})

for key, weights_path in exp1_ablation_models.items():
    m = YOLO(str(weights_path)).val(
        data=str(YAML_EXP1_VAL), split="val", verbose=False, plots=False,
        project=str(BASE / "runs/detect"), name=f"val_{key}", exist_ok=True,
    )
    rows.append({
        "group": "YOLO transfer learning", "variant": key,
        "mAP50": safe_float(m.box.map50), "mAP50_95": safe_float(m.box.map),
        "AP_Ashpilanta_50": None, "AP_Ashpilanta_50_95": None,
    })

master_df = pd.DataFrame(rows)
master_csv = PATH_RESULTS / "ablation_master_results.csv"
master_df.to_csv(master_csv, index=False)

print(master_df.to_string(index=False))
print("\nSaved:", master_csv)
print("\nMarkdown (paste-ready for the thesis):\n")
print(master_df.to_markdown(index=False))

print("\n" + "=" * 70)
print("Metric-learning (CLIP) ablations are already covered separately by:")
print("  - Cell 13/61 -> thesis_results/clip_kshot_ablation.csv (+ .png)")
print("  - Cell 28b/87 -> thesis_results/three_way_pipeline_comparison_synthetic.csv")
print("Combine those two CSVs with this one for the full ablation chapter.")
print("=" * 70)

In [ ]:
# ==============================================================================
#  ARCHIVE SECTION 8 — bundle all ablation results + weights into one zip.
#  Safe to re-run; overwrites the same-named zip. Skips anything not present.
# ==============================================================================
import shutil, zipfile, json, time
from pathlib import Path

def _add_if_exists(zf, path, arcname):
    path = Path(path)
    if path.is_file():
        zf.write(path, arcname); return 1
    elif path.is_dir():
        n = 0
        for f in path.rglob("*"):
            if f.is_file():
                zf.write(f, str(Path(arcname) / f.relative_to(path))); n += 1
        return n
    return 0

_stamp = time.strftime("%Y%m%d_%H%M%S")
ARCHIVE_S8 = BASE / f"archive_section8_ablations_{_stamp}.zip"
_runs = BASE / "runs" / "detect"

_ablation_run_names = [
    "ablation_real_only", "ablation_no_poisson", "ablation_no_distractors",
    "ablation_no_pad_aware", "ablation_unfrozen_backbone", "ablation_fresh_imagenet_init",
    "ablation_no_copy_paste_aug", "ablation_exp1_scratch_init", "ablation_exp1_yolov8n",
]

_count = 0
with zipfile.ZipFile(ARCHIVE_S8, "w", zipfile.ZIP_DEFLATED) as zf:
    # master results table
    _count += _add_if_exists(zf, PATH_RESULTS / "ablation_master_results.csv",
                             "ablation_master_results.csv")
    # per-run weights + training plots/metrics
    for rn in _ablation_run_names:
        _count += _add_if_exists(zf, _runs / rn / "weights" / "best.pt",
                                 f"{rn}/best.pt")
        for extra in ["results.csv", "results.png", "args.yaml",
                      "confusion_matrix.png", "PR_curve.png"]:
            _count += _add_if_exists(zf, _runs / rn / extra, f"{rn}/{extra}")
        # val-run folders (val_<name>)
        _count += _add_if_exists(zf, _runs / f"val_{rn}", f"val_{rn}")

    # also stash the in-memory table as json, in case the CSV path differs
    try:
        zf.writestr("master_df.json", master_df.to_json(orient="records", indent=2))
        _count += 1
    except Exception as e:
        print("  (master_df not in memory:", e, ")")

print(f"Section 8 archive written: {ARCHIVE_S8}")
print(f"  files added: {_count}   size: {ARCHIVE_S8.stat().st_size/1e6:.1f} MB")
print("Download it from the Kaggle output panel, or 'Save Version' to persist it.")

**How each row maps back to the methodology text:**

| Methodology claim | Ablation row(s) | What a real drop would confirm |
|---|---|---|
| "Synthetic compositing provides useful training signal" | `real_only` vs. `Exp3 (full method)` | Synthetic data materially helps Ashpilanta AP |
| "Poisson blending improves realism enough to matter" | `no_poisson` vs. reference | Blending, not just placement, matters |
| "Distractors teach the model to reject visually similar non-targets" | `no_distractors` vs. reference | Distractors reduce false positives / improve AP |
| "Pad-aware placement calibrates scale correctly" | `no_pad_aware` vs. reference | Scale calibration matters, not just random placement |
| "Transfer learning from ImageNet is worthwhile" | `scratch_init` vs. `Exp1` | Pretrained features beat random init given limited data |
| "YOLOv8s capacity is justified over YOLOv8n" | `yolov8n_arch` vs. `Exp1` | The larger model earns its extra cost |
| "Freezing the backbone during fine-tuning is the right call" | `unfrozen_backbone` vs. reference | Freezing preserves useful features / avoids overfitting |
| "Fine-tuning from Exp1 (not fresh ImageNet) is the right transfer chain" | `fresh_imagenet_init` vs. reference | The intermediate 3-class stage adds value |
| "copy_paste augmentation specifically helps the rare class" | `no_copy_paste_aug` vs. reference | The augmentation choice is doing real work |
| "The CLIP few-shot filter needs only a few support examples" | `clip_kshot_ablation.csv` | Performance plateaus at low K |
| "The CLIP filter improves precision without destroying recall" | `three_way_pipeline_comparison_synthetic.csv` | `baseline_clip` beats `baseline` on F1 |

If a row shows *no* meaningful drop, that's still a valid, reportable result — say so plainly in
the thesis (you already did this for tiling/CLIP in Chapter 7); it means that component isn't
earning its complexity and is worth flagging as a limitation or a candidate for removal.

## Section 9 — Foundation-model baseline: OWL-ViT zero-/few-shot detection

Fills the gap identified in `few_shot_techniques_menu.md`: your pipeline has never been
compared against a foundation-model detector doing the job with **no training at all**.
This section runs OWL-ViT (open-vocabulary detection, pretrained, frozen) in two modes:

1. **Zero-shot, text-prompted** — no Ashpilanta examples at all, just text prompts
   (`"insect"`, `"small insect"`, `"moth"`).
2. **Few-shot, image-guided** — K real Ashpilanta crops as visual exemplars
   (K=1 and K=5, matching your existing CLIP K-shot reporting for a like-for-like comparison).

Both are scored on the *same* `target_domain_val` set, with the *same* IoU-matching logic
used everywhere else in this notebook (`load_ashpilanta_gt_as_xyxy` / `box_iou_xyxy` /
`match_ashpilanta_detections` from Cell 22 — reimplemented here in a self-contained way
so this section doesn't require Cells 69-78 to have run first).

This is inference-only — no training, no GPU-hours budget concern — but OWL-ViT inference
over the full validation set can still take a while on a T4/P100; `max_images` below lets
you cap it for a quick check before running the full set.

### 9.1 — Setup: model, support crops, self-contained evaluation harness

In [ ]:
# ==============================================================================
#  SECTION 9 SETUP — self-contained: only depends on Section 3 having run
#  (BASE, PATH_TARGET_VAL, AP_CLASS_ID, PATH_RESULTS, RANDOM_SEED). Does NOT
#  require the deployment pipeline (Cells 69-78) or Section 8.
# ==============================================================================

import torch, cv2, numpy as np, random, json, time
from pathlib import Path
from PIL import Image

owlvit_device = "cuda" if torch.cuda.is_available() else "cpu"

from transformers import OwlViTProcessor, OwlViTForObjectDetection

OWLVIT_MODEL_NAME = "google/owlvit-base-patch32"
owlvit_processor  = OwlViTProcessor.from_pretrained(OWLVIT_MODEL_NAME)
owlvit_model      = OwlViTForObjectDetection.from_pretrained(OWLVIT_MODEL_NAME).to(owlvit_device)
owlvit_model.eval()
print(f"Loaded {OWLVIT_MODEL_NAME} on {owlvit_device}")

# ── Ground-truth loading + IoU matching (self-contained copies of the Cell 22
#    helpers, so this section works even if Cells 69-78 were never run) ──────

def load_ashpilanta_gt_as_xyxy_s9(label_path, img_w, img_h, ap_class_id):
    label_path = Path(label_path)
    if not label_path.exists():
        return np.empty((0, 4))
    boxes = []
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        if not parts:
            continue
        if int(float(parts[0])) != ap_class_id:
            continue
        xc, yc, w, h = map(float, parts[1:5])
        boxes.append([(xc-w/2)*img_w, (yc-h/2)*img_h, (xc+w/2)*img_w, (yc+h/2)*img_h])
    return np.array(boxes) if boxes else np.empty((0, 4))

def box_iou_xyxy_s9(box, boxes):
    if len(boxes) == 0:
        return np.empty((0,))
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    area_box = max(0, box[2]-box[0]) * max(0, box[3]-box[1])
    area_boxes = np.maximum(0, boxes[:,2]-boxes[:,0]) * np.maximum(0, boxes[:,3]-boxes[:,1])
    union = area_box + area_boxes - inter
    return np.where(union > 0, inter/union, 0)

def match_ashpilanta_detections_s9(pred_boxes, gt_boxes, iou_thresh=0.5):
    gt_matched = np.zeros(len(gt_boxes), dtype=bool)
    tp, fp = 0, 0
    for pb in pred_boxes:
        unmatched_idx = np.where(~gt_matched)[0]
        if len(unmatched_idx) == 0:
            fp += 1
            continue
        ious = box_iou_xyxy_s9(pb, gt_boxes[unmatched_idx])
        best = np.argmax(ious) if len(ious) else -1
        if best >= 0 and ious[best] >= iou_thresh:
            gt_matched[unmatched_idx[best]] = True
            tp += 1
        else:
            fp += 1
    fn = int((~gt_matched).sum())
    return tp, fp, fn


# ── K real Ashpilanta exemplar crops, drawn from the REAL (non-synthetic)
#    training images already merged into target_domain_val's training split.
#    Filenames prefixed "real_train_" identify genuine photographed
#    specimens (see Cell "REPLACEMENT CELL 6 -- BUILD EXPERIMENT 3 DATASET"),
#    so no synthetic composites and no validation-set leakage.
def get_real_ashpilanta_exemplars(k, seed=RANDOM_SEED):
    img_dir   = PATH_TARGET_VAL / "images" / "train"
    label_dir = PATH_TARGET_VAL / "labels" / "train"
    candidates = sorted(p for p in img_dir.glob("real_train_*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    random.Random(seed).shuffle(candidates)

    crops = []
    for p in candidates:
        if len(crops) >= k:
            break
        label_path = label_dir / f"{p.stem}.txt"
        img = cv2.imread(str(p))
        if img is None:
            continue
        h, w = img.shape[:2]
        gt = load_ashpilanta_gt_as_xyxy_s9(label_path, w, h, AP_CLASS_ID)
        if len(gt) == 0:
            continue
        x1, y1, x2, y2 = gt[0].astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop_bgr = img[y1:y2, x1:x2]
        crops.append(Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)))
    print(f"Collected {len(crops)}/{k} real Ashpilanta exemplar crops")
    return crops

EXEMPLARS_K5 = get_real_ashpilanta_exemplars(5)
EXEMPLARS_K1 = EXEMPLARS_K5[:1]
print("Exemplar counts -> K1:", len(EXEMPLARS_K1), " K5:", len(EXEMPLARS_K5))

### 9.2 — OWL-ViT inference functions (text zero-shot, image-guided few-shot)

In [ ]:
def owlvit_text_detect(pil_image, text_queries, score_thresh=0.05):
    """Zero-shot: text-prompted detection, no Ashpilanta examples at all."""
    inputs = owlvit_processor(text=[text_queries], images=pil_image, return_tensors="pt").to(owlvit_device)
    with torch.no_grad():
        outputs = owlvit_model(**inputs)
    target_sizes = torch.tensor([pil_image.size[::-1]]).to(owlvit_device)
    results = owlvit_processor.post_process_object_detection(
        outputs=outputs, threshold=score_thresh, target_sizes=target_sizes)[0]
    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    return boxes, scores


def owlvit_image_guided_detect(pil_image, exemplar_crops, score_thresh=0.5, nms_thresh=0.3):
    """Few-shot: image-guided detection using one or more exemplar crops.
    Runs one query per exemplar and merges + NMS's the results, so K
    exemplars behave like a K-shot support set (mirrors the CLIP K-shot setup)."""
    all_boxes, all_scores = [], []
    for exemplar in exemplar_crops:
        inputs = owlvit_processor(query_images=exemplar, images=pil_image, return_tensors="pt").to(owlvit_device)
        with torch.no_grad():
            outputs = owlvit_model.image_guided_detection(**inputs)
        target_sizes = torch.tensor([pil_image.size[::-1]]).to(owlvit_device)
        results = owlvit_processor.post_process_image_guided_detection(
            outputs=outputs, threshold=score_thresh, nms_threshold=nms_thresh,
            target_sizes=target_sizes)[0]
        all_boxes.append(results["boxes"].cpu().numpy())
        all_scores.append(results["scores"].cpu().numpy())

    if not all_boxes or all(len(b) == 0 for b in all_boxes):
        return np.empty((0, 4)), np.empty((0,))
    boxes  = np.concatenate(all_boxes, axis=0)
    scores = np.concatenate(all_scores, axis=0)

    # Merge duplicate detections across exemplars with a simple greedy NMS
    order = np.argsort(-scores)
    boxes, scores = boxes[order], scores[order]
    keep = []
    suppressed = np.zeros(len(boxes), dtype=bool)
    for i in range(len(boxes)):
        if suppressed[i]:
            continue
        keep.append(i)
        ious = box_iou_xyxy_s9(boxes[i], boxes[i+1:])
        suppressed[i+1:][ious >= nms_thresh] = True
    return boxes[keep], scores[keep]

print("owlvit_text_detect() and owlvit_image_guided_detect() defined.")

### 9.3 — Evaluation: run each variant over target_domain_val and score against ground truth

In [ ]:
def evaluate_owlvit_variant(name, detect_fn, score_thresh, image_dir, label_dir,
                             iou_thresh=0.5, max_images=None):
    image_dir, label_dir = Path(image_dir), Path(label_dir)
    img_paths = sorted(list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.jpeg")) +
                        list(image_dir.glob("*.png")))
    if max_images:
        img_paths = img_paths[:max_images]

    total_tp, total_fp, total_fn = 0, 0, 0
    t0 = time.time()
    print(f"\nEvaluating '{name}' on {len(img_paths)} images...")

    for i, img_path in enumerate(img_paths):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]
        pil_image = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

        gt_boxes = load_ashpilanta_gt_as_xyxy_s9(
            label_dir / f"{img_path.stem}.txt", w, h, AP_CLASS_ID)

        boxes, scores = detect_fn(pil_image)
        keep = scores >= score_thresh
        boxes = boxes[keep]

        tp, fp, fn = match_ashpilanta_detections_s9(boxes, gt_boxes, iou_thresh)
        total_tp += tp; total_fp += fp; total_fn += fn

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(img_paths)}]")

    precision = total_tp / max(total_tp + total_fp, 1)
    recall    = total_tp / max(total_tp + total_fn, 1)
    f1        = 2 * precision * recall / max(precision + recall, 1e-9)
    elapsed   = time.time() - t0

    print(f"  TP={total_tp} FP={total_fp} FN={total_fn}  "
          f"P={precision:.3f} R={recall:.3f} F1={f1:.3f}  ({elapsed:.1f}s)")
    return {"variant": name, "tp": total_tp, "fp": total_fp, "fn": total_fn,
            "precision": precision, "recall": recall, "f1": f1, "seconds": elapsed}


IMG_DIR   = PATH_TARGET_VAL / "images" / "val"
LABEL_DIR = PATH_TARGET_VAL / "labels" / "val"

# Set a cap while sanity-checking; set to None for the full validation set.
FOUNDATION_MAX_IMAGES = 100

foundation_rows = []

foundation_rows.append(evaluate_owlvit_variant(
    "owlvit_zeroshot_text",
    detect_fn=lambda im: owlvit_text_detect(im, ["insect", "small insect", "moth"], score_thresh=0.05),
    score_thresh=0.05, image_dir=IMG_DIR, label_dir=LABEL_DIR, max_images=FOUNDATION_MAX_IMAGES))

foundation_rows.append(evaluate_owlvit_variant(
    "owlvit_imageguided_k1",
    detect_fn=lambda im: owlvit_image_guided_detect(im, EXEMPLARS_K1, score_thresh=0.5),
    score_thresh=0.5, image_dir=IMG_DIR, label_dir=LABEL_DIR, max_images=FOUNDATION_MAX_IMAGES))

foundation_rows.append(evaluate_owlvit_variant(
    "owlvit_imageguided_k5",
    detect_fn=lambda im: owlvit_image_guided_detect(im, EXEMPLARS_K5, score_thresh=0.5),
    score_thresh=0.5, image_dir=IMG_DIR, label_dir=LABEL_DIR, max_images=FOUNDATION_MAX_IMAGES))

foundation_df = pd.DataFrame(foundation_rows)
foundation_csv = PATH_RESULTS / "foundation_model_baseline_results.csv"
foundation_df.to_csv(foundation_csv, index=False)
print("\n" + foundation_df.to_string(index=False))
print("\nSaved:", foundation_csv)

**Reading the table:** compare `owlvit_*` precision/recall/F1 against your Exp3 pipeline's
own precision/recall (available from `evaluate_field_pipeline(..., pipeline="baseline")` in
Cell 23, or from the AP numbers in Section 8.4). A large gap in favor of your trained pipeline
is the actual evidence for the thesis claim "the data-oriented approach was the right choice
for this problem" — a foundation model with zero or minimal exemplars struggling on a small,
visually subtle target-domain insect is exactly the expected/interesting result, and either
outcome (it struggles, or it's surprisingly competitive) is worth reporting honestly.

**Before running on the full validation set:** `FOUNDATION_MAX_IMAGES = 100` above caps the
run for a quick sanity check first. OWL-ViT inference is CPU/GPU-bound per image (no training),
so the full set is just slower, not riskier — once the 100-image run looks sane, set
`FOUNDATION_MAX_IMAGES = None` and re-run Cell 9.3's last three calls for the real numbers.

In [ ]:
# ==============================================================================
#  ARCHIVE SECTION 9 — OWL-ViT foundation-model baseline results.
# ==============================================================================
import zipfile, time
from pathlib import Path

def _add_if_exists(zf, path, arcname):
    path = Path(path)
    if path.is_file():
        zf.write(path, arcname); return 1
    return 0

_stamp = time.strftime("%Y%m%d_%H%M%S")
ARCHIVE_S9 = BASE / f"archive_section9_foundation_owlvit_{_stamp}.zip"

_count = 0
with zipfile.ZipFile(ARCHIVE_S9, "w", zipfile.ZIP_DEFLATED) as zf:
    _count += _add_if_exists(zf, PATH_RESULTS / "foundation_model_baseline_results.csv",
                             "foundation_model_baseline_results.csv")
    try:
        zf.writestr("foundation_df.json", foundation_df.to_json(orient="records", indent=2))
        _count += 1
    except Exception as e:
        print("  (foundation_df not in memory:", e, ")")

print(f"Section 9 archive written: {ARCHIVE_S9}")
print(f"  files added: {_count}   size: {ARCHIVE_S9.stat().st_size/1e6:.2f} MB")
print("Download it from the Kaggle output panel, or 'Save Version' to persist it.")

## Section 10 — Model-oriented baseline: TFA-style frozen-detector fine-tuning (Experiment 4)

Fills the model-oriented gap from `few_shot_techniques_menu.md`: your Exp3 fine-tuning
recipe has never been compared against the classic FSOD model-oriented baseline, Two-stage
Fine-tuning Approach (TFA, Wang et al. 2020) — freeze the whole base-class detector, train
only a new classifier for the novel class.

YOLOv8's decoupled head has no swappable per-class classifier (`cv3` outputs all class
logits from one shared conv stack), so this is a TFA-style approximation: **freeze
everything except the classification branch (`cv3`) of the Detect head** — backbone, neck,
and box regression (`cv2`) stay frozen; only classification weights update. This is the
adaptation the menu itself identifies as feasible for Ultralytics YOLO.

Starts from Exp1's weights (the trained 3-class base detector — the correct TFA starting
point, since TFA assumes a competent base-class detector already exists), trains on the same
`YAML_EXP3` data as your main pipeline, and evaluates on the same `target_domain_val` set —
so its Ashpilanta AP is directly comparable to the Exp3 row already in your results table.

Only depends on Section 3 (config/dataset cells) and Cell 60 (`m_exp3` reference, `get_class_ap`,
`safe_float`) — no dependency on Section 8 or 9.

In [ ]:
# ==============================================================================
#  EXPERIMENT 4 — TFA-style baseline: freeze backbone+neck+box-regression,
#  fine-tune only the classification branch of the Detect head.
#  Self-contained beyond Section 3 + Cell 60 (m_exp3, get_class_ap, safe_float).
# ==============================================================================

from ultralytics import YOLO

def tfa_freeze_callback(trainer):
    """Registered on 'on_train_start'. Runs AFTER Ultralytics' own freeze-by-index
    logic, so it can override it: freeze every parameter, then re-enable
    gradients only for the classification branch (cv3) of the final Detect
    head -- the closest YOLOv8 equivalent to TFA's 'train only the new-class
    classifier' step."""
    model = trainer.model
    head = model.model[-1]  # Detect() head, has .cv2 (box) and .cv3 (cls)

    for p in model.parameters():
        p.requires_grad_(False)

    if not hasattr(head, "cv3"):
        raise AttributeError(
            "Detect head has no 'cv3' attribute -- Ultralytics internals may have "
            "changed. Inspect trainer.model.model[-1] to find the classification "
            "branch and update this callback accordingly."
        )
    n_unfrozen = 0
    for p in head.cv3.parameters():
        p.requires_grad_(True)
        n_unfrozen += p.numel()

    n_total = sum(p.numel() for p in model.parameters())
    print(f"[TFA] Unfrozen (classification head only): {n_unfrozen:,} / {n_total:,} "
          f"params ({100*n_unfrozen/n_total:.2f}%)")


print("=" * 70)
print("EXPERIMENT 4: TFA-style frozen-detector fine-tuning")
print(f"  data={YAML_EXP3}  init={PATH_MODEL_EXP1}  (backbone+neck+box-reg frozen, cls-head only)")
print("=" * 70)

model_exp4 = YOLO(str(PATH_MODEL_EXP1))
model_exp4.add_callback("on_train_start", tfa_freeze_callback)

model_exp4.train(
    data=str(YAML_EXP3), epochs=EXP3_EPOCHS, imgsz=EXP3_IMGSZ, batch=EXP3_BATCH,
    device=DEVICE, workers=EXP3_WORKERS, patience=EXP3_PATIENCE,
    freeze=[],  # freezing is handled entirely by the callback above, not by index
    lr0=EXP3_LR0, lrf=EXP3_LRF, warmup_epochs=EXP3_WARMUP_EPOCHS,
    mosaic=EXP3_MOSAIC, copy_paste=EXP3_COPY_PASTE, degrees=EXP3_DEGREES,
    fliplr=EXP3_FLIPLR, flipud=EXP3_FLIPUD, scale=EXP3_SCALE, translate=EXP3_TRANSLATE,
    hsv_h=EXP3_HSV_H, hsv_s=EXP3_HSV_S, hsv_v=EXP3_HSV_V,
    cos_lr=True, close_mosaic=15, label_smoothing=0.05,
    project=str(BASE / "runs/detect"), name="experiment4_tfa", exist_ok=True, save=True, plots=True,
)

PATH_MODEL_EXP4 = BASE / "runs/detect/experiment4_tfa/weights/best.pt"
print("Finished Experiment 4 ->", PATH_MODEL_EXP4, " exists:", PATH_MODEL_EXP4.exists())

# ── Evaluate on the same target-domain val set as Exp3, build comparison rows ──
m_exp4 = YOLO(str(PATH_MODEL_EXP4)).val(
    data=str(YAML_TARGET_VAL), split="val", verbose=False, plots=False,
    project=str(BASE / "runs/detect"), name="val_experiment4_tfa", exist_ok=True,
)

exp4_rows = [
    {"variant": "Exp3 (full method, data-oriented)",
     "mAP50": safe_float(m_exp3.box.map50), "mAP50_95": safe_float(m_exp3.box.map),
     "AP_Ashpilanta_50": get_class_ap(m_exp3, AP_CLASS_ID, "map50"),
     "AP_Ashpilanta_50_95": get_class_ap(m_exp3, AP_CLASS_ID, "map")},
    {"variant": "Exp4 (TFA-style, model-oriented)",
     "mAP50": safe_float(m_exp4.box.map50), "mAP50_95": safe_float(m_exp4.box.map),
     "AP_Ashpilanta_50": get_class_ap(m_exp4, AP_CLASS_ID, "map50"),
     "AP_Ashpilanta_50_95": get_class_ap(m_exp4, AP_CLASS_ID, "map")},
]
exp4_df = pd.DataFrame(exp4_rows)
exp4_csv = PATH_RESULTS / "experiment4_tfa_vs_exp3.csv"
exp4_df.to_csv(exp4_csv, index=False)

print("\n" + exp4_df.to_string(index=False))
print("\nSaved:", exp4_csv)
print("\nMarkdown (paste-ready):\n")
print(exp4_df.to_markdown(index=False))

In [ ]:
# ==============================================================================
#  ARCHIVE SECTION 10 — Experiment 4 (TFA-style) results + weights.
# ==============================================================================
import zipfile, time
from pathlib import Path

def _add_if_exists(zf, path, arcname):
    path = Path(path)
    if path.is_file():
        zf.write(path, arcname); return 1
    elif path.is_dir():
        n = 0
        for f in path.rglob("*"):
            if f.is_file():
                zf.write(f, str(Path(arcname) / f.relative_to(path))); n += 1
        return n
    return 0

_stamp = time.strftime("%Y%m%d_%H%M%S")
ARCHIVE_S10 = BASE / f"archive_section10_exp4_tfa_{_stamp}.zip"
_runs = BASE / "runs" / "detect"

_count = 0
with zipfile.ZipFile(ARCHIVE_S10, "w", zipfile.ZIP_DEFLATED) as zf:
    _count += _add_if_exists(zf, PATH_RESULTS / "experiment4_tfa_vs_exp3.csv",
                             "experiment4_tfa_vs_exp3.csv")
    _count += _add_if_exists(zf, _runs / "experiment4_tfa" / "weights" / "best.pt",
                             "experiment4_tfa/best.pt")
    for extra in ["results.csv", "results.png", "args.yaml", "PR_curve.png",
                  "confusion_matrix.png"]:
        _count += _add_if_exists(zf, _runs / "experiment4_tfa" / extra,
                                 f"experiment4_tfa/{extra}")
    _count += _add_if_exists(zf, _runs / "val_experiment4_tfa", "val_experiment4_tfa")
    try:
        zf.writestr("exp4_df.json", exp4_df.to_json(orient="records", indent=2))
        _count += 1
    except Exception as e:
        print("  (exp4_df not in memory:", e, ")")

print(f"Section 10 archive written: {ARCHIVE_S10}")
print(f"  files added: {_count}   size: {ARCHIVE_S10.stat().st_size/1e6:.1f} MB")
print("Download it from the Kaggle output panel, or 'Save Version' to persist it.")

## Section 11 — Algorithm-oriented baseline: Prototypical Network crop classifier (Experiment 5)

Fills the algorithm/metric-learning gap from `few_shot_techniques_menu.md`: your CLIP filter
(Cell 13) uses a *frozen foundation-model* embedding. This section trains a *task-specific*
embedding on your own crops with a prototypical loss (Snell et al. 2017), then classifies
the same query crops the same way — mean-prototype + cosine similarity. That makes it a clean
"foundation-model embedding vs. task-trained embedding" comparison, holding everything else
(the crops, the support/query split, the prototype-similarity decision rule) fixed.

**Same data as the CLIP experiment, on purpose:**
- Support set = the same real Ashpilanta training crops (`support_images_all` logic).
- Query set = the same merged-val object crops, binary-labelled Ashpilanta vs. non-Ashpilanta.
- Same K sweep (`CLIP_K_VALUES`), same mean-similarity threshold rule.

The only thing that changes is *what produces the embedding*: a small CNN trained with
episodic prototypical loss on your crops, instead of frozen CLIP. Any difference in
precision/recall/F1 at matched K is therefore attributable to the embedding, which is exactly
the ablation the menu asks for.

Depends only on Section 3 and on Cell 13 having run first (it reuses Cell 13's
`crop_objects_from_dataset`, `support_images_all`, `query_crops`, `query_binary_labels`,
`clip_results`). Run Cell 13 before this section.

In [ ]:
# ==============================================================================
#  EXPERIMENT 5 — Prototypical Network crop classifier (algorithm-oriented).
#  Trains a task-specific embedding with episodic prototypical loss on YOUR
#  crops, to compare head-to-head against the frozen-CLIP embedding (Cell 13).
#  Reuses Cell 13's crop objects, so RUN CELL 13 FIRST.
# ==============================================================================

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, random
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import torchvision.models as tvm

proto_device = "cuda" if torch.cuda.is_available() else "cpu"
PROTO_SEED   = globals().get("RANDOM_SEED", 42)
random.seed(PROTO_SEED); np.random.seed(PROTO_SEED); torch.manual_seed(PROTO_SEED)

# ── Guard: this section reuses Cell 13's in-memory crops ──────────────────────
for _needed in ["support_images_all", "query_crops", "query_binary_labels",
                "crop_objects_from_dataset", "PATH_MERGED", "CLIP_K_VALUES"]:
    if _needed not in globals():
        raise RuntimeError(
            f"'{_needed}' not found. Run Cell 13 (CLIP few-shot experiment) first -- "
            "Section 11 reuses its crops and support/query split so the two baselines "
            "are scored on identical data."
        )

# ── Embedding network: small ResNet18 backbone -> L2-normalised embedding ─────
class ProtoEmbedNet(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        backbone = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT)
        backbone.fc = nn.Linear(backbone.fc.in_features, emb_dim)
        self.backbone = backbone
    def forward(self, x):
        return F.normalize(self.backbone(x), p=2, dim=-1)

proto_tf = T.Compose([
    T.Resize((96, 96)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def crops_to_tensor(pil_crops):
    if len(pil_crops) == 0:
        return torch.empty(0, 3, 96, 96)
    return torch.stack([proto_tf(c.convert("RGB")) for c in pil_crops])

# ── Build a TRAIN pool of crops (Ashpilanta vs. non-Ashpilanta) from the
#    TRAIN split of PATH_MERGED -- kept strictly separate from the val query
#    crops the classifier is later evaluated on, to avoid leakage. ────────────
train_crops, train_binary_labels, _ = crop_objects_from_dataset(
    PATH_MERGED, split="train", padding=0.10)
train_binary_labels = np.asarray(train_binary_labels)

pos_idx = np.where(train_binary_labels == 1)[0]
neg_idx = np.where(train_binary_labels == 0)[0]
print(f"Prototypical training pool -> AP crops: {len(pos_idx)}, non-AP crops: {len(neg_idx)}")

if len(pos_idx) < 2 or len(neg_idx) < 2:
    raise RuntimeError(
        "Not enough train crops of each class to train a prototypical network "
        f"(AP={len(pos_idx)}, non-AP={len(neg_idx)}). Need >=2 of each. "
        "Check that PATH_MERGED/images/train contains Ashpilanta labels."
    )

# Pre-tensorise the whole pool once (crops are small; this is cheap).
train_pos_tensors = crops_to_tensor([train_crops[i] for i in pos_idx])
train_neg_tensors = crops_to_tensor([train_crops[i] for i in neg_idx])

def sample_episode(n_support=5, n_query=5):
    """One 2-way (AP vs non-AP) episode. Returns support/query tensors + labels."""
    def pick(tensors, n):
        n = min(n, len(tensors))
        idx = torch.randperm(len(tensors))[:n]
        return tensors[idx]
    s_pos = pick(train_pos_tensors, n_support); q_pos = pick(train_pos_tensors, n_query)
    s_neg = pick(train_neg_tensors, n_support); q_neg = pick(train_neg_tensors, n_query)
    support = torch.cat([s_pos, s_neg], 0)
    query   = torch.cat([q_pos, q_neg], 0)
    q_labels = torch.cat([torch.ones(len(q_pos)), torch.zeros(len(q_neg))]).long()
    split = (len(s_pos), len(s_neg))
    return support, split, query, q_labels

def prototypical_loss(embed_net, support, split, query, q_labels):
    s_emb = embed_net(support.to(proto_device))
    q_emb = embed_net(query.to(proto_device))
    n_pos, n_neg = split
    proto_pos = s_emb[:n_pos].mean(0, keepdim=True)
    proto_neg = s_emb[n_pos:n_pos+n_neg].mean(0, keepdim=True)
    protos = torch.cat([proto_neg, proto_pos], 0)          # index 0=non-AP, 1=AP
    dists  = torch.cdist(q_emb, protos)                    # [Nq, 2]
    logits = -dists
    loss = F.cross_entropy(logits, q_labels.to(proto_device))
    acc  = (logits.argmax(1).cpu() == q_labels).float().mean().item()
    return loss, acc

# ── Train ─────────────────────────────────────────────────────────────────────
PROTO_EPISODES = 400
embed_net = ProtoEmbedNet(emb_dim=128).to(proto_device)
opt = torch.optim.Adam(embed_net.parameters(), lr=1e-4)

print("\nTraining prototypical embedding network...")
embed_net.train()
for ep in range(1, PROTO_EPISODES + 1):
    support, split, query, q_labels = sample_episode(n_support=5, n_query=5)
    loss, acc = prototypical_loss(embed_net, support, split, query, q_labels)
    opt.zero_grad(); loss.backward(); opt.step()
    if ep % 50 == 0:
        print(f"  episode {ep:4d}/{PROTO_EPISODES}  loss={loss.item():.4f}  episode_acc={acc:.3f}")

embed_net.eval()
print("Prototypical embedding network trained.")

In [ ]:
# ==============================================================================
#  EXPERIMENT 5 (cont.) — evaluate the trained prototypical embedding on the
#  SAME query crops and K sweep as Cell 13, using the SAME mean-similarity
#  threshold rule, then merge into a CLIP-vs-Prototypical comparison table.
# ==============================================================================

from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import pandas as pd

@torch.no_grad()
def proto_embed_pil(pil_crops, batch_size=64):
    embs = []
    for i in range(0, len(pil_crops), batch_size):
        batch = crops_to_tensor(pil_crops[i:i+batch_size]).to(proto_device)
        embs.append(embed_net(batch).cpu())
    return torch.cat(embs, 0) if embs else torch.empty(0, 128)

# Embed the identical query crops Cell 13 used.
proto_query_emb = proto_embed_pil(query_crops)

proto_results = {}
for k in CLIP_K_VALUES:
    k_eff = min(k, len(support_images_all))
    was_capped = k_eff < k

    # Same support crops, same "first k_eff" selection as Cell 13.
    support_subset = support_images_all[:k_eff]
    support_emb = proto_embed_pil(support_subset)
    proto = F.normalize(support_emb.mean(dim=0, keepdim=True), p=2, dim=-1)

    sims = (proto_query_emb @ proto.T).squeeze(1).numpy()
    threshold = float(np.mean(sims))                 # identical decision rule to Cell 13
    pred = (sims >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        query_binary_labels, pred, average="binary", zero_division=0)
    acc = accuracy_score(query_binary_labels, pred)

    proto_results[f"Proto_{k}_shot_crop_classifier"] = {
        "requested_k": int(k), "accuracy": float(acc),
        "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "threshold": threshold, "num_query_crops": int(len(query_crops)),
        "num_support": int(k_eff), "capped_by_support_pool": bool(was_capped),
    }
    print(f"Proto K={k:>2} (eff {k_eff:>2})  P={precision:.3f} R={recall:.3f} F1={f1:.3f} acc={acc:.3f}")

# ── Merge CLIP (Cell 13) and Prototypical results into one comparison table ───
rows = []
for k in CLIP_K_VALUES:
    clip_row  = clip_results.get(f"CLIP_{k}_shot_crop_classifier")
    proto_row = proto_results.get(f"Proto_{k}_shot_crop_classifier")
    if clip_row:
        rows.append({"embedding": "CLIP (frozen foundation model)", "K": k,
                     "precision": clip_row["precision"], "recall": clip_row["recall"],
                     "f1": clip_row["f1"], "accuracy": clip_row["accuracy"],
                     "num_support": clip_row["num_support"]})
    if proto_row:
        rows.append({"embedding": "Prototypical (task-trained)", "K": k,
                     "precision": proto_row["precision"], "recall": proto_row["recall"],
                     "f1": proto_row["f1"], "accuracy": proto_row["accuracy"],
                     "num_support": proto_row["num_support"]})

proto_vs_clip_df = pd.DataFrame(rows)
proto_csv = PATH_RESULTS / "proto_vs_clip_embedding_comparison.csv"
proto_vs_clip_df.to_csv(proto_csv, index=False)

print("\n" + proto_vs_clip_df.to_string(index=False))
print("\nSaved:", proto_csv)
print("\nMarkdown (paste-ready):\n")
print(proto_vs_clip_df.to_markdown(index=False))

**Reading the table:** at each K, compare the two `embedding` rows. If the task-trained
prototypical embedding beats frozen CLIP, that argues a task-specific metric-learned
embedding is worth the extra training for this problem; if CLIP wins or ties, that *justifies
your original choice* of the frozen foundation-model filter (no training needed, broad visual
vocabulary) — which is itself a clean, reportable result. Either way this is the
algorithm-family comparison the methodology was missing, holding crops, split, and decision
rule fixed so the embedding is the only variable.

**Caveat to state in the write-up:** the prototypical network trains on the *merged* train
split (real + synthetic Ashpilanta), so a very small real-AP crop count there limits how much
task-specific signal it can learn — a genuine constraint of the few-shot regime, not a bug.
If the prototypical embedding underperforms, this is the most likely reason and worth naming
explicitly rather than presenting it as a clean loss for metric learning in general.

In [ ]:
# ==============================================================================
#  ARCHIVE SECTION 11 — Prototypical-network (Exp 5) results + trained embedding.
# ==============================================================================
import zipfile, time, torch
from pathlib import Path

def _add_if_exists(zf, path, arcname):
    path = Path(path)
    if path.is_file():
        zf.write(path, arcname); return 1
    return 0

_stamp = time.strftime("%Y%m%d_%H%M%S")
ARCHIVE_S11 = BASE / f"archive_section11_proto_exp5_{_stamp}.zip"

# Save the trained embedding weights so you never have to retrain the network.
_proto_weights = BASE / "proto_embed_net.pt"
try:
    torch.save(embed_net.state_dict(), _proto_weights)
    print("Saved prototypical embedding weights ->", _proto_weights)
except Exception as e:
    print("  (could not save embed_net:", e, ")")

_count = 0
with zipfile.ZipFile(ARCHIVE_S11, "w", zipfile.ZIP_DEFLATED) as zf:
    _count += _add_if_exists(zf, PATH_RESULTS / "proto_vs_clip_embedding_comparison.csv",
                             "proto_vs_clip_embedding_comparison.csv")
    _count += _add_if_exists(zf, _proto_weights, "proto_embed_net.pt")
    try:
        zf.writestr("proto_vs_clip_df.json", proto_vs_clip_df.to_json(orient="records", indent=2))
        _count += 1
    except Exception as e:
        print("  (proto_vs_clip_df not in memory:", e, ")")
    # also stash raw per-K dicts if present
    for _name in ["proto_results", "clip_results"]:
        if _name in globals():
            import json as _json
            zf.writestr(f"{_name}.json", _json.dumps(globals()[_name], indent=2)); _count += 1

print(f"Section 11 archive written: {ARCHIVE_S11}")
print(f"  files added: {_count}   size: {ARCHIVE_S11.stat().st_size/1e6:.2f} MB")
print("Download it from the Kaggle output panel, or 'Save Version' to persist it.")